# 14-block study domain and groundwater-level trends

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Mann–Kendall/Sen trends with FDR and TFPW checks (Steps 7–12), well coordinates (Step 13), the **14-block Alwar analysis domain** (Step 14), trends restricted to that domain (Step 15), and maps, block summaries and spatial clustering (Steps 16–18).

| | |
|---|---|
| **Reads** | `gwl/diagnostics/Alwar_Source_Recovery/Step_6…`, `gw_framework/Shapefile/LGD_Blocks.geojson`, GEC 2024-25 workbook |
| **Writes** | `gwl/diagnostics/Alwar_Source_Recovery/Step_7 … Step_18_*` |
| **Cells** | 37 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Trend tests on all Alwar candidate series (Steps 7–12)
- 2. Well coordinates (Step 13)
- 3. The 14-block study domain (Step 14)
- 4. Trends inside the domain (Step 15)
- 5. Maps, block summaries, clustering (Steps 16–18)

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["pymannkendall", "geopandas", "libpysal", "esda", "openpyxl"], modflow=False)
del TOKEN, URL


## 1. Trend tests on all Alwar candidate series (Steps 7–12)

**[orig #102]** Step 7 — candidate series and Mann–Kendall / Sen trends  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Install if needed:
# !pip -q install pymannkendall

from scipy.stats import theilslopes
import pymannkendall as mk

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

INPUT = BASE / "Step_2_QC" / "estimated_heads_deduplicated_analysis_copy.csv"
CANDIDATE_FILE = (
    BASE / "Step_6_Trend_Candidate_QC" /
    "03_initial_trend_candidates.csv"
)

OUT = BASE / "Step_7_Seasonal_Trend_Analysis"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)
candidates = pd.read_csv(CANDIDATE_FILE, low_memory=False)

# ---------------------------------------------------------
# FILTER ALWAR AND CANDIDATE WELLS
# ---------------------------------------------------------
alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

candidate_ids = candidates["Well_key"].dropna().astype(str).unique()
alwar["Well_key"] = alwar["Well_key"].astype(str)

obs = alwar[
    alwar["Well_key"].isin(candidate_ids)
    & pd.to_numeric(
        alwar["Depth_to_water_bgl_m"], errors="coerce"
    ).notna()
].copy()

obs["Year"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["DTW_m_bgl"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)

obs = obs.dropna(subset=["Year", "DTW_m_bgl", "Season"])
obs["Year"] = obs["Year"].astype(int)

# Normalize season labels for reporting, retaining original Season.
obs["Season_clean"] = (
    obs["Season"].astype(str).str.strip().str.casefold()
)

print("Candidate wells:", len(candidate_ids))
print("Numeric observations included:", len(obs))

# ---------------------------------------------------------
# 1. SEASONAL CONSISTENCY PER WELL
# ---------------------------------------------------------
season_counts = (
    obs.groupby(["Well_key", "Season_clean"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

season_counts.to_csv(
    OUT / "01_candidate_well_season_counts.csv", index=False
)

# Number of unique years with numeric data per well and season
season_years = (
    obs.groupby(["Well_key", "Season_clean"])["Year"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

season_years.to_csv(
    OUT / "02_candidate_well_season_year_coverage.csv", index=False
)

# ---------------------------------------------------------
# 2. YEAR-SEASON COVERAGE
# ---------------------------------------------------------
year_season = (
    obs.groupby(["Year", "Season_clean"])
    .agg(
        numeric_records=("DTW_m_bgl", "size"),
        wells=("Well_key", "nunique"),
        mean_DTW_m=("DTW_m_bgl", "mean"),
        median_DTW_m=("DTW_m_bgl", "median"),
        sd_DTW_m=("DTW_m_bgl", "std")
    )
    .reset_index()
)

year_season.to_csv(
    OUT / "03_year_season_coverage_and_summary.csv", index=False
)

# ---------------------------------------------------------
# 3. PRELIMINARY WELL-LEVEL TREND TESTS
# Each season is analysed separately within each well.
# This avoids mixing seasonal differences in a single series.
# ---------------------------------------------------------
trend_rows = []

for (well_id, season), g in obs.groupby(
    ["Well_key", "Season_clean"]
):
    # One value per well/year/season is expected from Step 6.
    # Use median defensively if any duplicate remains.
    ts = (
        g.groupby("Year")["DTW_m_bgl"]
        .median()
        .sort_index()
    )

    n = len(ts)
    span = int(ts.index.max() - ts.index.min()) if n >= 2 else 0

    result = {
        "Well_key": well_id,
        "Season": season,
        "n_years": n,
        "year_span": span,
        "first_year": int(ts.index.min()) if n else np.nan,
        "last_year": int(ts.index.max()) if n else np.nan,
        "sen_slope_m_per_year": np.nan,
        "mk_p_value": np.nan,
        "mk_trend": "NOT_TESTED"
    }

    if n >= 8 and span >= 7 and ts.nunique() > 1:
        x = ts.index.to_numpy(dtype=float)
        y = ts.to_numpy(dtype=float)

        slope, intercept, low_slope, high_slope = theilslopes(
            y, x, alpha=0.95
        )

        try:
            mk_result = mk.original_test(y)
            result["sen_slope_m_per_year"] = float(slope)
            result["sen_slope_ci_low"] = float(low_slope)
            result["sen_slope_ci_high"] = float(high_slope)
            result["mk_p_value"] = float(mk_result.p)
            result["mk_trend"] = str(mk_result.trend)
        except Exception as e:
            result["mk_trend"] = "TEST_ERROR: " + str(e)

    trend_rows.append(result)

trends = pd.DataFrame(trend_rows)

# DTW is depth below ground:
# positive Sen slope = increasing depth to water (water table deepening)
# negative Sen slope = decreasing depth to water (water table becoming shallower)
trends["direction_interpretation"] = np.select(
    [
        trends["sen_slope_m_per_year"] > 0,
        trends["sen_slope_m_per_year"] < 0,
        trends["sen_slope_m_per_year"] == 0
    ],
    [
        "DTW increasing; water table deepening",
        "DTW decreasing; water table becoming shallower",
        "No linear change in DTW"
    ],
    default="Not estimated"
)

trends.to_csv(
    OUT / "04_exploratory_well_season_trends.csv", index=False
)

# ---------------------------------------------------------
# 4. DESCRIPTIVE SUMMARY
# ---------------------------------------------------------
tested = trends[trends["mk_p_value"].notna()].copy()

summary = {
    "candidate_wells": int(len(candidate_ids)),
    "wells_with_numeric_observations": int(obs["Well_key"].nunique()),
    "well_season_series": int(len(trends)),
    "series_tested": int(len(tested)),
    "series_not_tested": int(len(trends) - len(tested)),
    "tested_series_p_below_0_05_unadjusted": int(
        (tested["mk_p_value"] < 0.05).sum()
    ),
    "tested_series_positive_sen_slope": int(
        (tested["sen_slope_m_per_year"] > 0).sum()
    ),
    "tested_series_negative_sen_slope": int(
        (tested["sen_slope_m_per_year"] < 0).sum()
    )
}

pd.Series(summary, name="value").to_csv(
    OUT / "05_exploratory_trend_summary.csv"
)

print("\nEXPLORATORY TREND SUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nOutputs saved to:", OUT)


**[orig #103]** Install `pymannkendall`  
<sub>✅ ran OK</sub>

In [ ]:
%pip install pymannkendall

**[orig #104]** Step 7 — check the trend output files  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery/"
    "Step_7_Seasonal_Trend_Analysis"
)

files = [
    "04_exploratory_well_season_trends.csv",
    "05_exploratory_trend_summary.csv",
    "02_candidate_well_season_year_coverage.csv",
]

for filename in files:
    path = BASE / filename
    print("\n" + "=" * 80)
    print("FILE:", filename)

    if not path.exists():
        print("NOT FOUND:", path)
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())

**[orig #107]** Step 8 — trend validation with FDR correction  
<sub>✅ ran OK</sub>

In [ ]:

# ============================================================
# STEP 8: ALWAR GROUNDWATER TREND VALIDATION
# Corrected, self-contained version
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP7 = ROOT / "Step_7_Seasonal_Trend_Analysis"
INPUT = (
    ROOT / "Step_2_QC" /
    "estimated_heads_deduplicated_analysis_copy.csv"
)
TREND_FILE = STEP7 / "04_exploratory_well_season_trends.csv"
OUTPUT = ROOT / "Step_8_Trend_Validation"
OUTPUT.mkdir(parents=True, exist_ok=True)

for f in [INPUT, TREND_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"File not found: {f}")

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------
trends = pd.read_csv(TREND_FILE)
obs = pd.read_csv(INPUT)

required_trends = [
    "Well_key", "Season", "n_years", "year_span",
    "first_year", "last_year", "sen_slope_m_per_year",
    "mk_p_value", "mk_trend",
    "sen_slope_ci_low", "sen_slope_ci_high"
]
required_obs = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Depth_to_water_bgl_m", "District"
]

missing_trend_cols = [c for c in required_trends if c not in trends.columns]
missing_obs_cols = [c for c in required_obs if c not in obs.columns]

if missing_trend_cols:
    raise ValueError(f"Missing trend columns: {missing_trend_cols}")
if missing_obs_cols:
    raise ValueError(f"Missing observation columns: {missing_obs_cols}")

# ------------------------------------------------------------
# 3. Standardize keys and prepare Alwar observations
# ------------------------------------------------------------
for df in [trends]:
    df["Well_key"] = df["Well_key"].astype(str).str.strip()
    df["Season_clean"] = df["Season"].astype(str).str.strip().str.lower()

obs["Well_key"] = (
    obs["Source_file"].astype(str).str.strip() + "|" +
    obs["Source_sheet"].astype(str).str.strip() + "|" +
    obs["Source_row_number"].astype(str).str.strip()
)
obs["Season_clean"] = obs["Season"].astype(str).str.strip().str.lower()
obs["District_clean"] = obs["District"].astype(str).str.strip().str.lower()
obs["Year_num"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["DTW_num"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)

alwar = obs.loc[
    (obs["District_clean"] == "alwar") &
    obs["DTW_num"].notna() &
    obs["Year_num"].notna() &
    obs["Season_clean"].isin(["pre", "post"])
].copy()

alwar["Year_num"] = alwar["Year_num"].astype(int)

# ------------------------------------------------------------
# 4. Temporal and autocorrelation diagnostics
# ------------------------------------------------------------
diagnostics = []

for (well, season), g in alwar.groupby(
    ["Well_key", "Season_clean"], dropna=False
):
    g = g.sort_values("Year_num")
    years = g["Year_num"].to_numpy(dtype=int)
    values = g["DTW_num"].to_numpy(dtype=float)
    unique_years = np.unique(years)

    gaps = np.diff(unique_years) if len(unique_years) >= 2 else np.array([])

    if len(gaps):
        max_gap = int(gaps.max())
        mean_gap = float(gaps.mean())
        missing_years = int(np.maximum(gaps - 1, 0).sum())
    else:
        max_gap = np.nan
        mean_gap = np.nan
        missing_years = 0

    # Correlation between adjacent available readings.
    # This is only a diagnostic; it does not correct the MK test.
    if (
        len(values) >= 4
        and np.std(values[:-1]) > 0
        and np.std(values[1:]) > 0
    ):
        lag1 = float(np.corrcoef(values[:-1], values[1:])[0, 1])
    else:
        lag1 = np.nan

    diagnostics.append({
        "Well_key": str(well),
        "Season_clean": str(season),
        "n_numeric_observations": int(len(values)),
        "n_unique_observation_years": int(len(unique_years)),
        "first_observation_year": int(years.min()),
        "last_observation_year": int(years.max()),
        "observed_year_span": int(years.max() - years.min()),
        "max_gap_between_observed_years": max_gap,
        "mean_gap_between_observed_years": mean_gap,
        "missing_intervening_years": missing_years,
        "duplicate_observations_within_year": int(len(years) - len(unique_years)),
        "lag1_autocorrelation_diagnostic": lag1,
        "possible_positive_serial_dependence": (
            bool(lag1 > 0.3) if pd.notna(lag1) else False
        )
    })

diag = pd.DataFrame(diagnostics)

# Validate keys before joining
for name, df in [("trends", trends), ("diagnostics", diag)]:
    if df.duplicated(["Well_key", "Season_clean"]).any():
        duplicates = df.loc[
            df.duplicated(["Well_key", "Season_clean"], keep=False),
            ["Well_key", "Season_clean"]
        ]
        raise ValueError(
            f"Duplicate keys found in {name}:\n{duplicates.head(20)}"
        )

# ------------------------------------------------------------
# 5. Benjamini-Hochberg FDR correction
# ------------------------------------------------------------
trends["mk_p_value_numeric"] = pd.to_numeric(
    trends["mk_p_value"], errors="coerce"
)

p = trends["mk_p_value_numeric"]
valid_p = p.notna() & p.between(0, 1)
p_values = p.loc[valid_p].to_numpy()
m = len(p_values)

trends["mk_p_fdr_bh"] = np.nan

if m > 0:
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    ranks = np.arange(1, m + 1)

    adjusted_sorted = sorted_p * m / ranks
    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]
    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    adjusted_original_order = np.empty(m)
    adjusted_original_order[order] = adjusted_sorted

    trends.loc[valid_p, "mk_p_fdr_bh"] = adjusted_original_order

trends["fdr_significant_05"] = (
    trends["mk_p_fdr_bh"].notna() &
    (trends["mk_p_fdr_bh"] < 0.05)
)

# ------------------------------------------------------------
# 6. Merge trend and diagnostic tables
# ------------------------------------------------------------
result = trends.merge(
    diag,
    on=["Well_key", "Season_clean"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Merge status:")
print(result["_merge"].value_counts(dropna=False))

unmatched = result.loc[result["_merge"] != "both"].copy()
if not unmatched.empty:
    unmatched.to_csv(
        OUTPUT / "06_unmatched_trend_series.csv", index=False
    )
    print("WARNING: unmatched series saved for review.")

result.drop(columns=["_merge"], inplace=True)

result["observation_count_matches"] = (
    pd.to_numeric(result["n_years"], errors="coerce") ==
    result["n_unique_observation_years"]
)

# ------------------------------------------------------------
# 7. Classify results conservatively
# ------------------------------------------------------------
def classify(row):
    pval = row["mk_p_value_numeric"]
    fdr = row["mk_p_fdr_bh"]
    slope = row["sen_slope_m_per_year"]

    if pd.isna(pval) or pd.isna(slope):
        return "NOT_TESTED_OR_MISSING_STATISTICS"

    if pd.isna(fdr):
        return "REVIEW_MISSING_FDR_VALUE"

    if fdr >= 0.05:
        return "NO_FDR_SUPPORTED_TREND"

    if slope > 0:
        return "FDR_SUPPORTED_DTW_INCREASE"
    if slope < 0:
        return "FDR_SUPPORTED_DTW_DECREASE"
    return "FDR_SUPPORTED_ZERO_SLOPE_REVIEW"

result["trend_classification"] = result.apply(classify, axis=1)

result["review_flag"] = ""
result.loc[
    result["max_gap_between_observed_years"] > 5,
    "review_flag"
] += "GAP_OVER_5_YEARS; "

result.loc[
    result["possible_positive_serial_dependence"].fillna(False),
    "review_flag"
] += "POSSIBLE_POSITIVE_SERIAL_DEPENDENCE; "

result.loc[
    ~result["observation_count_matches"].fillna(False),
    "review_flag"
] += "OBSERVATION_COUNT_MISMATCH; "

result.loc[
    result["duplicate_observations_within_year"].fillna(0) > 0,
    "review_flag"
] += "MULTIPLE_OBSERVATIONS_WITHIN_YEAR; "

result["review_flag"] = result["review_flag"].replace("", "NONE")

# ------------------------------------------------------------
# 8. Export tables
# ------------------------------------------------------------
result.to_csv(
    OUTPUT / "01_validated_well_season_trends.csv", index=False
)
diag.to_csv(
    OUTPUT / "02_temporal_and_autocorrelation_diagnostics.csv",
    index=False
)

review = result.loc[
    result["review_flag"].ne("NONE") |
    result["trend_classification"].str.startswith("NOT_TESTED")
].copy()
review.to_csv(OUTPUT / "03_manual_review_queue.csv", index=False)

counts = (
    result.groupby("trend_classification", dropna=False)
    .size()
    .rename("series_count")
    .reset_index()
)
counts.to_csv(OUTPUT / "04_trend_classification_counts.csv", index=False)

summary = {
    "trend_rows": int(len(result)),
    "valid_original_mk_p_values": int(valid_p.sum()),
    "missing_or_invalid_mk_p_values": int((~valid_p).sum()),
    "FDR_significant_series": int(result["fdr_significant_05"].sum()),
    "FDR_supported_DTW_increase": int(
        (result["trend_classification"] == "FDR_SUPPORTED_DTW_INCREASE").sum()
    ),
    "FDR_supported_DTW_decrease": int(
        (result["trend_classification"] == "FDR_SUPPORTED_DTW_DECREASE").sum()
    ),
    "no_FDR_supported_trend": int(
        (result["trend_classification"] == "NO_FDR_SUPPORTED_TREND").sum()
    ),
    "series_with_gap_over_5_years": int(
        (result["max_gap_between_observed_years"] > 5).sum()
    ),
    "series_flagged_possible_positive_serial_dependence": int(
        result["possible_positive_serial_dependence"].fillna(False).sum()
    ),
    "series_with_observation_count_mismatch": int(
        (~result["observation_count_matches"].fillna(False)).sum()
    ),
    "caveat": (
        "FDR corrects multiple testing only. Lag-1 autocorrelation is a "
        "diagnostic and does not correct Mann-Kendall significance. "
        "Interpret trend significance provisionally until serial dependence, "
        "sampling irregularity, and source measurement quality are assessed."
    )
}

with open(OUTPUT / "05_step8_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("STEP 8 FINISHED")
print("=" * 70)
print(json.dumps(summary, indent=2))
print("\nTrend classification counts:")
display(counts)
print("\nSaved to:", OUTPUT)


**[orig #108]** Step 9 — serial-dependence-aware robustness (TFPW)  
<sub>✅ ran OK</sub>

In [ ]:

# ============================================================
# STEP 9: SERIAL-DEPENDENCE-AWARE TREND ROBUSTNESS
# Study: Alwar groundwater, pre- and post-monsoon DTW
# ============================================================

%pip -q install pymannkendall

from pathlib import Path
import json
import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import theilslopes

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

OBS_FILE = (
    ROOT / "Step_2_QC" /
    "estimated_heads_deduplicated_analysis_copy.csv"
)
STEP7_FILE = (
    ROOT / "Step_7_Seasonal_Trend_Analysis" /
    "04_exploratory_well_season_trends.csv"
)
STEP8_FILE = (
    ROOT / "Step_8_Trend_Validation" /
    "01_validated_well_season_trends.csv"
)

OUT = ROOT / "Step_9_Serial_Dependence_Robustness"
OUT.mkdir(parents=True, exist_ok=True)

for path in [OBS_FILE, STEP7_FILE, STEP8_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------------------
# 2. Load data and standardize observation keys
# ------------------------------------------------------------
obs = pd.read_csv(OBS_FILE)
step7 = pd.read_csv(STEP7_FILE)
step8 = pd.read_csv(STEP8_FILE)

obs["Well_key"] = (
    obs["Source_file"].astype(str).str.strip() + "|" +
    obs["Source_sheet"].astype(str).str.strip() + "|" +
    obs["Source_row_number"].astype(str).str.strip()
)

obs["Season_clean"] = obs["Season"].astype(str).str.strip().str.lower()
obs["District_clean"] = obs["District"].astype(str).str.strip().str.lower()
obs["Year_num"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["DTW_num"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)

obs = obs.loc[
    (obs["District_clean"] == "alwar") &
    obs["Season_clean"].isin(["pre", "post"]) &
    obs["Year_num"].notna() &
    obs["DTW_num"].notna()
].copy()

obs["Year_num"] = obs["Year_num"].astype(int)

# Repeated records in the same well-season-year are not silently
# averaged: stop and inspect them if present.
dup_mask = obs.duplicated(
    ["Well_key", "Season_clean", "Year_num"], keep=False
)

if dup_mask.any():
    obs.loc[dup_mask].sort_values(
        ["Well_key", "Season_clean", "Year_num"]
    ).to_csv(OUT / "00_duplicate_well_season_year_records.csv", index=False)

    raise ValueError(
        "Duplicate numeric observations exist for a well-season-year. "
        "Review 00_duplicate_well_season_year_records.csv before continuing."
    )

# ------------------------------------------------------------
# 3. Prepare baseline results
# ------------------------------------------------------------
step7["Season_clean"] = step7["Season"].astype(str).str.strip().str.lower()
step8["Season_clean"] = step8["Season"].astype(str).str.strip().str.lower()

for df in [step7, step8]:
    df["Well_key"] = df["Well_key"].astype(str).str.strip()

baseline_cols = [
    "Well_key", "Season_clean", "sen_slope_m_per_year",
    "mk_p_value", "mk_trend"
]
baseline7 = step7[baseline_cols].copy()

if "mk_p_fdr_bh" in step8.columns:
    baseline8 = step8[
        ["Well_key", "Season_clean", "mk_p_fdr_bh",
         "trend_classification"]
    ].copy()
else:
    baseline8 = step8[
        ["Well_key", "Season_clean", "trend_classification"]
    ].copy()
    baseline8["mk_p_fdr_bh"] = np.nan

# ------------------------------------------------------------
# 4. Run robustness analysis
# ------------------------------------------------------------
records = []

for (well, season), g in obs.groupby(
    ["Well_key", "Season_clean"], sort=True
):
    g = g.sort_values("Year_num")
    years = g["Year_num"].to_numpy(dtype=int)
    y = g["DTW_num"].to_numpy(dtype=float)

    n = len(y)
    year_span = int(years[-1] - years[0]) if n else 0
    gaps = np.diff(years)
    consecutive_years = bool(n >= 2 and np.all(gaps == 1))

    row = {
        "Well_key": well,
        "Season": season,
        "n_observations": n,
        "first_year": int(years[0]),
        "last_year": int(years[-1]),
        "year_span": year_span,
        "consecutive_annual_series": consecutive_years,
        "max_gap_years": int(gaps.max()) if len(gaps) else np.nan,
        "original_scale_sen_slope_m_per_year": np.nan,
        "recomputed_original_mk_p": np.nan,
        "tfpw_test_applied": False,
        "tfpw_mk_p": np.nan,
        "tfpw_trend": "not assessed",
        "tfpw_reason": "",
        "tfpw_sen_slope_m_per_year": np.nan,
        "analysis_warning": ""
    }

    # Retain short series, but do not claim inferential testing.
    if n < 8 or year_span < 7:
        row["analysis_warning"] = (
            "Insufficient observations or time span for this robustness screen"
        )
        records.append(row)
        continue

    # Original MK and Sen slope on observed sequence.
    # Original MK does not account for serial dependence.
    try:
        original_test = mk.original_test(y)
        row["recomputed_original_mk_p"] = float(original_test.p)
        row["original_mk_trend_recomputed"] = original_test.trend

        row["original_scale_sen_slope_m_per_year"] = float(
            theilslopes(y, years)[0]
        )
    except Exception as e:
        row["analysis_warning"] = f"Original test error: {e}"
        records.append(row)
        continue

    # TFPW implementation assumes equally spaced observations.
    # Apply only where every year is represented.
    if not consecutive_years:
        row["tfpw_reason"] = (
            "Not applied: missing years make observations unequally spaced. "
            "Use an irregular-time-series method or a justified gap strategy."
        )
        records.append(row)
        continue

    try:
        tfpw_test = mk.trend_free_pre_whitening_modification_test(y)

        row["tfpw_test_applied"] = True
        row["tfpw_mk_p"] = float(tfpw_test.p)
        row["tfpw_trend"] = str(tfpw_test.trend)
        row["tfpw_reason"] = (
            "Applied as a sensitivity test to consecutive annual observations"
        )

        # This slope is estimated on the pre-whitened transformed series.
        # It is NOT the preferred physical trend magnitude.
        row["tfpw_sen_slope_m_per_year"] = float(tfpw_test.slope)

    except Exception as e:
        row["tfpw_reason"] = f"TFPW test error: {e}"

    records.append(row)

robust = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. Merge baseline results
# ------------------------------------------------------------
robust = robust.merge(
    baseline7,
    on=["Well_key", "Season_clean"] if "Season_clean" in robust.columns
    else ["Well_key", "Season"],
    how="left"
) if False else robust

# Normalize season key for safe merging
robust["Season_clean"] = robust["Season"].astype(str).str.lower().str.strip()

robust = robust.merge(
    baseline7,
    on=["Well_key", "Season_clean"],
    how="left",
    validate="one_to_one"
)

robust = robust.merge(
    baseline8,
    on=["Well_key", "Season_clean"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_step8")
)

# ------------------------------------------------------------
# 6. Conservative robustness classification
# ------------------------------------------------------------
def robust_classification(r):
    if not r["tfpw_test_applied"]:
        return "NOT_ASSESSED_FOR_TFPW"

    p = r["tfpw_mk_p"]
    slope = r["original_scale_sen_slope_m_per_year"]

    if pd.isna(p) or pd.isna(slope):
        return "REVIEW_MISSING_STATISTICS"

    if p >= 0.05:
        return "TFPW_NOT_SIGNIFICANT_AT_0.05"

    if slope > 0:
        return "TFPW_SIGNIFICANT_DTW_INCREASE"
    if slope < 0:
        return "TFPW_SIGNIFICANT_DTW_DECREASE"

    return "TFPW_SIGNIFICANT_ZERO_SLOPE_REVIEW"

robust["tfpw_robustness_classification"] = robust.apply(
    robust_classification, axis=1
)

robust["original_and_tfpw_direction_agree"] = np.where(
    robust["tfpw_test_applied"],
    np.sign(robust["original_scale_sen_slope_m_per_year"]) ==
    np.sign(robust["sen_slope_m_per_year"]),
    np.nan
)

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------
robust.to_csv(OUT / "02_robustness_comparison.csv", index=False)

review = robust.loc[
    (~robust["tfpw_test_applied"]) |
    robust["analysis_warning"].ne("") |
    robust["tfpw_reason"].str.startswith("TFPW test error")
].copy()

review.to_csv(OUT / "01_serial_dependence_review.csv", index=False)

classification_counts = (
    robust.groupby("tfpw_robustness_classification", dropna=False)
    .size()
    .rename("series_count")
    .reset_index()
)
classification_counts.to_csv(
    OUT / "03_robust_trend_classification.csv", index=False
)

summary = {
    "series_in_robustness_table": int(len(robust)),
    "series_with_original_mk_p": int(
        robust["recomputed_original_mk_p"].notna().sum()
    ),
    "series_eligible_for_tfpw": int(robust["tfpw_test_applied"].sum()),
    "series_not_assessed_for_tfpw": int((~robust["tfpw_test_applied"]).sum()),
    "tfpw_significant_dtw_increase": int(
        (robust["tfpw_robustness_classification"] ==
         "TFPW_SIGNIFICANT_DTW_INCREASE").sum()
    ),
    "tfpw_significant_dtw_decrease": int(
        (robust["tfpw_robustness_classification"] ==
         "TFPW_SIGNIFICANT_DTW_DECREASE").sum()
    ),
    "tfpw_not_significant": int(
        (robust["tfpw_robustness_classification"] ==
         "TFPW_NOT_SIGNIFICANT_AT_0.05").sum()
    ),
    "important_limitations": [
        "TFPW is applied only to consecutive annual series.",
        "TFPW p-values here have not been FDR-adjusted.",
        "The test is a sensitivity analysis, not proof that autocorrelation is fully resolved.",
        "Do not use the slope from the pre-whitened series as the physical trend magnitude.",
        "The original Sen slope remains the reported trend magnitude pending further validation."
    ]
}

with open(OUT / "04_step9_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 72)
print("STEP 9 COMPLETED")
print("=" * 72)
print(json.dumps(summary, indent=2))
print("\nClassification counts:")
display(classification_counts)
print("\nOutput folder:", OUT)
print("\nFiles:")
for f in sorted(OUT.iterdir()):
    print("-", f.name)


**[orig #109]** Step 10 — export unmatched keys; check coverage and p-values  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP7 = ROOT / "Step_7_Seasonal_Trend_Analysis"
STEP8 = ROOT / "Step_8_Trend_Validation"
STEP9 = ROOT / "Step_9_Serial_Dependence_Robustness"

f7 = STEP7 / "04_exploratory_well_season_trends.csv"
f8 = STEP8 / "01_validated_well_season_trends.csv"
f9 = STEP9 / "02_robustness_comparison.csv"

for f in [f7, f8, f9]:
    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")

d7 = pd.read_csv(f7)
d8 = pd.read_csv(f8)
d9 = pd.read_csv(f9)

def prepare_keys(df, label):
    required = ["Well_key", "Season"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{label} missing columns: {missing}")

    out = df[["Well_key", "Season"]].copy()
    out["Well_key"] = out["Well_key"].astype(str).str.strip()
    out["Season"] = out["Season"].astype(str).str.strip().str.lower()
    out["source_table"] = label
    return out

k7 = prepare_keys(d7, "Step7")
k8 = prepare_keys(d8, "Step8")
k9 = prepare_keys(d9, "Step9")

def key_set(df):
    return set(zip(df["Well_key"], df["Season"]))

s7, s8, s9 = key_set(k7), key_set(k8), key_set(k9)

print("=" * 65)
print("STEP 10: SERIES POPULATION RECONCILIATION")
print("=" * 65)
print("Step 7 rows / unique keys:", len(d7), len(s7))
print("Step 8 rows / unique keys:", len(d8), len(s8))
print("Step 9 rows / unique keys:", len(d9), len(s9))
print()
print("Step 7 keys missing from Step 9:", len(s7 - s9))
print("Step 9 keys absent from Step 7:", len(s9 - s7))
print("Step 7 and Step 8 key differences:", len(s7.symmetric_difference(s8)))

# Export the actual unmatched keys
extra9 = sorted(s9 - s7)
missing9 = sorted(s7 - s9)

pd.DataFrame(extra9, columns=["Well_key", "Season"]).to_csv(
    STEP9 / "05_series_present_in_step9_not_step7.csv", index=False
)
pd.DataFrame(missing9, columns=["Well_key", "Season"]).to_csv(
    STEP9 / "06_series_present_in_step7_not_step9.csv", index=False
)

# Check test coverage and p-values in Step 9
tfpw_applied = (
    d9["tfpw_test_applied"].astype(str).str.lower().isin(["true", "1"])
)

tfpw_p = pd.to_numeric(d9["tfpw_mk_p"], errors="coerce")
orig_p = pd.to_numeric(d9["recomputed_original_mk_p"], errors="coerce")

print("\nStep 9 test coverage:")
print("TFPW applied:", int(tfpw_applied.sum()))
print("TFPW p-values present:", int(tfpw_p.notna().sum()))
print("Original MK p-values present:", int(orig_p.notna().sum()))

if "tfpw_robustness_classification" in d9.columns:
    print("\nTFPW classification counts:")
    print(d9["tfpw_robustness_classification"].value_counts(dropna=False))

# Check whether the trend baseline merged successfully
if "mk_p_value" in d9.columns:
    print(
        "\nStep 7 baseline p-values present after merge:",
        int(pd.to_numeric(d9["mk_p_value"], errors="coerce").notna().sum())
    )

print("\nAudit files saved in:", STEP9)


**[orig #110]** Step 10 — restrict Step 9 results to the 186 Step 7 candidates  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP7 = ROOT / "Step_7_Seasonal_Trend_Analysis"
STEP8 = ROOT / "Step_8_Trend_Validation"
STEP9 = ROOT / "Step_9_Serial_Dependence_Robustness"

TREND7 = STEP7 / "04_exploratory_well_season_trends.csv"
TREND8 = STEP8 / "01_validated_well_season_trends.csv"
ROBUST9 = STEP9 / "02_robustness_comparison.csv"

OUT = ROOT / "Step_10A_Candidate_Population_Correction"
OUT.mkdir(parents=True, exist_ok=True)

for p in [TREND7, TREND8, ROBUST9]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

s7 = pd.read_csv(TREND7)
s8 = pd.read_csv(TREND8)
s9 = pd.read_csv(ROBUST9)

def standardize_keys(df):
    df = df.copy()
    df["Well_key"] = df["Well_key"].astype(str).str.strip()
    df["Season_clean"] = df["Season"].astype(str).str.strip().str.lower()
    return df

s7 = standardize_keys(s7)
s8 = standardize_keys(s8)
s9 = standardize_keys(s9)

key = ["Well_key", "Season_clean"]

if s7.duplicated(key).any():
    raise ValueError("Step 7 contains duplicate candidate keys.")
if s8.duplicated(key).any():
    raise ValueError("Step 8 contains duplicate candidate keys.")
if s9.duplicated(key).any():
    raise ValueError("Step 9 contains duplicate keys.")

# Step 7 defines the master population: exactly 186 series.
master = s7[key].drop_duplicates()

# Keep only Step 9 results belonging to the Step 7 candidates.
corrected = master.merge(
    s9,
    on=key,
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Master candidate series:", len(master))
print("Matched Step 9 series:", int((corrected["_merge"] == "both").sum()))
print("Missing Step 9 candidate results:", int((corrected["_merge"] != "both").sum()))

corrected.drop(columns=["_merge"], inplace=True)

# Add Step 8 FDR results from the same candidate population.
step8_cols = [
    "Well_key", "Season_clean",
    "mk_p_fdr_bh", "fdr_significant_05",
    "trend_classification"
]
step8_cols = [c for c in step8_cols if c in s8.columns]

corrected = corrected.merge(
    s8[step8_cols],
    on=key,
    how="left",
    validate="one_to_one",
    suffixes=("", "_step8")
)

# Compare the existing TFPW results with the original candidate series.
if "tfpw_test_applied" in corrected.columns:
    corrected["tfpw_test_applied"] = (
        corrected["tfpw_test_applied"]
        .astype(str).str.lower().isin(["true", "1"])
    )

if "tfpw_mk_p" in corrected.columns:
    corrected["tfpw_mk_p"] = pd.to_numeric(
        corrected["tfpw_mk_p"], errors="coerce"
    )

if "tfpw_test_applied" in corrected.columns:
    corrected["tfpw_p_fdr_bh"] = np.nan

    # FDR correction only among TFPW tests in the candidate population.
    eligible = (
        corrected["tfpw_test_applied"] &
        corrected["tfpw_mk_p"].notna() &
        corrected["tfpw_mk_p"].between(0, 1)
    )

    p = corrected.loc[eligible, "tfpw_mk_p"].to_numpy()
    m = len(p)

    if m:
        order = np.argsort(p)
        sorted_p = p[order]
        ranks = np.arange(1, m + 1)

        adjusted_sorted = sorted_p * m / ranks
        adjusted_sorted = np.minimum.accumulate(
            adjusted_sorted[::-1]
        )[::-1]
        adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

        adjusted = np.empty(m)
        adjusted[order] = adjusted_sorted
        corrected.loc[eligible, "tfpw_p_fdr_bh"] = adjusted

    corrected["tfpw_fdr_significant_05"] = (
        corrected["tfpw_p_fdr_bh"].notna() &
        (corrected["tfpw_p_fdr_bh"] < 0.05)
    )

corrected.to_csv(
    OUT / "01_corrected_186_candidate_robustness_comparison.csv",
    index=False
)

# Export the 393 extra Step 9 series separately for traceability.
extra = s9.merge(master, on=key, how="left", indicator=True)
extra = extra.loc[extra["_merge"] == "left_only"].drop(columns=["_merge"])
extra.to_csv(OUT / "02_excluded_extra_step9_series.csv", index=False)

summary = {
    "step7_master_series": int(len(master)),
    "corrected_candidate_rows": int(len(corrected)),
    "candidate_rows_missing_step9_result": int(
        corrected.get("tfpw_test_applied", pd.Series(False, index=corrected.index))
        .eq(False).sum()
    ),
    "extra_step9_series_excluded": int(len(extra)),
    "tfpw_tested_within_candidate_population": int(
        corrected["tfpw_test_applied"].sum()
    ) if "tfpw_test_applied" in corrected.columns else 0,
    "tfpw_p_values_fdr_adjusted_within_candidates": int(
        corrected["tfpw_p_fdr_bh"].notna().sum()
    ) if "tfpw_p_fdr_bh" in corrected.columns else 0,
    "note": (
        "The existing TFPW calculations are filtered to the original Step 7 "
        "population. This correction does not recalculate TFPW from the "
        "candidate observations; inspect missing candidate test results "
        "before treating the comparison as final."
    )
}

with open(OUT / "03_population_correction_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSTEP 10A COMPLETE")
print(json.dumps(summary, indent=2))
print("\nOutput folder:", OUT)


**[orig #111]** Step 10B — recompute TFPW on the 186 candidates  
<sub>✅ ran OK</sub>

In [ ]:

%pip -q install pymannkendall

from pathlib import Path
import json
import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import theilslopes

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

OBS_FILE = (
    ROOT / "Step_2_QC" /
    "estimated_heads_deduplicated_analysis_copy.csv"
)
STEP7_FILE = (
    ROOT / "Step_7_Seasonal_Trend_Analysis" /
    "04_exploratory_well_season_trends.csv"
)
OUT = ROOT / "Step_10B_Candidate_TFPW_Recalculation"
OUT.mkdir(parents=True, exist_ok=True)

for p in [OBS_FILE, STEP7_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input: {p}")

# 1. Load the exact candidate list from Step 7
candidates = pd.read_csv(STEP7_FILE)
candidates["Well_key"] = candidates["Well_key"].astype(str).str.strip()
candidates["Season_clean"] = (
    candidates["Season"].astype(str).str.strip().str.lower()
)

keys = ["Well_key", "Season_clean"]
if candidates.duplicated(keys).any():
    raise ValueError("Step 7 contains duplicate well-season keys.")

# 2. Load source observations
obs = pd.read_csv(OBS_FILE)

obs["Well_key"] = (
    obs["Source_file"].astype(str).str.strip() + "|" +
    obs["Source_sheet"].astype(str).str.strip() + "|" +
    obs["Source_row_number"].astype(str).str.strip()
)
obs["Season_clean"] = obs["Season"].astype(str).str.strip().str.lower()
obs["District_clean"] = obs["District"].astype(str).str.strip().str.lower()
obs["Year_num"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["DTW_num"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)

obs = obs.loc[
    (obs["District_clean"] == "alwar") &
    obs["Season_clean"].isin(["pre", "post"]) &
    obs["Year_num"].notna() &
    obs["DTW_num"].notna()
].copy()
obs["Year_num"] = obs["Year_num"].astype(int)

# Restrict observations to the 186 Step 7 candidates
candidate_key_set = set(zip(candidates["Well_key"], candidates["Season_clean"]))
obs = obs.loc[
    obs.apply(
        lambda r: (r["Well_key"], r["Season_clean"]) in candidate_key_set,
        axis=1
    )
].copy()

# Do not silently average repeated observations
dup = obs.duplicated(
    ["Well_key", "Season_clean", "Year_num"], keep=False
)
if dup.any():
    obs.loc[dup].to_csv(
        OUT / "00_duplicate_candidate_year_records.csv", index=False
    )
    raise ValueError(
        "Duplicate candidate well-season-year records found. "
        "Review 00_duplicate_candidate_year_records.csv."
    )

# 3. Calculate results for every candidate, retaining non-tested series
records = []

for _, c in candidates.iterrows():
    well = c["Well_key"]
    season = c["Season_clean"]

    g = obs.loc[
        (obs["Well_key"] == well) &
        (obs["Season_clean"] == season)
    ].sort_values("Year_num")

    years = g["Year_num"].to_numpy(dtype=int)
    y = g["DTW_num"].to_numpy(dtype=float)
    n = len(y)

    row = {
        "Well_key": well,
        "Season": season,
        "step7_n_years": c.get("n_years", np.nan),
        "step7_year_span": c.get("year_span", np.nan),
        "n_observations_recalculated": n,
        "first_year": int(years.min()) if n else np.nan,
        "last_year": int(years.max()) if n else np.nan,
        "year_span_recalculated": int(years.max() - years.min()) if n else np.nan,
        "consecutive_annual_observations": False,
        "original_mk_p_recalculated": np.nan,
        "original_sen_slope_m_per_year": np.nan,
        "tfpw_applied": False,
        "tfpw_p": np.nan,
        "tfpw_trend": "not assessed",
        "tfpw_reason": "",
    }

    if n < 8:
        row["tfpw_reason"] = "Fewer than 8 numeric observations"
        records.append(row)
        continue

    span = int(years.max() - years.min())
    if span < 7:
        row["tfpw_reason"] = "Year span shorter than 7 years"
        records.append(row)
        continue

    consecutive = bool(np.all(np.diff(years) == 1))
    row["consecutive_annual_observations"] = consecutive

    try:
        original = mk.original_test(y)
        row["original_mk_p_recalculated"] = float(original.p)
        row["original_sen_slope_m_per_year"] = float(
            theilslopes(y, years)[0]
        )
    except Exception as e:
        row["tfpw_reason"] = f"Original MK error: {e}"
        records.append(row)
        continue

    if not consecutive:
        row["tfpw_reason"] = (
            "Not tested: missing annual observations; "
            "equal-spacing TFPW assumption not satisfied"
        )
        records.append(row)
        continue

    try:
        test = mk.trend_free_pre_whitening_modification_test(y)
        row["tfpw_applied"] = True
        row["tfpw_p"] = float(test.p)
        row["tfpw_trend"] = str(test.trend)
        row["tfpw_reason"] = "TFPW sensitivity test completed"
    except Exception as e:
        row["tfpw_reason"] = f"TFPW calculation error: {e}"

    records.append(row)

res = pd.DataFrame(records)

# 4. Adjust TFPW p-values for multiple testing within the tested candidates
res["tfpw_p_fdr_bh"] = np.nan
res["tfpw_fdr_significant_05"] = False

valid = (
    res["tfpw_applied"] &
    res["tfpw_p"].notna() &
    res["tfpw_p"].between(0, 1)
)
p = res.loc[valid, "tfpw_p"].to_numpy()
m = len(p)

if m:
    order = np.argsort(p)
    sorted_p = p[order]
    ranks = np.arange(1, m + 1)

    adj_sorted = sorted_p * m / ranks
    adj_sorted = np.minimum.accumulate(adj_sorted[::-1])[::-1]
    adj_sorted = np.clip(adj_sorted, 0, 1)

    adj = np.empty(m)
    adj[order] = adj_sorted

    res.loc[valid, "tfpw_p_fdr_bh"] = adj
    res["tfpw_fdr_significant_05"] = (
        res["tfpw_p_fdr_bh"].notna() &
        (res["tfpw_p_fdr_bh"] < 0.05)
    )

# 5. Save outputs
res.to_csv(
    OUT / "01_candidate_tfpw_results.csv", index=False
)

not_tested = res.loc[~res["tfpw_applied"]].copy()
not_tested.to_csv(
    OUT / "02_candidates_not_tfpw_tested.csv", index=False
)

counts = (
    res.groupby(["tfpw_applied", "tfpw_reason"], dropna=False)
    .size()
    .rename("series_count")
    .reset_index()
)
counts.to_csv(OUT / "03_test_eligibility_counts.csv", index=False)

summary = {
    "step7_candidate_series": int(len(candidates)),
    "candidate_results_returned": int(len(res)),
    "tfpw_tested": int(res["tfpw_applied"].sum()),
    "tfpw_not_tested": int((~res["tfpw_applied"]).sum()),
    "tfpw_p_values_fdr_adjusted": int(res["tfpw_p_fdr_bh"].notna().sum()),
    "tfpw_fdr_significant": int(res["tfpw_fdr_significant_05"].sum()),
    "tfpw_fdr_significant_dtw_increase": int(
        (
            res["tfpw_fdr_significant_05"] &
            (res["original_sen_slope_m_per_year"] > 0)
        ).sum()
    ),
    "tfpw_fdr_significant_dtw_decrease": int(
        (
            res["tfpw_fdr_significant_05"] &
            (res["original_sen_slope_m_per_year"] < 0)
        ).sum()
    ),
    "note": (
        "TFPW is used as a sensitivity test only for consecutive annual "
        "series. FDR applies to the TFPW tests that were actually performed. "
        "Non-tested series are not classified as having no trend."
    )
}

with open(OUT / "04_step10b_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 70)
print("STEP 10B COMPLETED")
print("=" * 70)
print(json.dumps(summary, indent=2))
print("\nTest eligibility counts:")
display(counts)
print("\nOutput folder:", OUT)


**[orig #112]** Step 11 — compare Step 8 (MK/FDR) and Step 10B (TFPW/FDR) significance  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP8 = ROOT / "Step_8_Trend_Validation"
STEP10B = ROOT / "Step_10B_Candidate_TFPW_Recalculation"
OUT = ROOT / "Step_11_Trend_Robustness_Comparison"
OUT.mkdir(parents=True, exist_ok=True)

F8 = STEP8 / "01_validated_well_season_trends.csv"
F10 = STEP10B / "01_candidate_tfpw_results.csv"

for f in [F8, F10]:
    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")

s8 = pd.read_csv(F8)
s10 = pd.read_csv(F10)

def prep(df):
    df = df.copy()
    df["Well_key"] = df["Well_key"].astype(str).str.strip()
    df["Season_clean"] = df["Season"].astype(str).str.strip().str.lower()
    return df

s8 = prep(s8)
s10 = prep(s10)
keys = ["Well_key", "Season_clean"]

if s8.duplicated(keys).any() or s10.duplicated(keys).any():
    raise ValueError("Duplicate well-season keys detected.")

# Use Step 8 as the master population.
comparison = s8.merge(
    s10,
    on=keys,
    how="left",
    suffixes=("_step8", "_step10b"),
    validate="one_to_one",
    indicator=True
)

print("Step 8 series:", len(s8))
print("Step 10B series:", len(s10))
print("Matched series:", int((comparison["_merge"] == "both").sum()))
print("Unmatched series:", int((comparison["_merge"] != "both").sum()))

# Step 8 original MK/FDR significance
comparison["step8_fdr_p"] = pd.to_numeric(
    comparison["mk_p_fdr_bh"], errors="coerce"
)
comparison["step8_fdr_significant"] = (
    comparison["step8_fdr_p"] < 0.05
)

# Step 10B TFPW/FDR significance
comparison["tfpw_fdr_p"] = pd.to_numeric(
    comparison["tfpw_p_fdr_bh"], errors="coerce"
)
comparison["tfpw_tested"] = (
    comparison["tfpw_applied"].astype(str).str.lower().isin(["true", "1"])
)
comparison["tfpw_fdr_significant"] = (
    comparison["tfpw_tested"] &
    comparison["tfpw_fdr_p"].notna() &
    (comparison["tfpw_fdr_p"] < 0.05)
)

# Use original Sen slope for physical trend direction.
slope_col = "sen_slope_m_per_year"
comparison["original_sen_slope"] = pd.to_numeric(
    comparison[slope_col], errors="coerce"
)

comparison["original_direction"] = np.select(
    [
        comparison["original_sen_slope"] > 0,
        comparison["original_sen_slope"] < 0
    ],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_UNKNOWN"
)

# TFPW trend direction from the recalculated original-scale Sen slope
tfpw_slope = pd.to_numeric(
    comparison["original_sen_slope_m_per_year"], errors="coerce"
)
comparison["tfpw_input_direction"] = np.select(
    [tfpw_slope > 0, tfpw_slope < 0],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_UNKNOWN"
)

# The same source Sen slope may be available in both tables.
# Compare directions using Step 8 slope vs Step 10B original-scale slope.
comparison["direction_agrees"] = np.where(
    comparison["tfpw_tested"] &
    comparison["original_sen_slope"].notna() &
    tfpw_slope.notna(),
    np.sign(comparison["original_sen_slope"]) == np.sign(tfpw_slope),
    np.nan
)

def classification(r):
    if not r["tfpw_tested"]:
        return "NOT_ASSESSED_BY_TFPW"

    orig_sig = bool(r["step8_fdr_significant"])
    tfpw_sig = bool(r["tfpw_fdr_significant"])
    agree = r["direction_agrees"]

    if orig_sig and tfpw_sig and agree is True:
        return "SIGNIFICANT_IN_BOTH_METHODS_DIRECTION_AGREES"
    if orig_sig and tfpw_sig:
        return "SIGNIFICANT_IN_BOTH_METHODS_DIRECTION_REVIEW"
    if orig_sig and not tfpw_sig:
        return "ORIGINAL_ONLY_TFPW_NOT_FDR_SIGNIFICANT"
    if not orig_sig and tfpw_sig:
        return "TFPW_ONLY_FDR_SIGNIFICANT"
    return "NOT_FDR_SIGNIFICANT_IN_EITHER_METHOD"

comparison["robustness_classification"] = comparison.apply(
    classification, axis=1
)

comparison.drop(columns=["_merge"], inplace=True)

comparison.to_csv(
    OUT / "01_step8_vs_step10b_comparison.csv", index=False
)

counts = (
    comparison.groupby("robustness_classification", dropna=False)
    .size().rename("series_count").reset_index()
)
counts.to_csv(OUT / "02_robustness_classification_counts.csv", index=False)

tested = comparison.loc[comparison["tfpw_tested"]].copy()

summary = {
    "candidate_series": int(len(comparison)),
    "tfpw_tested": int(comparison["tfpw_tested"].sum()),
    "tfpw_not_tested": int((~comparison["tfpw_tested"]).sum()),
    "step8_fdr_significant": int(comparison["step8_fdr_significant"].sum()),
    "tfpw_fdr_significant": int(comparison["tfpw_fdr_significant"].sum()),
    "tested_series_significant_in_both": int(
        (
            tested["step8_fdr_significant"] &
            tested["tfpw_fdr_significant"]
        ).sum()
    ),
    "tested_series_significant_only_step8": int(
        (
            tested["step8_fdr_significant"] &
            ~tested["tfpw_fdr_significant"]
        ).sum()
    ),
    "tested_series_significant_only_tfpw": int(
        (
            ~tested["step8_fdr_significant"] &
            tested["tfpw_fdr_significant"]
        ).sum()
    ),
    "note": (
        "This is a method comparison, not proof of causal attribution. "
        "TFPW was applied only to eligible consecutive annual series. "
        "Interpret direction agreement cautiously and inspect source slopes "
        "and missing-year patterns before final classification."
    )
}

with open(OUT / "03_step11_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSTEP 11 COMPLETE")
print(json.dumps(summary, indent=2))
print("\nRobustness classifications:")
display(counts)
print("\nOutput folder:", OUT)


**[orig #114]** Step 12 — join slopes and significance into one table  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

F8 = ROOT / "Step_8_Trend_Validation" / \
     "01_validated_well_season_trends.csv"
F10 = ROOT / "Step_10B_Candidate_TFPW_Recalculation" / \
      "01_candidate_tfpw_results.csv"

OUT = ROOT / "Step_12_Trend_Direction_QC"
OUT.mkdir(parents=True, exist_ok=True)

s8 = pd.read_csv(F8)
s10 = pd.read_csv(F10)

def prepare(df, label):
    df = df.copy()
    if not {"Well_key", "Season"}.issubset(df.columns):
        raise ValueError(
            f"{label}: expected Well_key and Season. "
            f"Columns found: {df.columns.tolist()}"
        )
    df["Well_key"] = df["Well_key"].astype(str).str.strip()
    df["Season_clean"] = (
        df["Season"].astype(str).str.strip().str.lower()
    )
    return df

s8 = prepare(s8, "Step 8")
s10 = prepare(s10, "Step 10B")

keys = ["Well_key", "Season_clean"]

if s8.duplicated(keys).any() or s10.duplicated(keys).any():
    raise ValueError("Duplicate well-season keys detected.")

# Rename fields before joining to avoid pandas suffix ambiguity.
left = s8[keys].copy()
left["step8_sen_slope"] = pd.to_numeric(
    s8["sen_slope_m_per_year"], errors="coerce"
)
left["step8_fdr_p"] = pd.to_numeric(
    s8["mk_p_fdr_bh"], errors="coerce"
)

right = s10[keys].copy()
right["step10b_original_sen_slope"] = pd.to_numeric(
    s10["original_sen_slope_m_per_year"], errors="coerce"
)
right["tfpw_applied"] = (
    s10["tfpw_applied"].astype(str).str.lower().isin(["true", "1"])
)
right["tfpw_p_fdr_bh"] = pd.to_numeric(
    s10["tfpw_p_fdr_bh"], errors="coerce"
)

d = left.merge(
    right, on=keys, how="outer",
    validate="one_to_one", indicator=True
)

print("Merge status:")
print(d["_merge"].value_counts())

if len(d) != 186 or not (d["_merge"] == "both").all():
    raise ValueError(
        "The Step 8 and Step 10B populations do not match exactly. "
        "Inspect merge status before continuing."
    )

d.drop(columns="_merge", inplace=True)

slope8 = d["step8_sen_slope"]
slope10 = d["step10b_original_sen_slope"]

d["step8_direction"] = np.select(
    [slope8 > 0, slope8 < 0],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_MISSING"
)
d["step10b_direction"] = np.select(
    [slope10 > 0, slope10 < 0],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_MISSING"
)

d["direction_agreement"] = np.where(
    slope8.notna() & slope10.notna(),
    np.sign(slope8) == np.sign(slope10),
    np.nan
)
d["absolute_slope_difference_m_per_year"] = (
    slope8 - slope10
).abs()

d["step8_fdr_significant"] = d["step8_fdr_p"] < 0.05
d["tfpw_fdr_significant"] = (
    d["tfpw_applied"] &
    d["tfpw_p_fdr_bh"].notna() &
    (d["tfpw_p_fdr_bh"] < 0.05)
)
d["significant_in_both"] = (
    d["step8_fdr_significant"] &
    d["tfpw_fdr_significant"]
)

def classify(r):
    if not r["significant_in_both"]:
        return "NOT_SIGNIFICANT_IN_BOTH_METHODS"

    if pd.isna(r["direction_agreement"]):
        return "BOTH_SIGNIFICANT_DIRECTION_UNAVAILABLE"

    if not bool(r["direction_agreement"]):
        return "BOTH_SIGNIFICANT_DIRECTION_DISAGREEMENT"

    return "BOTH_SIGNIFICANT_DIRECTION_AGREES"

d["direction_qc_classification"] = d.apply(classify, axis=1)

d.to_csv(OUT / "01_candidate_direction_audit.csv", index=False)

review = d.loc[
    d["significant_in_both"] |
    (d["direction_agreement"] == False) |
    d["absolute_slope_difference_m_per_year"].gt(0.5)
].copy()
review.to_csv(OUT / "02_manual_trend_review.csv", index=False)

counts = (
    d["direction_qc_classification"]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="series_count")
)
counts.to_csv(OUT / "03_direction_qc_counts.csv", index=False)

summary = {
    "candidate_series": int(len(d)),
    "direction_agreement_count": int(
        (d["direction_agreement"] == True).sum()
    ),
    "direction_disagreement_count": int(
        (d["direction_agreement"] == False).sum()
    ),
    "direction_unavailable_count": int(
        d["direction_agreement"].isna().sum()
    ),
    "significant_in_both_methods": int(d["significant_in_both"].sum()),
    "both_significant_direction_agrees": int(
        (d["direction_qc_classification"] ==
         "BOTH_SIGNIFICANT_DIRECTION_AGREES").sum()
    ),
    "both_significant_direction_disagrees": int(
        (d["direction_qc_classification"] ==
         "BOTH_SIGNIFICANT_DIRECTION_DISAGREEMENT").sum()
    ),
    "note": (
        "Direction comparison is a diagnostic, not causal attribution. "
        "Both slope columns should be estimates on the original DTW scale."
    )
}

with open(OUT / "04_step12_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSTEP 12 COMPLETE")
print(json.dumps(summary, indent=2))
print("\nClassification counts:")
display(counts)
print("\nOutput folder:", OUT)


## 2. Well coordinates (Step 13)

Cell 115 stops with an error, but only after it creates `alwar`, which cell 116 uses. Run both.

**[orig #115]** Step 13 — recreate well identifier, parse coordinates (stops with KeyError *after* creating `alwar`, which the next cell uses)  
<sub>⚠ last run ended with KeyError</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

OBS_FILE = (
    ROOT / "Step_2_QC" /
    "estimated_heads_deduplicated_analysis_copy.csv"
)
TREND_FILE = (
    ROOT / "Step_12_Trend_Direction_QC" /
    "01_candidate_direction_audit.csv"
)

OUT = ROOT / "Step_13_Spatial_Validation"
OUT.mkdir(parents=True, exist_ok=True)

for f in [OBS_FILE, TREND_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")

obs = pd.read_csv(OBS_FILE)
trends = pd.read_csv(TREND_FILE)

# Recreate the well identifier used by the trend analysis.
obs["Well_key"] = (
    obs["Source_file"].astype(str).str.strip() + "|" +
    obs["Source_sheet"].astype(str).str.strip() + "|" +
    obs["Source_row_number"].astype(str).str.strip()
)

obs["District_clean"] = (
    obs["District"].astype(str).str.strip().str.lower()
)

# Parse coordinates.
obs["Latitude_num"] = pd.to_numeric(obs["Latitude"], errors="coerce")
obs["Longitude_num"] = pd.to_numeric(obs["Longitude"], errors="coerce")

# Limit to Alwar wells and summarize coordinate consistency.
alwar = obs.loc[obs["District_clean"] == "alwar"].copy()

well_coords = (
    alwar.groupby("Well_key", dropna=False)
    .agg(
        district=("District", "first"),
        block=("Block", "first"),
        village=("Village", "first"),
        latitude_min=("Latitude_num", "min"),
        latitude_max=("Latitude_num", "max"),
        longitude_min=("Longitude_num", "min"),
        longitude_max=("Longitude_num", "max"),
        coordinate_records=("Latitude_num", "count"),
        total_records=("Well_key", "size")
    )
    .reset_index()
)

well_coords["coordinate_consistent"] = (
    (well_coords["latitude_min"] == well_coords["latitude_max"]) &
    (well_coords["longitude_min"] == well_coords["longitude_max"])
)

well_coords["latitude"] = well_coords["latitude_min"]
well_coords["longitude"] = well_coords["longitude_min"]

well_coords["coordinate_status"] = np.select(
    [
        well_coords["latitude"].isna() |
        well_coords["longitude"].isna(),
        ~well_coords["coordinate_consistent"],
        ~well_coords["latitude"].between(-90, 90) |
        ~well_coords["longitude"].between(-180, 180)
    ],
    [
        "MISSING_COORDINATES",
        "INCONSISTENT_COORDINATES",
        "INVALID_COORDINATE_RANGE"
    ],
    default="COORDINATES_PRESENT_BOUNDARY_NOT_YET_VERIFIED"
)

# Standardize trend keys.
trends["Well_key"] = trends["Well_key"].astype(str).str.strip()
trends["Season_clean"] = (
    trends["Season"].astype(str).str.strip().str.lower()
)

# Keep one row per candidate well-season series.
final = trends.merge(
    well_coords,
    on="Well_key",
    how="left",
    validate="many_to_one",
    indicator=True
)

final["coordinate_match_status"] = np.where(
    final["_merge"] == "both",
    final["coordinate_status"].fillna("COORDINATE_REVIEW"),
    "WELL_KEY_NOT_FOUND_IN_ALWAR_OBSERVATIONS"
)
final.drop(columns=["_merge"], inplace=True)

final.to_csv(
    OUT / "01_candidate_trends_with_coordinates.csv",
    index=False
)
well_coords.to_csv(
    OUT / "02_alwar_well_coordinate_inventory.csv",
    index=False
)

summary = {
    "candidate_well_season_series": int(len(final)),
    "unique_candidate_wells": int(final["Well_key"].nunique()),
    "series_with_coordinates_present": int(
        final["latitude"].notna().sum() &
        final["longitude"].notna().sum()
    ),
    "wells_with_missing_coordinates": int(
        well_coords["coordinate_status"].eq("MISSING_COORDINATES").sum()
    ),
    "wells_with_inconsistent_coordinates": int(
        well_coords["coordinate_status"].eq("INCONSISTENT_COORDINATES").sum()
    ),
    "boundary_validation": (
        "NOT PERFORMED: an authoritative Alwar district boundary is required."
    )
}

with open(OUT / "03_step13_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 65)
print("STEP 13 COORDINATE AUDIT COMPLETE")
print("=" * 65)
print(json.dumps(summary, indent=2))
print("\nCoordinate status counts:")
display(well_coords["coordinate_status"].value_counts().rename_axis(
    "coordinate_status"
).reset_index(name="well_count"))
print("\nOutput folder:", OUT)


**[orig #116]** Step 13 — coordinate consistency per well (uses `alwar`)  
<sub>✅ ran OK</sub>

In [ ]:
# Check which columns are available
print("Available columns:")
print(alwar.columns.tolist())

# Build aggregation using only columns that exist
agg_spec = {}

column_mapping = {
    "District": "district",
    "Block": "block",
    "Village": "village",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Source_file": "source_file",
    "Source_sheet": "source_sheet",
    "Source_row_number": "source_row_number",
}

for source_col, output_col in column_mapping.items():
    if source_col in alwar.columns:
        if source_col in ["Latitude", "Longitude"]:
            agg_spec[output_col] = (source_col, "median")
        else:
            agg_spec[output_col] = (source_col, "first")

if "Well_key" not in alwar.columns:
    raise KeyError(
        "Well_key is missing. Check the source file and Well_key creation step."
    )

well_coords = (
    alwar.groupby("Well_key", dropna=False)
    .agg(**agg_spec)
    .reset_index()
)

# Check whether coordinates are consistent within each well
if {"Latitude", "Longitude"}.issubset(alwar.columns):
    coordinate_counts = (
        alwar.groupby("Well_key", dropna=False)
        .agg(
            latitude_unique=("Latitude", "nunique"),
            longitude_unique=("Longitude", "nunique"),
        )
        .reset_index()
    )

    well_coords = well_coords.merge(
        coordinate_counts,
        on="Well_key",
        how="left",
    )

    well_coords["coordinate_consistency"] = "CONSISTENT"

    well_coords.loc[
        (well_coords["latitude_unique"] > 1)
        | (well_coords["longitude_unique"] > 1),
        "coordinate_consistency",
    ] = "INCONSISTENT_COORDINATES"
else:
    well_coords["coordinate_consistency"] = "COORDINATE_COLUMNS_MISSING"

print("Unique wells in coordinate inventory:", len(well_coords))
print(well_coords.head())

**[orig #117]** Step 13 — save the coordinate inventory  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
import os

# Use the coordinate inventory created in the previous cell
coords = well_coords.copy()

# Check required fields
required = ["Well_key", "latitude", "longitude"]
missing = [c for c in required if c not in coords.columns]

if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Convert coordinates to numeric
coords["latitude"] = pd.to_numeric(
    coords["latitude"], errors="coerce"
)
coords["longitude"] = pd.to_numeric(
    coords["longitude"], errors="coerce"
)

# Coordinate availability
coords["coordinates_present"] = (
    coords["latitude"].notna()
    & coords["longitude"].notna()
)

# Basic geographic validity
coords["coordinates_in_valid_range"] = (
    coords["latitude"].between(-90, 90)
    & coords["longitude"].between(-180, 180)
)

# Flag missing, invalid, or inconsistent coordinates
coords["coordinate_qc_status"] = "PASS_BASIC_CHECKS"

coords.loc[
    ~coords["coordinates_present"],
    "coordinate_qc_status"
] = "MISSING_COORDINATES"

coords.loc[
    coords["coordinates_present"]
    & ~coords["coordinates_in_valid_range"],
    "coordinate_qc_status"
] = "INVALID_COORDINATE_RANGE"

if "coordinate_consistency" in coords.columns:
    coords.loc[
        coords["coordinate_consistency"]
        == "INCONSISTENT_COORDINATES",
        "coordinate_qc_status"
    ] = "INCONSISTENT_COORDINATES"

# Summary
summary = {
    "unique_wells": int(coords["Well_key"].nunique()),
    "wells_with_both_coordinates": int(
        coords["coordinates_present"].sum()
    ),
    "wells_missing_coordinates": int(
        (~coords["coordinates_present"]).sum()
    ),
    "wells_with_valid_global_coordinate_ranges": int(
        coords["coordinates_in_valid_range"].sum()
    ),
    "wells_flagged_for_coordinate_review": int(
        (coords["coordinate_qc_status"] != "PASS_BASIC_CHECKS").sum()
    ),
    "official_district_boundary_checked": False,
}

print("STEP 13 — COORDINATE QC SUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nQC status counts:")
print(coords["coordinate_qc_status"].value_counts(dropna=False))

# Save outputs
output_dir = (
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_13_Spatial_Validation"
)

os.makedirs(output_dir, exist_ok=True)

coords.to_csv(
    os.path.join(output_dir, "02_alwar_well_coordinate_inventory.csv"),
    index=False
)

pd.DataFrame([summary]).to_json(
    os.path.join(output_dir, "04_coordinate_qc_summary.json"),
    orient="records",
    indent=2
)

print("\nSaved coordinate QC outputs to:")
print(output_dir)

## 3. The 14-block study domain (Step 14)

Result: 473 of 502 wells fall inside the union of blocks LGD 574–587. **This domain is your model area.**

**[orig #118]** Locate the block boundary file  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import geopandas as gpd

boundary_path = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

print("File exists:", os.path.exists(boundary_path))

if os.path.exists(boundary_path):
    print("File size (GB):",
          round(os.path.getsize(boundary_path) / 1e9, 3))

**[orig #119]** Read block polygons; CRS and feature count  
<sub>✅ ran OK</sub>

In [ ]:
import geopandas as gpd

blocks = gpd.read_file(boundary_path)

print("CRS:", blocks.crs)
print("Number of features:", len(blocks))
print("Geometry types:")
print(blocks.geometry.geom_type.value_counts())
print("\nAvailable attribute columns:")
print(blocks.columns.tolist())

print("\nFirst five records:")
print(blocks.drop(columns="geometry").head())

**[orig #120]** Step 14 — wells vs Alwar block polygons (stops with KeyError *after* creating the variables the next cell uses)  
<sub>⚠ last run ended with KeyError</sub>

In [ ]:

import os
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import Point

# ---------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------
BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

BOUNDARY_PATH = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

STEP13_DIR = os.path.join(BASE, "Step_13_Spatial_Validation")
OUT_DIR = os.path.join(BASE, "Step_14_Spatial_Validation")

os.makedirs(OUT_DIR, exist_ok=True)

COORD_PATH = os.path.join(
    STEP13_DIR, "02_alwar_well_coordinate_inventory.csv"
)

if not os.path.exists(BOUNDARY_PATH):
    raise FileNotFoundError(f"Boundary file not found: {BOUNDARY_PATH}")

if not os.path.exists(COORD_PATH):
    raise FileNotFoundError(f"Coordinate inventory not found: {COORD_PATH}")

# ---------------------------------------------------------
# 2. READ AND CHECK WELL COORDINATES
# ---------------------------------------------------------
wells = pd.read_csv(COORD_PATH)

required_cols = ["Well_key", "latitude", "longitude"]
missing = [c for c in required_cols if c not in wells.columns]

if missing:
    raise KeyError(f"Missing well columns: {missing}")

wells["latitude"] = pd.to_numeric(
    wells["latitude"], errors="coerce"
)
wells["longitude"] = pd.to_numeric(
    wells["longitude"], errors="coerce"
)

wells["coordinate_available"] = (
    wells["latitude"].notna()
    & wells["longitude"].notna()
)

wells["coordinate_valid_range"] = (
    wells["latitude"].between(-90, 90)
    & wells["longitude"].between(-180, 180)
)

valid = wells.loc[
    wells["coordinate_available"]
    & wells["coordinate_valid_range"]
].copy()

# Coordinates are EPSG:4326, i.e. longitude, latitude.
well_points = gpd.GeoDataFrame(
    valid,
    geometry=gpd.points_from_xy(
        valid["longitude"], valid["latitude"]
    ),
    crs="EPSG:4326"
)

print("Total wells:", wells["Well_key"].nunique())
print("Wells with valid coordinate ranges:", len(well_points))
print(
    "Wells missing coordinates:",
    int((~wells["coordinate_available"]).sum())
)

# ---------------------------------------------------------
# 3. READ ADMINISTRATIVE BLOCK BOUNDARIES
# ---------------------------------------------------------
blocks = gpd.read_file(BOUNDARY_PATH)

required_boundary_cols = [
    "state", "district", "block_name", "block_lgd"
]
missing = [
    c for c in required_boundary_cols
    if c not in blocks.columns
]

if missing:
    raise KeyError(
        f"Boundary fields missing: {missing}"
    )

if blocks.crs is None:
    raise ValueError(
        "Boundary CRS is undefined. Do not assume EPSG:4326."
    )

print("\nBoundary CRS:", blocks.crs)
print("All boundary features:", len(blocks))

# Filter state and district names robustly.
blocks["state_clean"] = (
    blocks["state"].astype("string").str.strip().str.upper()
)
blocks["district_clean"] = (
    blocks["district"].astype("string").str.strip().str.upper()
)

alwar_blocks = blocks.loc[
    blocks["state_clean"].eq("RAJASTHAN")
    & blocks["district_clean"].str.contains(
        r"\bALWAR\b", na=False, regex=True
    )
].copy()

if alwar_blocks.empty:
    print("\nRajasthan district names containing 'ALWAR':")
    print(
        blocks.loc[
            blocks["state_clean"].eq("RAJASTHAN"),
            "district"
        ].dropna().drop_duplicates().sort_values().to_list()
    )
    raise ValueError(
        "No Alwar block features matched. Inspect the district names above."
    )

print("\nAlwar-labelled block features:", len(alwar_blocks))
print("District labels:", alwar_blocks["district"].unique())
print("Block names:", sorted(
    alwar_blocks["block_name"].dropna().astype(str).unique()
))

# Transform boundaries to the point CRS.
alwar_blocks = alwar_blocks.to_crs(well_points.crs)

# ---------------------------------------------------------
# 4. BASIC GEOMETRY QC
# ---------------------------------------------------------
alwar_blocks["geometry_valid_before_repair"] = (
    alwar_blocks.geometry.is_valid
)

invalid_count = int(
    (~alwar_blocks["geometry_valid_before_repair"]).sum()
)

if invalid_count:
    warnings.warn(
        f"{invalid_count} Alwar block geometries are invalid. "
        "Applying make_valid where available."
    )
    try:
        alwar_blocks["geometry"] = (
            alwar_blocks.geometry.make_valid()
        )
    except Exception:
        # Buffer(0) is a fallback, not a substitute for manual QC.
        alwar_blocks["geometry"] = (
            alwar_blocks.geometry.buffer(0)
        )

# Remove empty geometries from the spatial join only.
alwar_blocks = alwar_blocks.loc[
    alwar_blocks.geometry.notna()
    & ~alwar_blocks.geometry.is_empty
].copy()

# ---------------------------------------------------------
# 5. POINT-IN-POLYGON TEST
# ---------------------------------------------------------
# "within" excludes points exactly on polygon boundaries.
# Run "intersects" too, so boundary cases can be flagged.
within_join = gpd.sjoin(
    well_points,
    alwar_blocks[
        ["district", "block_name", "block_lgd", "geometry"]
    ],
    how="left",
    predicate="within"
)

intersects_join = gpd.sjoin(
    well_points,
    alwar_blocks[
        ["district", "block_name", "block_lgd", "geometry"]
    ],
    how="left",
    predicate="intersects"
)

# Summarize possible block matches per well.
within_summary = (
    within_join.groupby("Well_key", dropna=False)
    .agg(
        n_blocks_within=("index_right", "count"),
        matched_blocks=(
            "block_name",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
        matched_districts=(
            "district",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
    )
    .reset_index()
)

intersects_summary = (
    intersects_join.groupby("Well_key", dropna=False)
    .agg(
        n_blocks_intersected=("index_right", "count")
    )
    .reset_index()
)

audit = wells.merge(
    within_summary, on="Well_key", how="left"
).merge(
    intersects_summary, on="Well_key", how="left"
)

audit["n_blocks_within"] = (
    audit["n_blocks_within"].fillna(0).astype(int)
)
audit["n_blocks_intersected"] = (
    audit["n_blocks_intersected"].fillna(0).astype(int)
)

# ---------------------------------------------------------
# 6. CLASSIFY SPATIAL RESULTS
# ---------------------------------------------------------
audit["spatial_status"] = "OUTSIDE_ALWAR_BLOCK_UNION"

audit.loc[
    ~audit["coordinate_available"]
    | ~audit["coordinate_valid_range"],
    "spatial_status"
] = "MISSING_OR_INVALID_COORDINATES"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] == 1),
    "spatial_status"
] = "WITHIN_ONE_ALWAR_BLOCK"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] > 1),
    "spatial_status"
] = "MULTIPLE_BLOCK_MATCHES_REVIEW"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] == 0)
    & (audit["n_blocks_intersected"] > 0),
    "spatial_status"
] = "BOUNDARY_POINT_REVIEW"

# A point on a shared boundary may intersect multiple polygons.
# Do not assign it to one block automatically.
print("\nSpatial status counts:")
print(audit["spatial_status"].value_counts(dropna=False))

# ---------------------------------------------------------
# 7. SAVE AUDIT TABLES AND SUMMARY
# ---------------------------------------------------------
audit_path = os.path.join(
    OUT_DIR, "01_well_spatial_validation.csv"
)
blocks_path = os.path.join(
    OUT_DIR, "02_alwar_block_boundary_inventory.csv"
)
summary_path = os.path.join(
    OUT_DIR, "03_step14_summary.json"
)

audit.to_csv(audit_path, index=False)

alwar_blocks.drop(
    columns=["state_clean", "district_clean"],
    errors="ignore"
).to_file(
    blocks_path, driver="GeoJSON"
)

summary = {
    "total_unique_wells": int(wells["Well_key"].nunique()),
    "wells_with_valid_coordinate_ranges": int(len(well_points)),
    "wells_missing_coordinates": int(
        (~wells["coordinate_available"]).sum()
    ),
    "all_boundary_features": int(len(blocks)),
    "alwar_labelled_block_features": int(len(alwar_blocks)),
    "invalid_geometries_before_repair": invalid_count,
    "spatial_status_counts": {
        str(k): int(v)
        for k, v in audit["spatial_status"]
        .value_counts(dropna=False).items()
    },
    "boundary_source_file": os.path.basename(BOUNDARY_PATH),
    "boundary_crs": str(alwar_blocks.crs),
    "validation_method": "Point-in-polygon against Alwar-labelled block polygons",
    "official_district_boundary_independently_verified": False,
    "boundary_vintage_verified": False,
    "notes": (
        "Block-union results are a provisional spatial check. "
        "They do not independently establish the official district boundary. "
        "Review boundary points, multiple matches, outside points, and "
        "the well with missing coordinates."
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:")
print(audit_path)
print(blocks_path)
print(summary_path)

print("\nSTEP 14 SUMMARY")
print(json.dumps(summary, indent=2))


**[orig #121]** Step 14 — point-in-polygon test (column-collision fix)  
<sub>✅ ran OK</sub>

In [ ]:

# ---------------------------------------------------------
# STEP 5 — POINT-IN-POLYGON TEST (COLUMN COLLISION FIX)
# ---------------------------------------------------------

# Rename boundary attributes before joining so they cannot
# collide with columns already present in the well inventory.
boundary_for_join = alwar_blocks[
    ["block_name", "block_lgd", "geometry"]
].copy()

boundary_for_join = boundary_for_join.rename(
    columns={
        "block_name": "matched_block_name",
        "block_lgd": "matched_block_lgd",
    }
)

# Add a unique ID for counting matched polygons
boundary_for_join = boundary_for_join.reset_index(drop=True)
boundary_for_join["boundary_feature_id"] = boundary_for_join.index

# Strictly inside a polygon
within_join = gpd.sjoin(
    well_points,
    boundary_for_join,
    how="left",
    predicate="within"
)

# Includes points lying exactly on polygon boundaries
intersects_join = gpd.sjoin(
    well_points,
    boundary_for_join,
    how="left",
    predicate="intersects"
)

# Aggregate matches by well
within_summary = (
    within_join.groupby("Well_key", dropna=False)
    .agg(
        n_blocks_within=("boundary_feature_id", "count"),
        matched_blocks=(
            "matched_block_name",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
        matched_block_codes=(
            "matched_block_lgd",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
    )
    .reset_index()
)

intersects_summary = (
    intersects_join.groupby("Well_key", dropna=False)
    .agg(
        n_blocks_intersected=("boundary_feature_id", "count")
    )
    .reset_index()
)

# Ensure only one coordinate-inventory row per well
if wells["Well_key"].duplicated().any():
    raise ValueError("Coordinate inventory contains duplicate Well_key values.")

audit = wells.merge(
    within_summary, on="Well_key", how="left",
    validate="one_to_one"
).merge(
    intersects_summary, on="Well_key", how="left",
    validate="one_to_one"
)

audit["n_blocks_within"] = (
    audit["n_blocks_within"].fillna(0).astype(int)
)
audit["n_blocks_intersected"] = (
    audit["n_blocks_intersected"].fillna(0).astype(int)
)

# Classify every well, retaining missing-coordinate records
audit["spatial_status"] = "OUTSIDE_SELECTED_BLOCK_POLYGONS"

audit.loc[
    ~audit["coordinate_available"]
    | ~audit["coordinate_valid_range"],
    "spatial_status"
] = "MISSING_OR_INVALID_COORDINATES"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] == 1),
    "spatial_status"
] = "WITHIN_ONE_BLOCK"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] > 1),
    "spatial_status"
] = "MULTIPLE_BLOCK_MATCHES_REVIEW"

audit.loc[
    audit["coordinate_available"]
    & audit["coordinate_valid_range"]
    & (audit["n_blocks_within"] == 0)
    & (audit["n_blocks_intersected"] > 0),
    "spatial_status"
] = "BOUNDARY_POINT_REVIEW"

print("STEP 14 — SPATIAL STATUS COUNTS")
print(audit["spatial_status"].value_counts(dropna=False))

# Save audit table
audit_path = os.path.join(
    OUT_DIR, "01_well_spatial_validation.csv"
)
audit.to_csv(audit_path, index=False)

print("\nSaved:", audit_path)
print("\nFirst records:")
print(
    audit[
        ["Well_key", "latitude", "longitude",
         "matched_blocks", "spatial_status"]
    ].head(10).to_string(index=False)
)


**[orig #122]** Step 14 — wells outside the selected polygons  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import geopandas as gpd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)
BOUNDARY_PATH = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)
STEP14_DIR = os.path.join(BASE, "Step_14_Spatial_Validation")

audit_path = os.path.join(
    STEP14_DIR, "01_well_spatial_validation.csv"
)

audit = pd.read_csv(audit_path)
all_blocks = gpd.read_file(BOUNDARY_PATH)

# Ensure both datasets use the same coordinate reference system
if all_blocks.crs is None:
    raise ValueError("Boundary CRS is missing.")

# Select only wells outside the initially selected Alwar polygons
outside = audit.loc[
    audit["spatial_status"].eq("OUTSIDE_SELECTED_BLOCK_POLYGONS")
    & audit["latitude"].notna()
    & audit["longitude"].notna()
].copy()

outside_points = gpd.GeoDataFrame(
    outside,
    geometry=gpd.points_from_xy(
        outside["longitude"], outside["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(all_blocks.crs)

# Rename fields to avoid collisions with well attributes
all_blocks = all_blocks.rename(
    columns={
        "state": "matched_state",
        "district": "matched_district",
        "block_name": "matched_block",
        "block_lgd": "matched_block_lgd",
    }
).copy()

# Test all outside points against the full boundary dataset
matches = gpd.sjoin(
    outside_points,
    all_blocks[
        [
            "matched_state",
            "matched_district",
            "matched_block",
            "matched_block_lgd",
            "geometry",
        ]
    ],
    how="left",
    predicate="within"
)

# Summarize possible matches per well
diagnosis = (
    matches.groupby("Well_key", dropna=False)
    .agg(
        n_all_india_block_matches=("index_right", "count"),
        matched_states=(
            "matched_state",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
        matched_districts=(
            "matched_district",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
        matched_blocks_all_india=(
            "matched_block",
            lambda s: "; ".join(
                sorted(s.dropna().astype(str).unique())
            )
        ),
    )
    .reset_index()
)

diagnosis["all_india_boundary_diagnosis"] = "NO_POLYGON_MATCH"
diagnosis.loc[
    diagnosis["n_all_india_block_matches"] == 1,
    "all_india_boundary_diagnosis"
] = "MATCHED_ONE_BLOCK"

diagnosis.loc[
    diagnosis["n_all_india_block_matches"] > 1,
    "all_india_boundary_diagnosis"
] = "MULTIPLE_MATCHES_REVIEW"

out_path = os.path.join(
    STEP14_DIR, "04_outside_wells_all_india_boundary_diagnosis.csv"
)
diagnosis.to_csv(out_path, index=False)

print("Outside wells diagnosed:", diagnosis["Well_key"].nunique())
print("\nDiagnosis counts:")
print(
    diagnosis["all_india_boundary_diagnosis"]
    .value_counts(dropna=False)
)

print("\nMost frequent matched districts:")
print(
    diagnosis["matched_districts"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nSample:")
print(diagnosis.head(15).to_string(index=False))
print("\nSaved:", out_path)


**[orig #123]** Step 14 — inspect GEC workbook sheets for block names  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import json

EXCEL_PATH = (
    "/content/drive/MyDrive/gw_framework/data/"
    "Rajasthan_gw_data/Alwar.xlsx"
)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"Excel file not found: {EXCEL_PATH}")

# Inspect workbook sheets and columns
xls = pd.ExcelFile(EXCEL_PATH)

print("Workbook sheets:", xls.sheet_names)

all_sheets = []

for sheet in xls.sheet_names:
    df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)

    print(f"\nSheet: {sheet}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    # Identify likely block column
    possible_block_cols = [
        c for c in df.columns
        if str(c).strip().lower() in [
            "block", "block_name", "block name",
            "blockname", "development block"
        ]
    ]

    if possible_block_cols:
        block_col = possible_block_cols[0]

        temp = df.copy()
        temp["Source_sheet_checked"] = sheet
        temp["Block_name_from_excel"] = (
            temp[block_col].astype("string").str.strip()
        )
        all_sheets.append(temp)

if not all_sheets:
    raise ValueError(
        "Could not automatically identify a block column. "
        "Inspect the printed column names and set the correct column manually."
    )

combined = pd.concat(all_sheets, ignore_index=True)

block_inventory = (
    combined.loc[
        combined["Block_name_from_excel"].notna()
        & combined["Block_name_from_excel"].ne("")
    ]
    .groupby("Block_name_from_excel")
    .size()
    .reset_index(name="Excel_row_count")
    .sort_values("Block_name_from_excel")
)

print("\nUNIQUE BLOCK NAMES IN ALWAR.XLSX")
print(block_inventory.to_string(index=False))

print("\nNumber of unique block names:", len(block_inventory))

out_dir = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery/Step_14_Spatial_Validation"
)
os.makedirs(out_dir, exist_ok=True)

inventory_path = os.path.join(
    out_dir, "05_block_inventory_from_Alwar_excel.csv"
)
block_inventory.to_csv(inventory_path, index=False)

print("\nSaved:", inventory_path)


**[orig #124]** Step 14 — match block names between GEC and polygons  
<sub>✅ ran OK</sub>

In [ ]:

import os
import re
import difflib
import pandas as pd
import geopandas as gpd

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery/Step_14_Spatial_Validation"
)

EXCEL_INVENTORY = os.path.join(
    OUT_DIR, "05_block_inventory_from_Alwar_excel.csv"
)

GEOJSON_PATH = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
excel = pd.read_csv(EXCEL_INVENTORY)
geo = gpd.read_file(GEOJSON_PATH)

# ---------------------------------------------------------
# 2. NORMALIZE SPELLING
# ---------------------------------------------------------
def normalize_name(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    value = re.sub(r"[^A-Z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

excel["block_key"] = (
    excel["Block_name_from_excel"].map(normalize_name)
)

geo["block_key"] = geo["block_name"].map(normalize_name)
geo["state_key"] = geo["state"].map(normalize_name)
geo["district_key"] = geo["district"].map(normalize_name)

# Compare against all Rajasthan blocks, not just Alwar-labelled blocks
rajasthan = geo.loc[
    geo["state_key"].eq("RAJASTHAN")
].copy()

print("Excel unique block names:", excel["block_key"].nunique())
print("Rajasthan boundary features:", len(rajasthan))
print(
    "Unique Rajasthan block names:",
    rajasthan["block_key"].nunique()
)

# ---------------------------------------------------------
# 3. EXACT AND FUZZY MATCH CANDIDATES
# ---------------------------------------------------------
results = []

for _, erow in excel.iterrows():
    excel_name = erow["Block_name_from_excel"]
    key = erow["block_key"]

    # Exact normalized-name matches
    exact = rajasthan.loc[
        rajasthan["block_key"].eq(key)
    ]

    if not exact.empty:
        for _, grow in exact.iterrows():
            results.append({
                "excel_block": excel_name,
                "excel_row_count": erow["Excel_row_count"],
                "geojson_block": grow["block_name"],
                "geojson_district": grow["district"],
                "geojson_state": grow["state"],
                "block_lgd": grow["block_lgd"],
                "dist_lgd": grow["dist_lgd"],
                "similarity_percent": 100.0,
                "match_type": "EXACT_NORMALIZED",
                "review_status": "CHECK_DISTRICT_AND_CODES",
            })
        continue

    # Fuzzy suggestions: compare distinct names, then retain all
    # geographic records corresponding to the best candidate names.
    distinct_names = (
        rajasthan[["block_key", "block_name"]]
        .drop_duplicates()
    )

    scored = []
    for _, candidate in distinct_names.iterrows():
        score = difflib.SequenceMatcher(
            None, key, candidate["block_key"]
        ).ratio() * 100

        scored.append((
            score,
            candidate["block_key"],
            candidate["block_name"]
        ))

    scored.sort(reverse=True)

    # Keep the top 5 suggestions with similarity >= 65%.
    # Lower scores are weak suggestions and need manual inspection.
    suggestions = [x for x in scored if x[0] >= 65][:5]

    if not suggestions:
        results.append({
            "excel_block": excel_name,
            "excel_row_count": erow["Excel_row_count"],
            "geojson_block": None,
            "geojson_district": None,
            "geojson_state": None,
            "block_lgd": None,
            "dist_lgd": None,
            "similarity_percent": None,
            "match_type": "NO_CANDIDATE_AT_65_PERCENT",
            "review_status": "MANUAL_REVIEW_REQUIRED",
        })
        continue

    for score, candidate_key, candidate_name in suggestions:
        matched_features = rajasthan.loc[
            rajasthan["block_key"].eq(candidate_key)
        ]

        for _, grow in matched_features.iterrows():
            results.append({
                "excel_block": excel_name,
                "excel_row_count": erow["Excel_row_count"],
                "geojson_block": grow["block_name"],
                "geojson_district": grow["district"],
                "geojson_state": grow["state"],
                "block_lgd": grow["block_lgd"],
                "dist_lgd": grow["dist_lgd"],
                "similarity_percent": round(score, 1),
                "match_type": "FUZZY_CANDIDATE",
                "review_status": "MANUAL_REVIEW_REQUIRED",
            })

matches = pd.DataFrame(results)

# ---------------------------------------------------------
# 4. SAVE ALL MATCH CANDIDATES
# ---------------------------------------------------------
match_path = os.path.join(
    OUT_DIR, "06_excel_to_rajasthan_fuzzy_block_matches.csv"
)
matches.to_csv(match_path, index=False)

# Show the best candidate for each Excel block
best = (
    matches.sort_values(
        ["excel_block", "similarity_percent"],
        ascending=[True, False],
        na_position="last"
    )
    .groupby("excel_block", as_index=False)
    .first()
)

print("\nBEST CANDIDATE FOR EACH EXCEL BLOCK")
print(
    best[
        [
            "excel_block",
            "geojson_block",
            "geojson_district",
            "block_lgd",
            "dist_lgd",
            "similarity_percent",
            "match_type",
            "review_status",
        ]
    ].to_string(index=False)
)

print("\nAll candidate matches saved to:")
print(match_path)


**[orig #125]** Step 14 — accepted historical study-area block names  
<sub>✅ ran OK</sub>

In [ ]:

import os
import re
import json
import pandas as pd
import geopandas as gpd

BOUNDARY_PATH = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery/Step_14_Spatial_Validation"
)
os.makedirs(OUT_DIR, exist_ok=True)

# Accepted historical study-area block names
aliases = {
    "BANSUR": ["BANSUR"],
    "BEHROR": ["BEHROR"],
    "KATHUMAR": ["KATHUMAR"],
    "KISHANGARH": ["KISHANGARH", "KISHANGARH BAS"],
    "KOTKASIM": ["KOTKASIM"],
    "LAXMANGARH": ["LAXMANGARH"],
    "MANDAWAR": ["MANDAWAR", "MUNDAWAR"],
    "NEEMRANA": ["NEEMRANA"],
    "RAJGARH": ["RAJGARH"],
    "RAMGARH": ["RAMGARH"],
    "RENI": ["RENI"],
    "THANAGAZI": ["THANAGAZI"],
    "TIJARA": ["TIJARA"],
    "UMRAIN": ["UMRAIN", "UMREN"],
}

def norm(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", re.sub(
        r"[^A-Z0-9]+", " ", str(x).strip().upper()
    )).strip()

geo = gpd.read_file(BOUNDARY_PATH)

for col in ["state", "district", "block_name", "dist_lgd", "block_lgd"]:
    if col not in geo.columns:
        raise KeyError(f"Missing boundary field: {col}")

geo["state_key"] = geo["state"].map(norm)
geo["district_key"] = geo["district"].map(norm)
geo["block_key"] = geo["block_name"].map(norm)

# Candidate filter: based on the district-code pattern observed
# in the user's match output. This must be confirmed by review.
geo["dist_lgd_num"] = pd.to_numeric(geo["dist_lgd"], errors="coerce")

rajasthan = geo[geo["state_key"] == "RAJASTHAN"].copy()
candidates = rajasthan[rajasthan["dist_lgd_num"] == 87].copy()

selected_parts = []
review_rows = []

for target, possible_names in aliases.items():
    alias_keys = [norm(n) for n in possible_names]

    found = candidates[candidates["block_key"].isin(alias_keys)].copy()

    # Explicitly record candidates from all Rajasthan districts
    all_name_matches = rajasthan[
        rajasthan["block_key"].isin(alias_keys)
    ].copy()

    if found.empty:
        review_rows.append({
            "target_block": target,
            "status": "NO_MATCH_IN_DIST_LGD_87",
            "candidate_names": "; ".join(
                all_name_matches["block_name"].dropna().astype(str).unique()
            ),
            "candidate_districts": "; ".join(
                all_name_matches["district"].dropna().astype(str).unique()
            ),
            "candidate_dist_lgd": "; ".join(
                all_name_matches["dist_lgd"].dropna().astype(str).unique()
            ),
        })
        continue

    # Do not automatically accept ambiguous matches
    if target == "RAJGARH" and (
        found["block_key"].nunique() != 1
        or not (found["dist_lgd_num"] == 87).all()
    ):
        review_rows.append({
            "target_block": target,
            "status": "RAJGARH_REQUIRES_MANUAL_REVIEW",
            "candidate_names": "; ".join(
                all_name_matches["block_name"].dropna().astype(str).unique()
            ),
            "candidate_districts": "; ".join(
                all_name_matches["district"].dropna().astype(str).unique()
            ),
            "candidate_dist_lgd": "; ".join(
                all_name_matches["dist_lgd"].dropna().astype(str).unique()
            ),
        })
        continue

    for _, row in found.iterrows():
        selected_parts.append(row.to_dict())

    review_rows.append({
        "target_block": target,
        "status": "CANDIDATE_SELECTED_FOR_REVIEW",
        "candidate_names": "; ".join(
            found["block_name"].dropna().astype(str).unique()
        ),
        "candidate_districts": "; ".join(
            found["district"].dropna().astype(str).unique()
        ),
        "candidate_dist_lgd": "; ".join(
            found["dist_lgd"].dropna().astype(str).unique()
        ),
    })

review = pd.DataFrame(review_rows)
review_path = os.path.join(
    OUT_DIR, "07_historical_alwar_block_selection_review.csv"
)
review.to_csv(review_path, index=False)

print("BLOCK SELECTION REVIEW")
print(review.to_string(index=False))

if selected_parts:
    selected = gpd.GeoDataFrame(
        selected_parts, geometry="geometry", crs=geo.crs
    )
    selected = selected.drop_duplicates(
        subset=["block_lgd", "block_key"]
    )

    selected_path = os.path.join(
        OUT_DIR, "08_historical_alwar_block_candidates.geojson"
    )
    selected.to_file(selected_path, driver="GeoJSON")

    print("\nCandidate block features selected:", len(selected))
    print("Unique candidate block names:", selected["block_key"].nunique())
    print("Saved candidate polygons:", selected_path)

print("\nReview table saved:", review_path)
print(
    "\nDo not use these polygons for final well classification until "
    "all 14 target blocks are accounted for and ambiguous names are resolved."
)


**[orig #126]** Step 14D — rebuild the domain from accepted blocks  
<sub>✅ ran OK</sub>

In [ ]:

import os
import json
import pandas as pd
import geopandas as gpd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)
STEP13_DIR = os.path.join(BASE, "Step_13_Spatial_Validation")
STEP14_DIR = os.path.join(BASE, "Step_14_Spatial_Validation")

CANDIDATE_PATH = os.path.join(
    STEP14_DIR, "08_historical_alwar_block_candidates.geojson"
)
COORD_PATH = os.path.join(
    STEP13_DIR, "02_alwar_well_coordinate_inventory.csv"
)

OUT_DIR = os.path.join(BASE, "Step_14D_Historical_Boundary_QC")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. LOAD INPUTS
# ---------------------------------------------------------
blocks = gpd.read_file(CANDIDATE_PATH)
wells = pd.read_csv(COORD_PATH)

if blocks.crs is None:
    raise ValueError("Block candidate CRS is missing.")

if wells["Well_key"].duplicated().any():
    raise ValueError("Duplicate Well_key values in coordinate inventory.")

for col in ["latitude", "longitude"]:
    wells[col] = pd.to_numeric(wells[col], errors="coerce")

valid_mask = (
    wells["latitude"].between(-90, 90)
    & wells["longitude"].between(-180, 180)
)

valid_wells = wells.loc[valid_mask].copy()

well_points = gpd.GeoDataFrame(
    valid_wells,
    geometry=gpd.points_from_xy(
        valid_wells["longitude"], valid_wells["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(blocks.crs)

# ---------------------------------------------------------
# 2. CHECK SELECTED BLOCK INVENTORY
# ---------------------------------------------------------
print("Candidate polygon features:", len(blocks))
print("Unique block names:", blocks["block_name"].nunique())
print("\nSelected blocks:")
print(
    blocks[["block_name", "district", "block_lgd", "dist_lgd"]]
    .drop_duplicates()
    .sort_values("block_name")
    .to_string(index=False)
)

if blocks["block_name"].nunique() != 14:
    raise ValueError(
        "Expected 14 unique selected blocks. Review selection before proceeding."
    )

# ---------------------------------------------------------
# 3. GEOMETRY QC
# ---------------------------------------------------------
blocks["valid_before_repair"] = blocks.geometry.is_valid
invalid_before = int((~blocks["valid_before_repair"]).sum())

if invalid_before:
    try:
        blocks["geometry"] = blocks.geometry.make_valid()
    except Exception:
        blocks["geometry"] = blocks.geometry.buffer(0)

blocks = blocks.loc[
    blocks.geometry.notna() & ~blocks.geometry.is_empty
].copy()

if blocks.empty:
    raise ValueError("No usable block geometries remain.")

# Dissolve into one multipart study-area geometry.
# This does not fill gaps between separate polygons.
study_area = gpd.GeoDataFrame(
    {"study_area_name": ["Historical_Alwar_Block_Union"]},
    geometry=[blocks.geometry.union_all()],
    crs=blocks.crs
)

study_area["geometry_valid"] = study_area.geometry.is_valid
study_area["geometry_area_crs_units"] = study_area.geometry.area

if not bool(study_area["geometry_valid"].iloc[0]):
    raise ValueError("Dissolved study-area geometry is invalid.")

# Save projected area in square kilometres using an appropriate
# metric CRS for Rajasthan, rather than using degrees squared.
metric_crs = "EPSG:32643"
study_metric = study_area.to_crs(metric_crs)
area_km2 = float(study_metric.geometry.area.iloc[0] / 1_000_000)

# ---------------------------------------------------------
# 4. TEST WELLS AGAINST THE DISSOLVED UNION
# ---------------------------------------------------------
union_geom = study_area.geometry.iloc[0]

well_points["inside_union"] = well_points.geometry.within(union_geom)
well_points["intersects_union"] = well_points.geometry.intersects(union_geom)

point_results = well_points[
    ["Well_key", "inside_union", "intersects_union"]
].copy()

audit = wells.merge(
    point_results, on="Well_key", how="left", validate="one_to_one"
)

audit["spatial_status"] = "OUTSIDE_RECONSTRUCTED_BLOCK_UNION"

audit.loc[
    ~valid_mask.values,
    "spatial_status"
] = "MISSING_OR_INVALID_COORDINATES"

audit.loc[
    audit["inside_union"].eq(True),
    "spatial_status"
] = "INSIDE_RECONSTRUCTED_BLOCK_UNION"

audit.loc[
    audit["inside_union"].eq(False)
    & audit["intersects_union"].eq(True),
    "spatial_status"
] = "BOUNDARY_POINT_REVIEW"

# ---------------------------------------------------------
# 5. COMPARE WITH PREVIOUS STEP 14
# ---------------------------------------------------------
old_path = os.path.join(
    STEP14_DIR, "01_well_spatial_validation.csv"
)

if os.path.exists(old_path):
    old = pd.read_csv(old_path)[["Well_key", "spatial_status"]]
    old = old.rename(columns={"spatial_status": "previous_step14_status"})
    audit = audit.merge(
        old, on="Well_key", how="left", validate="one_to_one"
    )

# ---------------------------------------------------------
# 6. SAVE OUTPUTS
# ---------------------------------------------------------
boundary_path = os.path.join(
    OUT_DIR, "01_reconstructed_historical_alwar_block_union.geojson"
)
audit_path = os.path.join(
    OUT_DIR, "02_well_spatial_validation.csv"
)
summary_path = os.path.join(
    OUT_DIR, "03_step14d_summary.json"
)

study_area.to_file(boundary_path, driver="GeoJSON")
audit.to_csv(audit_path, index=False)

summary = {
    "wells_in_inventory": int(wells["Well_key"].nunique()),
    "wells_with_valid_coordinate_ranges": int(valid_mask.sum()),
    "wells_with_missing_or_invalid_coordinates": int((~valid_mask).sum()),
    "selected_block_features_before_repair": int(len(blocks)),
    "unique_selected_block_names": int(blocks["block_name"].nunique()),
    "invalid_block_geometries_before_repair": invalid_before,
    "dissolved_geometry_valid": bool(study_area["geometry_valid"].iloc[0]),
    "dissolved_area_km2_utm43n": area_km2,
    "spatial_status_counts": {
        str(k): int(v)
        for k, v in audit["spatial_status"].value_counts(dropna=False).items()
    },
    "boundary_interpretation": (
        "Union of 14 selected block polygons; historical administrative "
        "boundary status requires independent verification."
    ),
    "historical_boundary_independently_verified": False
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSTEP 14D SUMMARY")
print(json.dumps(summary, indent=2))
print("\nSaved outputs:")
print(boundary_path)
print(audit_path)
print(summary_path)


**[orig #127]** Step 14D/E — the 28 wells outside the reconstructed union  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import geopandas as gpd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)
STEP14D = os.path.join(BASE, "Step_14D_Historical_Boundary_QC")
OUT_DIR = os.path.join(BASE, "Step_14E_Outside_Well_Review")
os.makedirs(OUT_DIR, exist_ok=True)

BOUNDARY_PATH = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)
AUDIT_PATH = os.path.join(
    STEP14D, "02_well_spatial_validation.csv"
)

audit = pd.read_csv(AUDIT_PATH)
geo = gpd.read_file(BOUNDARY_PATH)

# Identify the 28 wells outside the reconstructed union
outside = audit.loc[
    audit["spatial_status"].eq("OUTSIDE_RECONSTRUCTED_BLOCK_UNION")
].copy()

outside["latitude"] = pd.to_numeric(
    outside["latitude"], errors="coerce"
)
outside["longitude"] = pd.to_numeric(
    outside["longitude"], errors="coerce"
)

outside = outside.loc[
    outside["latitude"].between(-90, 90)
    & outside["longitude"].between(-180, 180)
].copy()

points = gpd.GeoDataFrame(
    outside,
    geometry=gpd.points_from_xy(
        outside["longitude"], outside["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(geo.crs)

# Avoid attribute-name collisions
geo = geo.rename(columns={
    "state": "candidate_state",
    "district": "candidate_district",
    "block_name": "candidate_block",
    "block_lgd": "candidate_block_lgd",
})

matches = gpd.sjoin(
    points,
    geo[[
        "candidate_state", "candidate_district",
        "candidate_block", "candidate_block_lgd",
        "geometry"
    ]],
    how="left",
    predicate="within"
)

diagnosis = (
    matches.groupby("Well_key", dropna=False)
    .agg(
        n_candidate_polygons=("index_right", "count"),
        candidate_states=(
            "candidate_state",
            lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))
        ),
        candidate_districts=(
            "candidate_district",
            lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))
        ),
        candidate_blocks=(
            "candidate_block",
            lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))
        ),
        candidate_block_codes=(
            "candidate_block_lgd",
            lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))
        ),
    )
    .reset_index()
)

diagnosis["diagnosis"] = "NO_MATCH_IN_BOUNDARY_DATASET"
diagnosis.loc[
    diagnosis["n_candidate_polygons"] == 1,
    "diagnosis"
] = "MATCHED_ONE_POLYGON"
diagnosis.loc[
    diagnosis["n_candidate_polygons"] > 1,
    "diagnosis"
] = "MULTIPLE_POLYGONS_REVIEW"

output_path = os.path.join(
    OUT_DIR, "01_outside_well_boundary_diagnosis.csv"
)
diagnosis.to_csv(output_path, index=False)

print("Outside wells reviewed:", diagnosis["Well_key"].nunique())
print("\nDiagnosis counts:")
print(diagnosis["diagnosis"].value_counts(dropna=False))

print("\nCandidate district counts:")
print(
    diagnosis["candidate_districts"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nDetailed results:")
print(diagnosis.to_string(index=False))
print("\nSaved:", output_path)


**[orig #128]** **MAIN** Step 14F — final 14-block domain: 473 wells in, 29 out → `Step_14F_Spatial_Analysis_Restriction/`  
<sub>✅ ran OK</sub>

In [ ]:

import os
import json
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP14D_DIR = os.path.join(
    BASE, "Step_14D_Historical_Boundary_QC"
)
STEP14E_DIR = os.path.join(
    BASE, "Step_14E_Outside_Well_Review"
)
OUT_DIR = os.path.join(
    BASE, "Step_14F_Spatial_Analysis_Restriction"
)
os.makedirs(OUT_DIR, exist_ok=True)

AUDIT_PATH = os.path.join(
    STEP14D_DIR, "02_well_spatial_validation.csv"
)
OUTSIDE_PATH = os.path.join(
    STEP14E_DIR, "01_outside_well_boundary_diagnosis.csv"
)

audit = pd.read_csv(AUDIT_PATH)
outside_review = pd.read_csv(OUTSIDE_PATH)

# ---------------------------------------------------------
# 1. Confirm unique well identifiers
# ---------------------------------------------------------
if audit["Well_key"].duplicated().any():
    raise ValueError("Duplicate Well_key in Step 14D audit.")

if outside_review["Well_key"].duplicated().any():
    raise ValueError("Duplicate Well_key in Step 14E outside-well review.")

# ---------------------------------------------------------
# 2. Define the requested analysis domain
# ---------------------------------------------------------
excluded_keys = set(outside_review["Well_key"].dropna().astype(str))

audit["Well_key"] = audit["Well_key"].astype(str)

audit["analysis_domain_status"] = "INCLUDE_IN_14_BLOCK_DOMAIN"

audit.loc[
    audit["Well_key"].isin(excluded_keys),
    "analysis_domain_status"
] = "EXCLUDE_OUTSIDE_14_BLOCK_DOMAIN"

# The well with missing coordinates is not spatially verified.
missing_coord = (
    audit["latitude"].isna() | audit["longitude"].isna()
)

audit.loc[
    missing_coord,
    "analysis_domain_status"
] = "EXCLUDE_PENDING_COORDINATE_VALIDATION"

# ---------------------------------------------------------
# 3. Split without deleting the original inventory
# ---------------------------------------------------------
included = audit.loc[
    audit["analysis_domain_status"].eq(
        "INCLUDE_IN_14_BLOCK_DOMAIN"
    )
].copy()

excluded = audit.loc[
    ~audit["analysis_domain_status"].eq(
        "INCLUDE_IN_14_BLOCK_DOMAIN"
    )
].copy()

# ---------------------------------------------------------
# 4. Save datasets and exclusion log
# ---------------------------------------------------------
included_path = os.path.join(
    OUT_DIR, "01_wells_in_14_block_analysis_domain.csv"
)
excluded_path = os.path.join(
    OUT_DIR, "02_wells_excluded_from_14_block_domain.csv"
)
full_audit_path = os.path.join(
    OUT_DIR, "03_all_wells_with_domain_decision.csv"
)
summary_path = os.path.join(
    OUT_DIR, "04_step14f_summary.json"
)

included.to_csv(included_path, index=False)
excluded.to_csv(excluded_path, index=False)
audit.to_csv(full_audit_path, index=False)

summary = {
    "original_wells": int(audit["Well_key"].nunique()),
    "included_in_14_block_domain": int(included["Well_key"].nunique()),
    "excluded_outside_14_block_domain": int(
        audit["Well_key"].isin(excluded_keys).sum()
    ),
    "excluded_pending_coordinate_validation": int(
        (audit["analysis_domain_status"]
         == "EXCLUDE_PENDING_COORDINATE_VALIDATION").sum()
    ),
    "total_excluded": int(excluded["Well_key"].nunique()),
    "analysis_domain": "Union of 14 selected block polygons",
    "exclusion_reason": (
        "Outside selected 14-block analysis domain; "
        "not interpreted as invalid well data."
    ),
    "original_source_data_modified": False
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("STEP 14F — ANALYSIS DOMAIN SUMMARY")
print(json.dumps(summary, indent=2))

print("\nSaved files:")
print(included_path)
print(excluded_path)
print(full_audit_path)
print(summary_path)


## 4. Trends inside the domain (Step 15)

Result: 90 wells × 2 seasons = 180 series; 129 testable (≥ 8 consecutive years). 70 are significant under both MK and TFPW, and every one of them shows DTW deepening (pre-monsoon 39, post-monsoon 31).

**[orig #132]** Step 15A — load files and standardise column names  
<sub>✅ ran OK</sub>

In [ ]:

# STEP 15A — LOAD FILES AND STANDARDIZE COLUMN NAMES

import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

INCLUSION_FILE = (
    BASE / "Step_14F_Spatial_Analysis_Restriction"
    / "01_wells_in_14_block_analysis_domain.csv"
)

STEP12_FILE = (
    BASE / "Step_12_Trend_Direction_QC"
    / "01_candidate_direction_audit.csv"
)

OBSERVATION_FILE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = BASE / "Step_15_Spatial_Trend_Restriction"
OUT.mkdir(parents=True, exist_ok=True)

# Load the original files
included = pd.read_csv(INCLUSION_FILE, low_memory=False)
step12 = pd.read_csv(STEP12_FILE, low_memory=False)
observations = pd.read_csv(OBSERVATION_FILE, low_memory=False)

# CRITICAL FIX: standardize the season field BEFORE validation
if "Season" not in step12.columns:
    if "Season_clean" in step12.columns:
        step12 = step12.rename(
            columns={"Season_clean": "Season"}
        )
    else:
        raise ValueError(
            "Neither Season nor Season_clean exists in Step 12."
        )

# Validate required columns
def require_columns(df, columns, label):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"{label}: missing columns {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

require_columns(
    included, ["Well_key"], "Step 14F inclusion file"
)
require_columns(
    step12, ["Well_key", "Season"], "Step 12 trend audit"
)
require_columns(
    observations, ["Well_key"], "Observation dataset"
)

print("SUCCESS: all input schemas validated.")
print("Included spatial wells:", included["Well_key"].nunique())
print("Step 12 trend series:", len(step12))
print("Seasons:", step12["Season"].dropna().unique().tolist())
print("Observation rows:", len(observations))


**[orig #133]** Step 15A — spatial restriction audit  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 15A — SPATIAL RESTRICTION AUDIT
# Continuation after successful input validation
# ================================================================

import numpy as np
import pandas as pd
import json
from datetime import datetime

# Output directory should already exist from the previous cell.
OUT.mkdir(parents=True, exist_ok=True)

# 1. Normalize keys in memory only
for df in [included, step12, observations]:
    df["Well_key"] = df["Well_key"].astype("string").str.strip()

step12["Season"] = step12["Season"].astype("string").str.strip()

# 2. Validate unique inclusion list and trend-series keys
if included["Well_key"].isna().any():
    raise ValueError("Step 14F inclusion list has missing Well_key values.")

if included["Well_key"].duplicated().any():
    raise ValueError("Step 14F inclusion list contains duplicate Well_key values.")

if step12[["Well_key", "Season"]].isna().any().any():
    raise ValueError("Step 12 has missing Well_key or Season values.")

if step12.duplicated(["Well_key", "Season"]).any():
    raise ValueError("Step 12 has duplicate Well_key + Season series.")

allowed_keys = set(included["Well_key"].dropna())

# 3. Spatially restrict Step 12 trend audit
step12["spatial_domain_status"] = np.where(
    step12["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

restricted_trends = step12[
    step12["Well_key"].isin(allowed_keys)
].copy()

excluded_trends = step12[
    ~step12["Well_key"].isin(allowed_keys)
].copy()

# 4. Spatially restrict groundwater observations
observations["spatial_domain_status"] = np.where(
    observations["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

restricted_observations = observations[
    observations["Well_key"].isin(allowed_keys)
].copy()

excluded_observations = observations[
    ~observations["Well_key"].isin(allowed_keys)
].copy()

# 5. Candidate-well retention audit
candidate_wells = set(step12["Well_key"].dropna())
retained_wells = candidate_wells.intersection(allowed_keys)
excluded_wells = candidate_wells.difference(allowed_keys)

well_audit = pd.DataFrame({
    "Well_key": sorted(candidate_wells)
})

well_audit["spatial_domain_status"] = np.where(
    well_audit["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

original_series_counts = (
    step12.groupby("Well_key")
    .size()
    .rename("original_well_season_series")
)

retained_series_counts = (
    restricted_trends.groupby("Well_key")
    .size()
    .rename("retained_well_season_series")
)

well_audit = (
    well_audit
    .merge(original_series_counts, on="Well_key", how="left")
    .merge(retained_series_counts, on="Well_key", how="left")
)

well_audit["retained_well_season_series"] = (
    well_audit["retained_well_season_series"].fillna(0).astype(int)
)

# 6. Summarize retained trend classifications
classification_summary = (
    restricted_trends
    .groupby(
        ["Season", "direction_qc_classification"],
        dropna=False
    )
    .size()
    .reset_index(name="series_count")
)

# 7. Save results
restricted_trends.to_csv(
    OUT / "01_spatially_restricted_candidate_trends.csv",
    index=False
)

restricted_observations.to_csv(
    OUT / "02_spatially_restricted_groundwater_observations.csv",
    index=False
)

well_audit.to_csv(
    OUT / "03_candidate_well_retention_audit.csv",
    index=False
)

excluded_trends.to_csv(
    OUT / "04_candidate_series_outside_domain.csv",
    index=False
)

excluded_observations.to_csv(
    OUT / "05_observations_outside_domain.csv",
    index=False
)

classification_summary.to_csv(
    OUT / "06_trend_classification_by_season.csv",
    index=False
)

summary = {
    "step": "15A_Spatial_Restriction_Audit",
    "analysis_domain": "Union of selected 14 block polygons",
    "spatially_included_inventory_wells": int(len(allowed_keys)),
    "original_candidate_wells": int(len(candidate_wells)),
    "retained_candidate_wells": int(len(retained_wells)),
    "excluded_candidate_wells": int(len(excluded_wells)),
    "original_well_season_series": int(len(step12)),
    "retained_well_season_series": int(len(restricted_trends)),
    "excluded_well_season_series": int(len(excluded_trends)),
    "original_observation_rows": int(len(observations)),
    "retained_observation_rows": int(len(restricted_observations)),
    "excluded_observation_rows": int(len(excluded_observations)),
    "original_source_files_modified": False,
    "trend_statistics_recalculated": False,
    "interpretation": (
        "Existing trend results were filtered spatially. "
        "Sen slopes and significance values were not recalculated."
    ),
    "created_at": datetime.now().isoformat(timespec="seconds")
}

with open(OUT / "07_step15a_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

classification_summary.to_csv(
    OUT / "08_step15a_classification_summary.csv",
    index=False
)

# 8. Print report
print("=" * 60)
print("STEP 15A — SPATIALLY RESTRICTED TREND AUDIT")
print("=" * 60)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nTrend classification by season:")
print(classification_summary.to_string(index=False))

print("\nOutputs saved to:")
print(OUT)


**[orig #134]** Step 15B part 1 — observation schema and numeric-record diagnosis  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 15B — PART 1
# Observation schema validation and numeric-record diagnosis
# Step 15A outputs remain untouched.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

STEP15A = BASE / "Step_15_Spatial_Trend_Restriction"
OUT = BASE / "Step_15B_Trend_Recalculation"
OUT.mkdir(parents=True, exist_ok=True)

OBS_FILE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

TREND_FILE = (
    STEP15A / "01_spatially_restricted_candidate_trends.csv"
)

# ---------- 1. Validate inputs ----------

for path in [OBS_FILE, TREND_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

obs = pd.read_csv(OBS_FILE, low_memory=False)
trends = pd.read_csv(TREND_FILE, low_memory=False)

for label, df in [("Observations", obs), ("Step 15A trends", trends)]:
    if "Well_key" not in df.columns:
        raise ValueError(
            f"{label}: Well_key not found. Columns: {df.columns.tolist()}"
        )

obs["Well_key"] = obs["Well_key"].astype("string").str.strip()
trends["Well_key"] = trends["Well_key"].astype("string").str.strip()

candidate_keys = set(trends["Well_key"].dropna().unique())

# ---------- 2. Filter observations to retained candidates ----------

candidate_obs = obs[obs["Well_key"].isin(candidate_keys)].copy()

print("=" * 65)
print("STEP 15B — OBSERVATION DATA DIAGNOSTICS")
print("=" * 65)
print("Step 15A candidate wells:", len(candidate_keys))
print("Original observation rows:", len(obs))
print("Rows belonging to retained candidate wells:", len(candidate_obs))
print("Unique candidate wells with observations:",
      candidate_obs["Well_key"].nunique())

# ---------- 3. Inspect all column names and data types ----------

schema = pd.DataFrame({
    "column": candidate_obs.columns,
    "dtype": [str(candidate_obs[c].dtype) for c in candidate_obs.columns],
    "non_null_count": [int(candidate_obs[c].notna().sum())
                       for c in candidate_obs.columns],
    "unique_non_null_values": [
        int(candidate_obs[c].nunique(dropna=True))
        for c in candidate_obs.columns
    ]
})

schema.to_csv(OUT / "01_observation_schema.csv", index=False)

print("\nCOLUMN SCHEMA")
print(schema.to_string(index=False))

# ---------- 4. Display sample records ----------

print("\nSAMPLE OBSERVATION RECORDS")
print(candidate_obs.head(8).to_string(index=False))

# ---------- 5. Identify likely date, value and status fields ----------
# These are suggestions only. No column is automatically assumed valid.

terms = {
    "date_candidates": [
        c for c in candidate_obs.columns
        if any(x in c.lower() for x in
               ["date", "year", "month", "season", "time"])
    ],
    "water_level_candidates": [
        c for c in candidate_obs.columns
        if any(x in c.lower() for x in
               ["dtw", "depth", "water_level", "waterlevel",
                "head", "wl", "groundwater", "measurement"])
    ],
    "status_candidates": [
        c for c in candidate_obs.columns
        if any(x in c.lower() for x in
               ["status", "remark", "reading", "quality", "flag"])
    ]
}

print("\nPOSSIBLE DATE/TIME FIELDS:")
print(terms["date_candidates"])

print("\nPOSSIBLE WATER-LEVEL FIELDS:")
print(terms["water_level_candidates"])

print("\nPOSSIBLE STATUS FIELDS:")
print(terms["status_candidates"])

# ---------- 6. Summarize candidate-well observation coverage ----------

well_coverage = (
    candidate_obs.groupby("Well_key", dropna=False)
    .size()
    .rename("observation_rows")
    .reset_index()
    .sort_values("observation_rows", ascending=False)
)

well_coverage.to_csv(
    OUT / "02_candidate_well_observation_coverage.csv",
    index=False
)

# ---------- 7. Save diagnostic summary ----------

summary = {
    "step": "15B_Part1_Observation_Diagnostics",
    "source_observation_file": str(OBS_FILE),
    "step15a_trend_file": str(TREND_FILE),
    "new_output_directory": str(OUT),
    "step15a_candidate_wells": int(len(candidate_keys)),
    "source_observation_rows": int(len(obs)),
    "candidate_observation_rows": int(len(candidate_obs)),
    "candidate_wells_with_observations": int(
        candidate_obs["Well_key"].nunique()
    ),
    "columns": candidate_obs.columns.tolist(),
    "possible_date_fields": terms["date_candidates"],
    "possible_water_level_fields": terms["water_level_candidates"],
    "possible_status_fields": terms["status_candidates"],
    "step15a_outputs_modified": False,
    "trend_statistics_recalculated": False
}

with open(OUT / "03_step15b_part1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nDIAGNOSTIC OUTPUTS SAVED TO:")
print(OUT)
print("\nStep 15A outputs were not modified.")
print("Trend recalculation has not yet been performed.")


**[orig #135]** **MAIN** Step 15B part 2 — DTW trends for the 14-block domain (90 wells, 180 series, 129 tested)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 15B — PART 2
# Spatially restricted DTW trend recalculation
#
# Primary variable: Depth_to_water_bgl_m
# Separate season tests: pre and post
# No Step 15A files are modified.
# ================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import kendalltau, norm
from statsmodels.stats.multitest import multipletests
from scipy.stats import theilslopes

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_15B_Trend_Recalculation"
)
OUT.mkdir(parents=True, exist_ok=True)

# Reuse `candidate_obs` from Part 1 if available.
# Reload the source and Step 15A trends for reproducibility.
OBS_FILE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)
TREND_FILE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_15_Spatial_Trend_Restriction/"
    "01_spatially_restricted_candidate_trends.csv"
)

obs = pd.read_csv(OBS_FILE, low_memory=False)
candidates = pd.read_csv(TREND_FILE, low_memory=False)

for df in [obs, candidates]:
    df["Well_key"] = df["Well_key"].astype("string").str.strip()

candidate_keys = set(candidates["Well_key"].dropna())
obs = obs[obs["Well_key"].isin(candidate_keys)].copy()

required = [
    "Well_key", "Year", "Season",
    "WL_status", "Depth_to_water_bgl_m"
]
missing = [c for c in required if c not in obs.columns]
if missing:
    raise ValueError(f"Missing required observation fields: {missing}")

obs["Year"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["Depth_to_water_bgl_m"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)
obs["Season"] = obs["Season"].astype("string").str.strip().str.lower()
obs["WL_status"] = obs["WL_status"].astype("string").str.strip()

# Numeric primary observations only; do not infer values for dry/no-reading.
numeric = obs[
    obs["WL_status"].str.lower().eq("numeric reading")
    & obs["Year"].notna()
    & obs["Depth_to_water_bgl_m"].notna()
].copy()

numeric["Year"] = numeric["Year"].astype(int)

# Check for repeated records for a well-year-season.
key_cols = ["Well_key", "Season", "Year"]
duplicates = numeric[numeric.duplicated(key_cols, keep=False)].copy()
duplicates.to_csv(OUT / "10_duplicate_well_season_year_records.csv", index=False)

if not duplicates.empty:
    raise ValueError(
        "Duplicate numeric Well_key + Season + Year records found. "
        "Review 10_duplicate_well_season_year_records.csv before trend testing."
    )

# Mann-Kendall p-value from Kendall tau; Sen slope via pairwise slopes.
# Exact trend direction uses Sen slope:
# positive DTW slope = water table becoming deeper;
# negative DTW slope = water table becoming shallower.
def analyze_series(group):
    g = group.sort_values("Year")
    x = g["Year"].to_numpy(dtype=float)
    y = g["Depth_to_water_bgl_m"].to_numpy(dtype=float)

    n = len(y)
    span = int(x.max() - x.min()) if n else 0
    consecutive = bool(n >= 2 and np.all(np.diff(x) == 1))

    result = {
        "n_numeric_years": n,
        "first_year": int(x.min()) if n else np.nan,
        "last_year": int(x.max()) if n else np.nan,
        "year_span": span,
        "consecutive_annual": consecutive,
        "sen_slope_m_per_year": np.nan,
        "sen_intercept": np.nan,
        "mk_tau": np.nan,
        "mk_p_value": np.nan,
        "trend_direction": "NOT_TESTED",
        "eligibility_status": "NOT_ELIGIBLE"
    }

    # Explicit eligibility criteria
    if n < 8:
        result["eligibility_status"] = "FEWER_THAN_8_NUMERIC_OBSERVATIONS"
        return pd.Series(result)

    if span < 7:
        result["eligibility_status"] = "YEAR_SPAN_BELOW_7"
        return pd.Series(result)

    if not consecutive:
        result["eligibility_status"] = "NONCONSECUTIVE_ANNUAL_RECORDS"
        return pd.Series(result)

    if np.unique(y).size < 2:
        result["eligibility_status"] = "NO_VARIATION"
        return pd.Series(result)

    slope, intercept, lo_slope, hi_slope = theilslopes(y, x)
    tau, p = kendalltau(x, y, method="auto")

    result.update({
        "sen_slope_m_per_year": float(slope),
        "sen_intercept": float(intercept),
        "mk_tau": float(tau),
        "mk_p_value": float(p),
        "trend_direction": (
            "DTW_INCREASING_DEEPER" if slope > 0 else
            "DTW_DECREASING_SHALLOWER" if slope < 0 else
            "NO_SLOPE"
        ),
        "eligibility_status": "TESTED"
    })

    return pd.Series(result)

# Analyze each well-season series.
records = []
for (well, season), group in numeric.groupby(
    ["Well_key", "Season"], dropna=False
):
    r = analyze_series(group).to_dict()
    r["Well_key"] = well
    r["Season"] = season
    records.append(r)

results = pd.DataFrame(records)

# Add series with no numeric observations to the audit.
all_series = candidates[["Well_key", "Season"]].drop_duplicates().copy()
all_series["Season"] = all_series["Season"].astype("string").str.strip().str.lower()

results = all_series.merge(
    results, on=["Well_key", "Season"], how="left", validate="one_to_one"
)

results["eligibility_status"] = results["eligibility_status"].fillna(
    "NO_NUMERIC_OBSERVATIONS"
)
results["trend_direction"] = results["trend_direction"].fillna("NOT_TESTED")

# FDR correction separately for pre and post, using eligible MK p-values.
results["mk_p_fdr_bh"] = np.nan
results["mk_fdr_significant"] = False

for season, idx in results.groupby("Season").groups.items():
    eligible_idx = [
        i for i in idx
        if results.loc[i, "eligibility_status"] == "TESTED"
        and pd.notna(results.loc[i, "mk_p_value"])
    ]

    if eligible_idx:
        pvals = results.loc[eligible_idx, "mk_p_value"].astype(float).to_numpy()
        reject, p_adj, _, _ = multipletests(
            pvals, alpha=0.05, method="fdr_bh"
        )
        results.loc[eligible_idx, "mk_p_fdr_bh"] = p_adj
        results.loc[eligible_idx, "mk_fdr_significant"] = reject

results["trend_classification"] = np.where(
    results["eligibility_status"] != "TESTED",
    results["eligibility_status"],
    np.where(
        results["mk_fdr_significant"],
        "FDR_SIGNIFICANT",
        "NOT_FDR_SIGNIFICANT"
    )
)

# Save detailed outputs.
results.to_csv(
    OUT / "11_step15b_dtw_trend_results.csv", index=False
)

numeric.to_csv(
    OUT / "12_step15b_numeric_dtw_observations.csv", index=False
)

eligibility_summary = (
    results.groupby(["Season", "eligibility_status"], dropna=False)
    .size().reset_index(name="series_count")
)
eligibility_summary.to_csv(
    OUT / "13_step15b_eligibility_summary.csv", index=False
)

trend_summary = (
    results.groupby(
        ["Season", "trend_classification"], dropna=False
    )
    .size().reset_index(name="series_count")
)
trend_summary.to_csv(
    OUT / "14_step15b_trend_summary.csv", index=False
)

summary = {
    "step": "15B_Part2_DTW_Trend_Recalculation",
    "candidate_wells": int(len(candidate_keys)),
    "candidate_wells_with_numeric_dtw": int(numeric["Well_key"].nunique()),
    "numeric_dtw_observation_rows": int(len(numeric)),
    "candidate_well_season_series": int(len(all_series)),
    "tested_series": int((results["eligibility_status"] == "TESTED").sum()),
    "not_tested_series": int((results["eligibility_status"] != "TESTED").sum()),
    "fdr_significant_series": int(results["mk_fdr_significant"].sum()),
    "fdr_method": "Benjamini-Hochberg",
    "fdr_families": "Separate families for pre and post seasons",
    "primary_measure": "Depth_to_water_bgl_m",
    "sen_slope_units": "m/year",
    "positive_slope_interpretation": "DTW increasing; water table becoming deeper",
    "negative_slope_interpretation": "DTW decreasing; water table becoming shallower",
    "tfpw_performed": False,
    "step15a_modified": False
}

with open(OUT / "15_step15b_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 65)
print("STEP 15B — DTW TREND RECALCULATION")
print("=" * 65)
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nELIGIBILITY SUMMARY")
print(eligibility_summary.to_string(index=False))

print("\nTREND SUMMARY")
print(trend_summary.to_string(index=False))

print("\nSaved to:", OUT)
print("Step 15A outputs were not modified.")


**[orig #136]** Step 15C — TFPW sensitivity  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 15C — TFPW SENSITIVITY AND ROBUSTNESS COMPARISON
# Preserves all Step 15A and Step 15B files.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime
from scipy.stats import kendalltau, theilslopes
from statsmodels.stats.multitest import multipletests

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)
STEP15B = BASE / "Step_15B_Trend_Recalculation"
OUT = BASE / "Step_15C_TFPW_Robustness"
OUT.mkdir(parents=True, exist_ok=True)

OBS_FILE = (
    BASE / "Step_2_QC"
    / "estimated_heads_deduplicated_analysis_copy.csv"
)
RESULT_FILE = STEP15B / "11_step15b_dtw_trend_results.csv"

for p in [OBS_FILE, RESULT_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

obs = pd.read_csv(OBS_FILE, low_memory=False)
base_results = pd.read_csv(RESULT_FILE, low_memory=False)

for df in [obs, base_results]:
    df["Well_key"] = df["Well_key"].astype("string").str.strip()

# Only series tested in Step 15B are eligible for this comparison.
tested = base_results[
    base_results["eligibility_status"].eq("TESTED")
].copy()

if tested.empty:
    raise ValueError("No eligible tested series found in Step 15B.")

obs["Year"] = pd.to_numeric(obs["Year"], errors="coerce")
obs["Depth_to_water_bgl_m"] = pd.to_numeric(
    obs["Depth_to_water_bgl_m"], errors="coerce"
)
obs["Season"] = obs["Season"].astype("string").str.strip().str.lower()
obs["WL_status"] = obs["WL_status"].astype("string").str.strip()

obs = obs[
    obs["WL_status"].str.lower().eq("numeric reading")
    & obs["Depth_to_water_bgl_m"].notna()
    & obs["Year"].notna()
    & obs["Well_key"].isin(set(tested["Well_key"]))
].copy()
obs["Year"] = obs["Year"].astype(int)

# TFPW:
# 1. Estimate the Sen trend.
# 2. Detrend the original series.
# 3. Estimate lag-1 correlation of detrended residuals.
# 4. Pre-whiten residuals using lag-1 correlation.
# 5. Restore the trend and test the adjusted series.
#
# This is a sensitivity implementation; it does not guarantee
# complete removal of all serial dependence.

def tfpw_test(group):
    g = group.sort_values("Year")
    x = g["Year"].to_numpy(dtype=float)
    y = g["Depth_to_water_bgl_m"].to_numpy(dtype=float)

    n = len(y)
    result = {
        "tfpw_n": n,
        "tfpw_rho1": np.nan,
        "tfpw_mk_tau": np.nan,
        "tfpw_mk_p_value": np.nan,
        "tfpw_sen_slope_m_per_year": np.nan,
        "tfpw_status": "NOT_TESTED"
    }

    if n < 8 or (x.max() - x.min() < 7):
        result["tfpw_status"] = "INSUFFICIENT_LENGTH"
        return pd.Series(result)

    if not np.all(np.diff(x) == 1):
        result["tfpw_status"] = "NONCONSECUTIVE_YEARS"
        return pd.Series(result)

    if np.unique(y).size < 2:
        result["tfpw_status"] = "NO_VARIATION"
        return pd.Series(result)

    slope, intercept, _, _ = theilslopes(y, x)
    residual = y - (intercept + slope * x)

    if np.std(residual[:-1]) == 0 or np.std(residual[1:]) == 0:
        rho = 0.0
    else:
        rho = float(np.corrcoef(residual[:-1], residual[1:])[0, 1])

    if not np.isfinite(rho):
        result["tfpw_status"] = "INVALID_LAG1_CORRELATION"
        return pd.Series(result)

    # Constrain numerical extremes to avoid unstable transformations.
    rho_used = float(np.clip(rho, -0.99, 0.99))

    # Pre-whiten detrended residuals, then restore the trend.
    whitened_residual = np.empty_like(residual)
    whitened_residual[0] = residual[0]
    whitened_residual[1:] = (
        residual[1:] - rho_used * residual[:-1]
    )
    adjusted = intercept + slope * x + whitened_residual

    tau, p = kendalltau(x, adjusted, method="auto")
    adjusted_slope, _, _, _ = theilslopes(adjusted, x)

    result.update({
        "tfpw_rho1": rho,
        "tfpw_mk_tau": float(tau),
        "tfpw_mk_p_value": float(p),
        "tfpw_sen_slope_m_per_year": float(adjusted_slope),
        "tfpw_status": "TESTED"
    })
    return pd.Series(result)

records = []

for (well, season), group in obs.groupby(
    ["Well_key", "Season"], dropna=False
):
    key = (well, season)
    if key not in set(zip(tested["Well_key"], tested["Season"].astype(str).str.lower())):
        continue

    r = tfpw_test(group).to_dict()
    r["Well_key"] = well
    r["Season"] = season
    records.append(r)

tfpw_results = pd.DataFrame(records)

# Ensure every Step 15B tested series appears in the output.
comparison = tested.copy()
comparison["Season"] = comparison["Season"].astype("string").str.lower()
tfpw_results["Season"] = tfpw_results["Season"].astype("string").str.lower()

comparison = comparison.merge(
    tfpw_results,
    on=["Well_key", "Season"],
    how="left",
    validate="one_to_one"
)

comparison["tfpw_status"] = comparison["tfpw_status"].fillna(
    "NO_MATCHING_OBSERVATION_SERIES"
)

# Apply FDR correction separately by season to valid TFPW tests.
comparison["tfpw_p_fdr_bh"] = np.nan
comparison["tfpw_fdr_significant"] = False

for season, idx in comparison.groupby("Season").groups.items():
    eligible_idx = [
        i for i in idx
        if comparison.loc[i, "tfpw_status"] == "TESTED"
        and pd.notna(comparison.loc[i, "tfpw_mk_p_value"])
    ]
    if eligible_idx:
        pvals = comparison.loc[
            eligible_idx, "tfpw_mk_p_value"
        ].astype(float).to_numpy()
        reject, adjusted_p, _, _ = multipletests(
            pvals, alpha=0.05, method="fdr_bh"
        )
        comparison.loc[eligible_idx, "tfpw_p_fdr_bh"] = adjusted_p
        comparison.loc[eligible_idx, "tfpw_fdr_significant"] = reject

comparison["significance_agreement"] = np.select(
    [
        comparison["mk_fdr_significant"].astype(bool)
        & comparison["tfpw_fdr_significant"].astype(bool),
        comparison["mk_fdr_significant"].astype(bool)
        & ~comparison["tfpw_fdr_significant"].astype(bool),
        ~comparison["mk_fdr_significant"].astype(bool)
        & comparison["tfpw_fdr_significant"].astype(bool),
    ],
    [
        "SIGNIFICANT_IN_BOTH",
        "MK_ONLY",
        "TFPW_ONLY"
    ],
    default="NOT_SIGNIFICANT_IN_EITHER"
)

comparison["slope_direction_agreement"] = (
    np.sign(comparison["sen_slope_m_per_year"])
    == np.sign(comparison["tfpw_sen_slope_m_per_year"])
)
comparison.loc[
    comparison["tfpw_status"] != "TESTED",
    "slope_direction_agreement"
] = False

# Save outputs in the new Step 15C folder.
comparison.to_csv(
    OUT / "01_step15c_tfpw_robustness_comparison.csv",
    index=False
)

agreement = (
    comparison.groupby(
        ["Season", "tfpw_status", "significance_agreement"],
        dropna=False
    )
    .size()
    .reset_index(name="series_count")
)
agreement.to_csv(
    OUT / "02_step15c_robustness_summary.csv",
    index=False
)

summary = {
    "step": "15C_TFPW_Robustness",
    "step15b_tested_series": int(len(tested)),
    "tfpw_tested_series": int((comparison["tfpw_status"] == "TESTED").sum()),
    "tfpw_not_tested_series": int((comparison["tfpw_status"] != "TESTED").sum()),
    "ordinary_mk_fdr_significant": int(
        comparison["mk_fdr_significant"].astype(bool).sum()
    ),
    "tfpw_fdr_significant": int(
        comparison["tfpw_fdr_significant"].sum()
    ),
    "significant_in_both": int(
        (comparison["significance_agreement"] == "SIGNIFICANT_IN_BOTH").sum()
    ),
    "mk_only": int(
        (comparison["significance_agreement"] == "MK_ONLY").sum()
    ),
    "tfpw_only": int(
        (comparison["significance_agreement"] == "TFPW_ONLY").sum()
    ),
    "fdr_method": "Benjamini-Hochberg separately by season",
    "method_note": (
        "TFPW sensitivity implementation; inspect lag-1 correlation "
        "and method assumptions before interpreting as a definitive "
        "serial-correlation correction."
    ),
    "step15a_and_step15b_modified": False,
    "created_at": datetime.now().isoformat(timespec="seconds")
}

with open(OUT / "03_step15c_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 65)
print("STEP 15C — TFPW ROBUSTNESS RESULTS")
print("=" * 65)
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nROBUSTNESS SUMMARY")
print(agreement.to_string(index=False))
print("\nOutputs saved to:", OUT)


**[orig #137]** Step 15D — robust-trend classification (MK vs TFPW)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 15D — TFPW AUDIT AND ROBUST-TREND CLASSIFICATION
# Alwar groundwater PhD | Read-only analysis of existing outputs
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

STEP15B = BASE / "Step_15B_Trend_Recalculation"
STEP15C = BASE / "Step_15C_TFPW_Robustness"
OUT = BASE / "Step_15D_TFPW_Audit"

OUT.mkdir(parents=True, exist_ok=True)

FILE_B = STEP15B / "11_step15b_dtw_trend_results.csv"
FILE_C = STEP15C / "01_step15c_tfpw_robustness_comparison.csv"

for f in [FILE_B, FILE_C]:
    if not f.exists():
        raise FileNotFoundError(f"Required file not found: {f}")

b = pd.read_csv(FILE_B)
c = pd.read_csv(FILE_C)

print("=" * 70)
print("STEP 15D — INPUT SCHEMA AUDIT")
print("=" * 70)
print("\nStep 15B columns:")
print(b.columns.tolist())
print("\nStep 15C columns:")
print(c.columns.tolist())

# ------------------------------------------------
# 1. Identify and standardize well-season keys
# ------------------------------------------------

def find_column(df, candidates, label):
    lower = {str(col).strip().lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    raise KeyError(
        f"Cannot identify {label}. Tried {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )

def standardize_keys(df, label):
    well_col = find_column(
        df, ["Well_key", "well_key"], f"{label} well key"
    )
    season_col = find_column(
        df, ["Season", "Season_clean", "season_clean"],
        f"{label} season"
    )

    result = df.copy()
    result["_well_key"] = result[well_col].astype(str).str.strip()
    result["_season"] = (
        result[season_col].astype(str).str.strip().str.lower()
    )

    result = result[
        result["_season"].isin(["pre", "post"])
    ].copy()

    if result.duplicated(["_well_key", "_season"]).any():
        duplicates = result.loc[
            result.duplicated(["_well_key", "_season"], keep=False),
            ["_well_key", "_season"]
        ]
        raise ValueError(
            f"{label} has duplicate well-season keys. "
            "Resolve these before merging:\n"
            f"{duplicates.head(20)}"
        )
    return result

b = standardize_keys(b, "Step 15B")
c = standardize_keys(c, "Step 15C")

# ------------------------------------------------
# 2. Find key statistical fields without assuming
#    every script uses identical column names
# ------------------------------------------------

def optional_column(df, candidates):
    lower = {str(col).strip().lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    return None

# Step 15B fields
b_slope_col = optional_column(
    b, ["sen_slope_m_per_year", "original_sen_slope_m_per_year",
        "sen_slope", "step15b_sen_slope_m_per_year"]
)
b_p_col = optional_column(
    b, ["mk_p_fdr_bh", "mk_fdr_p", "fdr_p", "p_fdr",
        "mk_fdr_p_value"]
)
b_sig_col = optional_column(
    b, ["mk_fdr_significant", "fdr_significant",
        "mk_significant_fdr", "significant_fdr"]
)

# Step 15C fields
c_sig_col = optional_column(
    c, ["tfpw_fdr_significant", "tfpw_significant_fdr"]
)
c_p_col = optional_column(
    c, ["tfpw_p_fdr_bh", "tfpw_fdr_p", "tfpw_fdr_p_value"]
)
c_rho_col = optional_column(
    c, ["lag1_rho", "lag1_correlation", "lag1_autocorrelation",
        "rho_lag1", "residual_lag1_rho", "ar1_rho"]
)
c_slope_col = optional_column(
    c, ["tfpw_sen_slope_m_per_year", "tfpw_sen_slope",
        "adjusted_sen_slope_m_per_year"]
)

print("\nDetected fields:")
for name, value in [
    ("Step 15B Sen slope", b_slope_col),
    ("Step 15B FDR p-value", b_p_col),
    ("Step 15B FDR significance", b_sig_col),
    ("Step 15C FDR p-value", c_p_col),
    ("Step 15C FDR significance", c_sig_col),
    ("Step 15C lag-1 correlation", c_rho_col),
    ("Step 15C adjusted-series slope", c_slope_col),
]:
    print(f"  {name}: {value}")

# ------------------------------------------------
# 3. Merge without altering original input files
# ------------------------------------------------

keep_b = ["_well_key", "_season"]
for col in [b_slope_col, b_p_col, b_sig_col]:
    if col and col not in keep_b:
        keep_b.append(col)

keep_c = ["_well_key", "_season"]
for col in [c_p_col, c_sig_col, c_rho_col, c_slope_col]:
    if col and col not in keep_c:
        keep_c.append(col)

audit = b[keep_b].merge(
    c[keep_c],
    on=["_well_key", "_season"],
    how="outer",
    indicator=True,
    suffixes=("_step15b", "_step15c")
)

audit = audit.rename(columns={
    "_well_key": "Well_key",
    "_season": "Season"
})

audit["step15b_present"] = audit["_merge"].isin(
    ["both", "left_only"]
)
audit["step15c_present"] = audit["_merge"].isin(
    ["both", "right_only"]
)

# Convert significance fields to nullable booleans.
def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")
    mapping = {
        "true": True, "1": True, "yes": True, "significant": True,
        "false": False, "0": False, "no": False,
        "not significant": False
    }
    return series.astype(str).str.strip().str.lower().map(mapping).astype(
        "boolean"
    )

if b_sig_col:
    audit["ordinary_mk_significant"] = to_bool(audit[b_sig_col])
else:
    audit["ordinary_mk_significant"] = pd.Series(
        pd.NA, index=audit.index, dtype="boolean"
    )

if c_sig_col:
    audit["tfpw_significant"] = to_bool(audit[c_sig_col])
else:
    audit["tfpw_significant"] = pd.Series(
        pd.NA, index=audit.index, dtype="boolean"
    )

def classify(row):
    mk = row["ordinary_mk_significant"]
    tfpw = row["tfpw_significant"]

    if pd.isna(mk) or pd.isna(tfpw):
        return "SIGNIFICANCE_FIELDS_UNAVAILABLE"
    if mk and tfpw:
        return "SIGNIFICANT_IN_BOTH"
    if mk and not tfpw:
        return "MK_ONLY"
    if not mk and tfpw:
        return "TFPW_ONLY"
    return "NOT_SIGNIFICANT_IN_EITHER"

audit["robustness_class"] = audit.apply(classify, axis=1)

# Original DTW Sen slope remains the primary physical trend estimate.
if b_slope_col:
    audit["primary_original_sen_slope_m_per_year"] = pd.to_numeric(
        audit[b_slope_col], errors="coerce"
    )
    slope = audit["primary_original_sen_slope_m_per_year"]
    audit["original_trend_direction"] = np.select(
        [slope > 0, slope < 0, slope == 0],
        ["DEEPENING_DTW", "SHALLOWING_DTW", "ZERO_SLOPE"],
        default="SLOPE_UNAVAILABLE"
    )
else:
    audit["primary_original_sen_slope_m_per_year"] = np.nan
    audit["original_trend_direction"] = "SLOPE_UNAVAILABLE"

# If TFPW slope exists, flag it for methodological review.
if c_slope_col:
    audit["tfpw_reported_slope_for_review"] = pd.to_numeric(
        audit[c_slope_col], errors="coerce"
    )
    audit["slope_difference_for_review"] = (
        audit["tfpw_reported_slope_for_review"]
        - audit["primary_original_sen_slope_m_per_year"]
    )

# Do not label direction as robust based on significance alone.
audit["robust_significant_direction"] = np.where(
    audit["robustness_class"].eq("SIGNIFICANT_IN_BOTH"),
    audit["original_trend_direction"],
    "NOT_ROBUSTLY_SIGNIFICANT"
)

# ------------------------------------------------
# 4. Save series-level audit and summaries
# ------------------------------------------------

audit = audit.drop(columns=["_merge"], errors="ignore")
audit.to_csv(OUT / "01_step15d_series_audit.csv", index=False)

summary = (
    audit.groupby(["Season", "robustness_class"], dropna=False)
    .size().reset_index(name="series_count")
)
summary.to_csv(
    OUT / "02_step15d_robustness_summary_by_season.csv",
    index=False
)

direction_summary = (
    audit[audit["robustness_class"] == "SIGNIFICANT_IN_BOTH"]
    .groupby(["Season", "original_trend_direction"], dropna=False)
    .size().reset_index(name="series_count")
)
direction_summary.to_csv(
    OUT / "03_step15d_robust_direction_summary.csv",
    index=False
)

# Lag-1 diagnostic, if present in the Step 15C output.
if c_rho_col:
    audit["lag1_rho_for_review"] = pd.to_numeric(
        audit[c_rho_col], errors="coerce"
    )
    rho_summary = (
        audit.groupby("Season")["lag1_rho_for_review"]
        .agg(
            series_count="count",
            mean_lag1_rho="mean",
            median_lag1_rho="median",
            minimum_lag1_rho="min",
            maximum_lag1_rho="max"
        ).reset_index()
    )
    rho_summary.to_csv(
        OUT / "04_step15d_lag1_diagnostic_summary.csv", index=False
    )
else:
    rho_summary = None

result = {
    "step": "15D_TFPW_Audit",
    "step15b_input_rows": int(len(b)),
    "step15c_input_rows": int(len(c)),
    "merged_well_season_rows": int(len(audit)),
    "robustness_class_counts": {
        str(k): int(v)
        for k, v in audit["robustness_class"].value_counts(
            dropna=False
        ).items()
    },
    "step15b_slope_column_detected": b_slope_col,
    "step15b_fdr_p_column_detected": b_p_col,
    "step15b_significance_column_detected": b_sig_col,
    "step15c_fdr_p_column_detected": c_p_col,
    "step15c_significance_column_detected": c_sig_col,
    "step15c_lag1_column_detected": c_rho_col,
    "step15c_slope_column_detected": c_slope_col,
    "interpretation_note": (
        "Original Step 15B Sen slope is retained as the primary DTW trend "
        "magnitude. Significance agreement is a sensitivity result, not "
        "proof that serial correlation is fully corrected or that trends "
        "are caused by climate change."
    ),
    "original_inputs_modified": False
}

with open(OUT / "05_step15d_summary.json", "w") as f:
    json.dump(result, f, indent=2)

print("\n" + "=" * 70)
print("STEP 15D COMPLETED")
print("=" * 70)
print(json.dumps(result, indent=2))
print("\nSeasonal robustness summary:")
print(summary.to_string(index=False))
print("\nRobust direction summary:")
print(direction_summary.to_string(index=False))
print(f"\nOutputs saved to: {OUT}")


**[orig #138]** Step 15D — lag-1 and slope audit  
<sub>✅ ran OK</sub>

In [ ]:

# STEP 15D — ADDITIONAL LAG-1 AND SLOPE AUDIT
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)
C = BASE / "Step_15C_TFPW_Robustness" / (
    "01_step15c_tfpw_robustness_comparison.csv"
)
B = BASE / "Step_15B_Trend_Recalculation" / (
    "11_step15b_dtw_trend_results.csv"
)
OUT = BASE / "Step_15D_TFPW_Audit"

tfpw = pd.read_csv(C)
mk = pd.read_csv(B)

keys = ["Well_key", "Season"]
tfpw["tfpw_rho1"] = pd.to_numeric(
    tfpw["tfpw_rho1"], errors="coerce"
)

# Lag-1 correlation diagnostics
rho = tfpw.loc[
    tfpw["tfpw_status"].astype(str).str.upper().eq("TESTED")
].copy()

rho_summary = pd.DataFrame([{
    "tested_series_with_rho": int(rho["tfpw_rho1"].notna().sum()),
    "missing_rho": int(rho["tfpw_rho1"].isna().sum()),
    "mean_rho": rho["tfpw_rho1"].mean(),
    "median_rho": rho["tfpw_rho1"].median(),
    "negative_rho_count": int((rho["tfpw_rho1"] < 0).sum()),
    "near_zero_abs_rho_le_0_1": int(
        (rho["tfpw_rho1"].abs() <= 0.1).sum()
    ),
    "positive_rho_count": int((rho["tfpw_rho1"] > 0).sum()),
    "rho_gt_0_3_count": int((rho["tfpw_rho1"] > 0.3).sum()),
    "rho_gt_0_5_count": int((rho["tfpw_rho1"] > 0.5).sum())
}])

rho_summary.to_csv(
    OUT / "06_step15d_lag1_diagnostic_summary.csv", index=False
)
rho.to_csv(
    OUT / "07_step15d_lag1_by_series.csv", index=False
)

# Compare slopes; preserve the original Sen slope as primary.
slope_check = mk[
    keys + ["sen_slope_m_per_year"]
].merge(
    tfpw[
        keys + [
            "tfpw_sen_slope_m_per_year",
            "tfpw_rho1",
            "tfpw_fdr_significant"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

slope_check["slope_difference_for_review"] = (
    slope_check["tfpw_sen_slope_m_per_year"]
    - slope_check["sen_slope_m_per_year"]
)

slope_check["original_slope_direction"] = np.select(
    [
        slope_check["sen_slope_m_per_year"] > 0,
        slope_check["sen_slope_m_per_year"] < 0
    ],
    ["DEEPENING_DTW", "SHALLOWING_DTW"],
    default="ZERO_OR_UNAVAILABLE"
)

slope_check.to_csv(
    OUT / "08_step15d_slope_comparison_for_review.csv",
    index=False
)

print("LAG-1 SUMMARY")
print(rho_summary.to_string(index=False))
print("\nSLOPE COMPARISON PREVIEW")
print(slope_check.head(10).to_string(index=False))
print(f"\nSaved additional diagnostics to: {OUT}")


## 5. Maps, block summaries, clustering (Steps 16–18)

**[orig #139]** Step 16 — maps of DTW trends  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 16 — SPATIAL MAPPING OF ALWAR GROUNDWATER TRENDS
# Study domain: 14 selected blocks
# Primary variable: Depth to water below ground level (DTW)
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Paths
# -----------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

DOMAIN_FILE = (
    BASE / "Step_14F_Spatial_Analysis_Restriction"
    / "01_wells_in_14_block_analysis_domain.csv"
)

TREND_FILE = (
    BASE / "Step_15B_Trend_Recalculation"
    / "11_step15b_dtw_trend_results.csv"
)

AUDIT_FILE = (
    BASE / "Step_15D_TFPW_Audit"
    / "01_step15d_series_audit.csv"
)

GEOJSON_FILE = Path(
    "/content/drive/MyDrive/gw_framework/Shapefile/"
    "LGD_Blocks.geojson"
)

OUT = BASE / "Step_16_Spatial_Trend_Mapping"
MAP_DIR = OUT / "maps"
MAP_DIR.mkdir(parents=True, exist_ok=True)

for f in [DOMAIN_FILE, TREND_FILE, AUDIT_FILE, GEOJSON_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"Required input not found: {f}")

# -----------------------------
# 2. Read inputs and inspect
# -----------------------------
domain = pd.read_csv(DOMAIN_FILE)
trends = pd.read_csv(TREND_FILE)
audit = pd.read_csv(AUDIT_FILE)
polygons = gpd.read_file(GEOJSON_FILE)

print("Domain inventory columns:")
print(domain.columns.tolist())
print("\nTrend columns:")
print(trends.columns.tolist())
print("\nAudit columns:")
print(audit.columns.tolist())
print("\nPolygon columns:")
print(polygons.columns.tolist())

# -----------------------------
# 3. Find required columns
# -----------------------------
def find_col(df, candidates, description, required=True):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise KeyError(
            f"Could not identify {description}. "
            f"Tried {candidates}. Available columns: {df.columns.tolist()}"
        )
    return None

domain_key = find_col(
    domain, ["Well_key", "well_key"], "well key in domain inventory"
)
lat_col = find_col(
    domain, ["Latitude", "latitude", "lat"], "latitude"
)
lon_col = find_col(
    domain, ["Longitude", "longitude", "lon", "long"], "longitude"
)
block_col = find_col(
    domain, ["Block", "block", "Block_clean", "block_name"],
    "block name", required=False
)

trend_key = find_col(trends, ["Well_key"], "trend well key")
trend_season = find_col(trends, ["Season"], "trend season")
slope_col = find_col(
    trends, ["sen_slope_m_per_year"], "original Sen slope"
)

audit_key = find_col(audit, ["Well_key"], "audit well key")
audit_season = find_col(audit, ["Season"], "audit season")
class_col = find_col(
    audit, ["robustness_class"], "robustness classification"
)

# -----------------------------
# 4. Standardize keys
# -----------------------------
def normalize_key(s):
    return s.astype(str).str.strip()

domain["Well_key_join"] = normalize_key(domain[domain_key])
trends["Well_key_join"] = normalize_key(trends[trend_key])
audit["Well_key_join"] = normalize_key(audit[audit_key])

trends["Season_join"] = (
    trends[trend_season].astype(str).str.strip().str.lower()
)
audit["Season_join"] = (
    audit[audit_season].astype(str).str.strip().str.lower()
)

domain[lat_col] = pd.to_numeric(domain[lat_col], errors="coerce")
domain[lon_col] = pd.to_numeric(domain[lon_col], errors="coerce")

# Validate unique well inventory.
if domain["Well_key_join"].duplicated().any():
    raise ValueError(
        "Domain inventory has duplicate Well_key values. "
        "Resolve duplicate keys before mapping."
    )

if trends.duplicated(["Well_key_join", "Season_join"]).any():
    raise ValueError(
        "Step 15B has duplicate well-season records."
    )

if audit.duplicated(["Well_key_join", "Season_join"]).any():
    raise ValueError(
        "Step 15D audit has duplicate well-season records."
    )

# -----------------------------
# 5. Prepare the spatial domain
# -----------------------------
# Known LGD block codes for the 14-block study domain.
# Filter by code, not by current district name, because administrative
# district labels may have changed since the source workbook was created.
block_code_col = find_col(
    polygons, ["block_lgd"], "LGD block code"
)

polygons[block_code_col] = pd.to_numeric(
    polygons[block_code_col], errors="coerce"
)

target_block_codes = list(range(574, 588))
study_polygons = polygons[
    polygons[block_code_col].isin(target_block_codes)
].copy()

if study_polygons.empty:
    raise ValueError(
        "No polygons matched LGD block codes 574–587. "
        "Inspect polygon fields and block codes before proceeding."
    )

# Calculate map area in a projected CRS if needed.
if study_polygons.crs is None:
    raise ValueError("The LGD GeoJSON has no CRS defined.")

if study_polygons.crs.to_epsg() != 4326:
    study_polygons = study_polygons.to_crs("EPSG:4326")

# Use a projected CRS for plotting, preserving the correct geometry.
study_polygons_plot = study_polygons.to_crs("EPSG:32643")

# -----------------------------
# 6. Join well coordinates, trends and audit
# -----------------------------
domain_keep = [
    "Well_key_join", lat_col, lon_col
]
if block_col:
    domain_keep.append(block_col)

well_info = domain[domain_keep].copy()
well_info = well_info.rename(columns={
    lat_col: "Latitude",
    lon_col: "Longitude"
})
if block_col:
    well_info = well_info.rename(columns={block_col: "Block"})

trend_keep = [
    "Well_key_join", "Season_join", slope_col
]
trend_part = trends[trend_keep].copy().rename(columns={
    slope_col: "sen_slope_m_per_year"
})

audit_keep = [
    "Well_key_join", "Season_join", class_col
]
audit_part = audit[audit_keep].copy().rename(columns={
    class_col: "robustness_class"
})

# Use inner join to keep only Step 15B trend series.
mapped = trend_part.merge(
    well_info,
    on="Well_key_join",
    how="left",
    validate="many_to_one",
    indicator="coordinate_join"
)

mapped = mapped.merge(
    audit_part,
    on=["Well_key_join", "Season_join"],
    how="left",
    validate="one_to_one"
)

mapped["coordinate_valid"] = (
    mapped["Latitude"].between(-90, 90)
    & mapped["Longitude"].between(-180, 180)
)

mapped["robustness_class"] = (
    mapped["robustness_class"]
    .fillna("SIGNIFICANCE_FIELDS_UNAVAILABLE")
)

# Create an explicit category for spatial communication.
mapped["map_class"] = mapped["robustness_class"].replace({
    "SIGNIFICANT_IN_BOTH": "Significant in both",
    "MK_ONLY": "Ordinary MK only",
    "TFPW_ONLY": "TFPW only",
    "NOT_SIGNIFICANT_IN_EITHER": "Neither significant",
    "SIGNIFICANCE_FIELDS_UNAVAILABLE": "TFPW not assessed"
})

mapped["trend_direction"] = np.select(
    [
        mapped["sen_slope_m_per_year"] > 0,
        mapped["sen_slope_m_per_year"] < 0,
        mapped["sen_slope_m_per_year"] == 0
    ],
    ["Deepening DTW", "Shallowing DTW", "Zero slope"],
    default="Slope unavailable"
)

# Save full records before plotting.
mapped.to_csv(
    OUT / "01_step16_all_mapped_well_season_trends.csv",
    index=False
)

# Report coordinate join failures.
unmatched = mapped[
    mapped["Latitude"].isna() | mapped["Longitude"].isna()
].copy()
unmatched.to_csv(
    OUT / "02_step16_trends_missing_coordinates.csv",
    index=False
)

# -----------------------------
# 7. Create point GeoDataFrame
# -----------------------------
valid = mapped[
    mapped["coordinate_valid"]
    & mapped["Latitude"].notna()
    & mapped["Longitude"].notna()
].copy()

points = gpd.GeoDataFrame(
    valid,
    geometry=gpd.points_from_xy(
        valid["Longitude"], valid["Latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32643")

# Check which points fall inside the selected block polygons.
domain_union = study_polygons_plot.geometry.union_all()
points["inside_study_polygon"] = points.geometry.apply(
    domain_union.covers
)

points.to_crs("EPSG:4326").drop(
    columns="geometry"
).to_csv(
    OUT / "03_step16_coordinate_and_boundary_qc.csv",
    index=False
)

# -----------------------------
# 8. Plot each season
# -----------------------------
# Sequential colour scale for signed Sen slope.
# Diverging map makes negative vs positive slope easier to distinguish.
for season in ["pre", "post"]:
    p = points[points["Season_join"] == season].copy()

    if p.empty:
        warnings.warn(f"No mapped points for season {season}")
        continue

    fig, ax = plt.subplots(figsize=(11, 10))
    study_polygons_plot.boundary.plot(
        ax=ax, linewidth=0.7
    )

    # Plot all well-season points with the original Sen slope.
    p.plot(
        ax=ax,
        column="sen_slope_m_per_year",
        cmap="RdBu_r",
        legend=True,
        markersize=35,
        edgecolor="black",
        linewidth=0.25,
        missing_kwds={"color": "lightgrey"},
        legend_kwds={
            "label": "Original Sen slope (m/year)",
            "shrink": 0.7
        }
    )

    ax.set_title(
        f"Alwar groundwater DTW trend — "
        f"{'Pre-monsoon' if season == 'pre' else 'Post-monsoon'}"
    )
    ax.set_xlabel("Easting (m), UTM Zone 43N")
    ax.set_ylabel("Northing (m), UTM Zone 43N")
    ax.set_aspect("equal")
    fig.tight_layout()
    fig.savefig(
        MAP_DIR / f"step16_{season}_sen_slope.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

    # Robustness category map.
    fig, ax = plt.subplots(figsize=(11, 10))
    study_polygons_plot.boundary.plot(
        ax=ax, linewidth=0.7
    )

    categories = [
        "Significant in both",
        "Ordinary MK only",
        "TFPW only",
        "Neither significant",
        "TFPW not assessed"
    ]
    markers = ["o", "s", "^", "x", "D"]

    for category, marker in zip(categories, markers):
        subset = p[p["map_class"] == category]
        if not subset.empty:
            subset.plot(
                ax=ax,
                color=None,
                marker=marker,
                markersize=42,
                edgecolor="black" if marker != "x" else None,
                linewidth=0.4,
                label=category
            )

    ax.set_title(
        f"Alwar groundwater trend robustness — "
        f"{'Pre-monsoon' if season == 'pre' else 'Post-monsoon'}"
    )
    ax.set_xlabel("Easting (m), UTM Zone 43N")
    ax.set_ylabel("Northing (m), UTM Zone 43N")
    ax.legend(loc="best", fontsize=8)
    ax.set_aspect("equal")
    fig.tight_layout()
    fig.savefig(
        MAP_DIR / f"step16_{season}_robustness.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

# -----------------------------
# 9. Seasonal and block summaries
# -----------------------------
season_summary = (
    mapped.groupby(
        ["Season_join", "map_class"], dropna=False
    ).size().reset_index(name="series_count")
)
season_summary.to_csv(
    OUT / "04_step16_seasonal_robustness_summary.csv",
    index=False
)

if "Block" in mapped.columns:
    block_summary = (
        mapped.groupby(
            ["Block", "Season_join", "map_class"], dropna=False
        ).size().reset_index(name="series_count")
    )
    block_summary.to_csv(
        OUT / "05_step16_block_robustness_summary.csv",
        index=False
    )

direction_summary = (
    mapped.groupby(
        ["Season_join", "trend_direction"], dropna=False
    ).size().reset_index(name="series_count")
)
direction_summary.to_csv(
    OUT / "06_step16_direction_summary.csv",
    index=False
)

# -----------------------------
# 10. Save QC summary
# -----------------------------
summary = {
    "step": "16_Spatial_Trend_Mapping",
    "study_block_codes": target_block_codes,
    "polygon_features_selected": int(len(study_polygons)),
    "step15b_well_season_trend_rows": int(len(trends)),
    "mapped_well_season_rows": int(len(mapped)),
    "rows_with_valid_coordinates": int(mapped["coordinate_valid"].sum()),
    "rows_missing_coordinates": int(len(unmatched)),
    "valid_coordinate_points_inside_polygon": int(
        points["inside_study_polygon"].sum()
    ),
    "valid_coordinate_points_outside_polygon": int(
        (~points["inside_study_polygon"]).sum()
    ),
    "primary_slope": "Original Step 15B Sen slope (m/year)",
    "interpolation_performed": False,
    "original_input_files_modified": False,
    "output_directory": str(OUT)
}

with open(OUT / "07_step16_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("STEP 16 COMPLETED")
print("=" * 70)
print(json.dumps(summary, indent=2))
print("\nSeasonal robustness summary:")
print(season_summary.to_string(index=False))
print("\nDirection summary:")
print(direction_summary.to_string(index=False))
print(f"\nMaps saved in: {MAP_DIR}")
print(f"All outputs saved in: {OUT}")


**[orig #140]** Step 17 — block-level trend assessment  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 17 — BLOCK-LEVEL GROUNDWATER TREND ASSESSMENT
# Study: Alwar, Rajasthan | 14-block analysis domain
# Primary variable: Depth to water below ground level (DTW)
# Seasons: Pre-monsoon and post-monsoon
# ================================================================

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------
# 1. INPUT AND OUTPUT PATHS
# ------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

DOMAIN_FILE = (
    BASE / "Step_14F_Spatial_Analysis_Restriction"
    / "01_wells_in_14_block_analysis_domain.csv"
)

TREND_FILE = (
    BASE / "Step_15B_Trend_Recalculation"
    / "11_step15b_dtw_trend_results.csv"
)

AUDIT_FILE = (
    BASE / "Step_15D_TFPW_Audit"
    / "01_step15d_series_audit.csv"
)

OUT = BASE / "Step_17_Block_Level_Trend_Assessment"
MAP_DIR = OUT / "figures"

OUT.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

for f in [DOMAIN_FILE, TREND_FILE, AUDIT_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"Missing input file: {f}")

# ------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------
domain = pd.read_csv(DOMAIN_FILE)
trends = pd.read_csv(TREND_FILE)
audit = pd.read_csv(AUDIT_FILE)

print("Domain rows:", len(domain))
print("Step 15B trend rows:", len(trends))
print("Step 15D audit rows:", len(audit))

print("\nDomain columns:", domain.columns.tolist())
print("Trend columns:", trends.columns.tolist())
print("Audit columns:", audit.columns.tolist())

# ------------------------------------------------
# 3. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------
required = {
    "domain": ["Well_key", "block"],
    "trends": [
        "Well_key", "Season", "sen_slope_m_per_year",
        "mk_fdr_significant"
    ],
    "audit": [
        "Well_key", "Season", "robustness_class",
        "tfpw_significant"
    ]
}

for name, cols in required.items():
    df = {"domain": domain, "trends": trends, "audit": audit}[name]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

# ------------------------------------------------
# 4. NORMALIZE BLOCK NAMES AND KEYS
# ------------------------------------------------
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip()).upper()

def canonical_block(value):
    name = clean_text(value)
    aliases = {
        "KISHANGARH": "KISHANGARH BAS",
        "KISHANGARH BAS": "KISHANGARH BAS",
        "UMRAIN": "UMREN",
        "UMREN": "UMREN",
        "BANSUR": "BANSUR",
        "LAXMANGARH": "LAXMANGARH",
        "KOTKASIM": "KOTKASIM",
        "NEEMRANA": "NEEMRANA",
        "THANAGAZI": "THANAGAZI",
        "RAMGARH": "RAMGARH",
        "RAJGARH": "RAJGARH",
        "RENI": "RENI",
        "MANDAWAR": "MANDAWAR",
        "TIJARA": "TIJARA",
        "BEHROR": "BEHROR",
        "KATHUMAR": "KATHUMAR"
    }
    return aliases.get(name, name)

domain["Well_key"] = domain["Well_key"].astype(str).str.strip()
trends["Well_key"] = trends["Well_key"].astype(str).str.strip()
audit["Well_key"] = audit["Well_key"].astype(str).str.strip()

domain["Block_clean"] = domain["block"].apply(canonical_block)
trends["Season_clean"] = (
    trends["Season"].astype(str).str.strip().str.lower()
)
audit["Season_clean"] = (
    audit["Season"].astype(str).str.strip().str.lower()
)

allowed_seasons = ["pre", "post"]
trends = trends[trends["Season_clean"].isin(allowed_seasons)].copy()
audit = audit[audit["Season_clean"].isin(allowed_seasons)].copy()

# The domain inventory must have one row per well.
if domain["Well_key"].duplicated().any():
    duplicated = domain.loc[
        domain["Well_key"].duplicated(keep=False), "Well_key"
    ].unique()
    raise ValueError(
        "Duplicate well keys in domain inventory: "
        + ", ".join(duplicated[:20])
    )

if trends.duplicated(["Well_key", "Season_clean"]).any():
    raise ValueError("Duplicate well-season records in Step 15B.")

if audit.duplicated(["Well_key", "Season_clean"]).any():
    raise ValueError("Duplicate well-season records in Step 15D.")

# ------------------------------------------------
# 5. JOIN DOMAIN, TREND AND ROBUSTNESS INFORMATION
# ------------------------------------------------
domain_keep = domain[[
    "Well_key", "Block_clean"
]].copy()

# Retain block assignment only from the validated domain inventory.
records = trends.merge(
    domain_keep,
    on="Well_key",
    how="left",
    validate="many_to_one",
    indicator="domain_join"
)

audit_keep = audit[[
    "Well_key", "Season_clean", "robustness_class",
    "tfpw_significant"
]].copy()

records = records.merge(
    audit_keep,
    on=["Well_key", "Season_clean"],
    how="left",
    validate="one_to_one"
)

records["sen_slope_m_per_year"] = pd.to_numeric(
    records["sen_slope_m_per_year"], errors="coerce"
)

records["robustness_class"] = records["robustness_class"].fillna(
    "SIGNIFICANCE_FIELDS_UNAVAILABLE"
)

records["trend_direction"] = np.select(
    [
        records["sen_slope_m_per_year"] > 0,
        records["sen_slope_m_per_year"] < 0,
        records["sen_slope_m_per_year"] == 0
    ],
    [
        "DEEPENING_DTW",
        "SHALLOWING_DTW",
        "ZERO_SLOPE"
    ],
    default="SLOPE_UNAVAILABLE"
)

# Check whether Step 15B well-season records all joined to the domain.
unmatched = records[records["Block_clean"].isna()].copy()
unmatched.to_csv(
    OUT / "01_step17_unmatched_trend_records.csv",
    index=False
)

if len(unmatched) > 0:
    warnings.warn(
        f"{len(unmatched)} trend records did not join to a study block. "
        "They will be excluded from block summaries and retained in QC."
    )

records = records[records["Block_clean"].notna()].copy()

records.to_csv(
    OUT / "02_step17_joined_well_season_records.csv",
    index=False
)

# ------------------------------------------------
# 6. DEFINE ALL 14 BLOCKS
# ------------------------------------------------
expected_blocks = [
    "BANSUR", "BEHROR", "KATHUMAR", "KISHANGARH BAS",
    "KOTKASIM", "LAXMANGARH", "MANDAWAR", "NEEMRANA",
    "RAJGARH", "RAMGARH", "RENI", "THANAGAZI",
    "TIJARA", "UMREN"
]

observed_blocks = sorted(domain["Block_clean"].dropna().unique())
print("\nObserved block names:", observed_blocks)

unexpected_blocks = sorted(
    set(observed_blocks) - set(expected_blocks)
)
missing_blocks = sorted(
    set(expected_blocks) - set(observed_blocks)
)

if unexpected_blocks or missing_blocks:
    print("WARNING: Block-name reconciliation required.")
    print("Unexpected block labels:", unexpected_blocks)
    print("Expected blocks not found:", missing_blocks)

# Use the validated inventory's observed labels in the summary.
# This avoids silently assigning a well to an unverified block.
block_list = observed_blocks
seasons = ["pre", "post"]

# ------------------------------------------------
# 7. SUMMARIZE EACH BLOCK AND SEASON
# ------------------------------------------------
summary_rows = []

for block in block_list:
    for season in seasons:
        sub = records[
            (records["Block_clean"] == block)
            & (records["Season_clean"] == season)
        ].copy()

        slopes = sub["sen_slope_m_per_year"].dropna()

        # Robustness categories are taken from the Step 15D audit.
        cls = sub["robustness_class"].astype(str)

        n_series = len(sub)
        n_wells = sub["Well_key"].nunique()
        n_slopes = int(slopes.size)

        # Slope summaries are descriptive summaries of well-level
        # Sen slopes, not a block-scale water-storage estimate.
        if n_slopes > 0:
            median_slope = float(slopes.median())
            mean_slope = float(slopes.mean())
            q25 = float(slopes.quantile(0.25))
            q75 = float(slopes.quantile(0.75))
            minimum = float(slopes.min())
            maximum = float(slopes.max())
        else:
            median_slope = mean_slope = q25 = q75 = np.nan
            minimum = maximum = np.nan

        summary_rows.append({
            "Block": block,
            "Season": season,
            "n_wells": int(n_wells),
            "n_well_season_series": int(n_series),
            "n_series_with_slope": n_slopes,
            "median_sen_slope_m_per_year": median_slope,
            "mean_sen_slope_m_per_year": mean_slope,
            "sen_slope_q25_m_per_year": q25,
            "sen_slope_q75_m_per_year": q75,
            "sen_slope_min_m_per_year": minimum,
            "sen_slope_max_m_per_year": maximum,
            "n_deepening_dtw": int(
                (sub["trend_direction"] == "DEEPENING_DTW").sum()
            ),
            "n_shallowing_dtw": int(
                (sub["trend_direction"] == "SHALLOWING_DTW").sum()
            ),
            "n_zero_slope": int(
                (sub["trend_direction"] == "ZERO_SLOPE").sum()
            ),
            "n_significant_both": int(
                (cls == "SIGNIFICANT_IN_BOTH").sum()
            ),
            "n_mk_only": int((cls == "MK_ONLY").sum()),
            "n_tfpw_only": int((cls == "TFPW_ONLY").sum()),
            "n_neither_significant": int(
                (cls == "NOT_SIGNIFICANT_IN_EITHER").sum()
            ),
            "n_tfpw_not_assessed": int(
                (cls == "SIGNIFICANCE_FIELDS_UNAVAILABLE").sum()
            )
        })

block_summary = pd.DataFrame(summary_rows)

block_summary.to_csv(
    OUT / "03_step17_block_season_summary.csv",
    index=False
)

# ------------------------------------------------
# 8. LONG-FORM ROBUSTNESS COUNTS BY BLOCK AND SEASON
# ------------------------------------------------
robustness = (
    records.groupby(
        ["Block_clean", "Season_clean", "robustness_class"],
        dropna=False
    )
    .agg(
        series_count=("Well_key", "size"),
        unique_wells=("Well_key", "nunique")
    )
    .reset_index()
    .rename(columns={
        "Block_clean": "Block",
        "Season_clean": "Season"
    })
)

robustness.to_csv(
    OUT / "04_step17_robustness_by_block_season.csv",
    index=False
)

# ------------------------------------------------
# 9. DIRECTION COUNTS BY BLOCK AND SEASON
# ------------------------------------------------
direction = (
    records.groupby(
        ["Block_clean", "Season_clean", "trend_direction"],
        dropna=False
    )
    .agg(
        series_count=("Well_key", "size"),
        unique_wells=("Well_key", "nunique")
    )
    .reset_index()
    .rename(columns={
        "Block_clean": "Block",
        "Season_clean": "Season"
    })
)

direction.to_csv(
    OUT / "05_step17_direction_by_block_season.csv",
    index=False
)

# ------------------------------------------------
# 10. FIGURE: MEDIAN SEN SLOPE BY BLOCK
# ------------------------------------------------
for season in seasons:
    plot_df = block_summary[
        block_summary["Season"] == season
    ].copy()

    plot_df = plot_df.sort_values(
        "median_sen_slope_m_per_year",
        na_position="last"
    )

    if plot_df["median_sen_slope_m_per_year"].notna().sum() == 0:
        print(f"No slopes available for {season}; skipping plot.")
        continue

    fig, ax = plt.subplots(figsize=(11, 7))

    vals = plot_df["median_sen_slope_m_per_year"]
    labels = plot_df["Block"]

    ax.barh(labels, vals)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Median original Sen slope (m/year)")
    ax.set_ylabel("Block")
    ax.set_title(
        "Block-level median DTW trend — "
        + ("Pre-monsoon" if season == "pre" else "Post-monsoon")
    )
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        MAP_DIR / f"step17_{season}_block_median_sen_slope.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

# ------------------------------------------------
# 11. FIGURE: ROBUST SIGNIFICANCE COUNTS
# ------------------------------------------------
for season in seasons:
    plot_df = block_summary[
        block_summary["Season"] == season
    ].copy().sort_values("n_significant_both")

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(plot_df["Block"], plot_df["n_significant_both"])
    ax.set_xlabel("Well-season series significant in both methods")
    ax.set_ylabel("Block")
    ax.set_title(
        "Robustly significant groundwater trends — "
        + ("Pre-monsoon" if season == "pre" else "Post-monsoon")
    )
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        MAP_DIR / f"step17_{season}_robust_significance_counts.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

# ------------------------------------------------
# 12. QUALITY-CONTROL SUMMARY
# ------------------------------------------------
qc = {
    "step": "17_Block_Level_Trend_Assessment",
    "domain_inventory_wells": int(len(domain)),
    "step15b_input_series": int(len(trends)),
    "joined_series_in_domain": int(len(records)),
    "unmatched_trend_series": int(len(unmatched)),
    "observed_block_labels": observed_blocks,
    "expected_block_labels_not_found": missing_blocks,
    "unexpected_block_labels": unexpected_blocks,
    "seasons": seasons,
    "primary_trend_measure": "Original Step 15B Sen slope in m/year",
    "block_slope_summary": (
        "Descriptive median, mean, quartiles, minimum and maximum of "
        "well-level Sen slopes; not a groundwater storage estimate."
    ),
    "tfpw_unassessed_kept_separate": True,
    "interpolation_performed": False,
    "original_inputs_modified": False,
    "output_directory": str(OUT)
}

with open(OUT / "06_step17_summary.json", "w") as f:
    json.dump(qc, f, indent=2)

print("\n" + "=" * 70)
print("STEP 17 COMPLETED")
print("=" * 70)
print(json.dumps(qc, indent=2))

print("\nBLOCK-SEASON SUMMARY")
print(block_summary.to_string(index=False))

print("\nSaved outputs:", OUT)
print("Figures:", MAP_DIR)


**[orig #141]** Step 18A — spatial clustering (Moran's I) and bootstrap uncertainty  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 18A — SPATIAL CLUSTERING AND UNCERTAINTY
# Alwar groundwater research | pre- and post-monsoon
#
# Primary measure: original Step 15B Sen slope (m/year)
# Spatial test: Global Moran's I with permutation inference
# Sensitivity: k = 4, 6, 8 nearest neighbours
# Uncertainty: bootstrap confidence intervals for block medians
#
# Does not modify Steps 14–17.
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.stats import rankdata

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

DOMAIN_FILE = (
    BASE / "Step_14F_Spatial_Analysis_Restriction"
    / "01_wells_in_14_block_analysis_domain.csv"
)

STEP16_FILE = (
    BASE / "Step_16_Spatial_Trend_Mapping"
    / "01_step16_all_mapped_well_season_trends.csv"
)

STEP17_FILE = (
    BASE / "Step_17_Block_Level_Trend_Assessment"
    / "02_step17_joined_well_season_records.csv"
)

OUT = BASE / "Step_18_Spatial_Clustering_Uncertainty"
FIG = OUT / "figures"

OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

for path in [DOMAIN_FILE, STEP16_FILE, STEP17_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")

domain = pd.read_csv(DOMAIN_FILE)
spatial = pd.read_csv(STEP16_FILE)
step17 = pd.read_csv(STEP17_FILE)

print("Domain wells:", len(domain))
print("Step 16 rows:", len(spatial))
print("Step 17 rows:", len(step17))
print("\nStep 16 columns:")
print(spatial.columns.tolist())

# ------------------------------------------------
# 2. Standardize keys and discover coordinate fields
# ------------------------------------------------
def find_col(df, candidates, description):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(
        f"Could not find {description}. "
        f"Expected one of {candidates}; found {df.columns.tolist()}"
    )

key_col = find_col(
    spatial, ["Well_key", "Well_key_join"], "well key"
)
season_col = find_col(
    spatial, ["Season", "Season_join"], "season"
)
slope_col = find_col(
    spatial, ["sen_slope_m_per_year"], "original Sen slope"
)
lat_col = find_col(
    spatial, ["Latitude", "latitude"], "latitude"
)
lon_col = find_col(
    spatial, ["Longitude", "longitude"], "longitude"
)
block_col = find_col(
    spatial, ["Block", "Block_clean", "block"], "block"
)

# Step 16 may store its robustness classification in map_class or
# robustness_class. Preserve the original labels if available.
robust_col = None
for candidate in ["robustness_class", "map_class"]:
    if candidate in spatial.columns:
        robust_col = candidate
        break

df = spatial.copy()
df["Well_key"] = df[key_col].astype(str).str.strip()
df["Season"] = df[season_col].astype(str).str.strip().str.lower()
df["Block"] = df[block_col].astype(str).str.strip().str.upper()
df["Latitude"] = pd.to_numeric(df[lat_col], errors="coerce")
df["Longitude"] = pd.to_numeric(df[lon_col], errors="coerce")
df["Sen_slope_m_per_year"] = pd.to_numeric(
    df[slope_col], errors="coerce"
)

if robust_col:
    df["Robustness"] = df[robust_col].astype(str)
else:
    df["Robustness"] = "UNAVAILABLE"

df = df[df["Season"].isin(["pre", "post"])].copy()

# Only one record per well and season is valid for this analysis.
if df.duplicated(["Well_key", "Season"]).any():
    raise ValueError(
        "Duplicate well-season records found in Step 16 input."
    )

# Keep only records with usable coordinates and an original Sen slope.
valid = df[
    df["Latitude"].notna()
    & df["Longitude"].notna()
    & df["Sen_slope_m_per_year"].notna()
].copy()

# Confirm that coordinates are geographically plausible.
valid = valid[
    valid["Latitude"].between(-90, 90)
    & valid["Longitude"].between(-180, 180)
].copy()

print("\nEligible slope records by season:")
print(valid.groupby("Season").size())

# ------------------------------------------------
# 3. Project coordinates to metres
# ------------------------------------------------
gdf = gpd.GeoDataFrame(
    valid,
    geometry=gpd.points_from_xy(
        valid["Longitude"], valid["Latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32643")

gdf["x_m"] = gdf.geometry.x
gdf["y_m"] = gdf.geometry.y

# ------------------------------------------------
# 4. Global Moran's I with k-nearest-neighbour weights
# ------------------------------------------------
def make_knn_weights(coords, k):
    """
    Binary, row-standardized k-nearest-neighbour weights.
    Self-neighbours are excluded.
    """
    n = len(coords)
    if n <= k:
        raise ValueError(f"Need more than k={k} observations.")

    distances = cdist(coords, coords)
    np.fill_diagonal(distances, np.inf)

    neighbours = np.argsort(distances, axis=1)[:, :k]
    W = np.zeros((n, n), dtype=float)

    for i in range(n):
        W[i, neighbours[i]] = 1.0

    # Row-standardization.
    W = W / W.sum(axis=1, keepdims=True)
    return W


def morans_i(values, W):
    values = np.asarray(values, dtype=float)
    z = values - values.mean()
    denominator = np.sum(z ** 2)

    if denominator == 0:
        return np.nan

    n = len(values)
    s0 = W.sum()
    return (n / s0) * ((z @ W @ z) / denominator)


def permutation_moran(values, W, n_perm=999, seed=2026):
    """
    Permutation distribution under exchangeability of observed values
    across fixed locations. This is a spatial-randomization test.
    """
    values = np.asarray(values, dtype=float)
    observed = morans_i(values, W)
    rng = np.random.default_rng(seed)
    simulated = np.empty(n_perm, dtype=float)

    for j in range(n_perm):
        simulated[j] = morans_i(rng.permutation(values), W)

    valid_sim = simulated[np.isfinite(simulated)]
    if not np.isfinite(observed) or len(valid_sim) == 0:
        return observed, np.nan, np.nan, np.nan, simulated

    # One-sided alternative: positive spatial autocorrelation.
    p_positive = (
        1 + np.sum(valid_sim >= observed)
    ) / (1 + len(valid_sim))

    # Two-sided descriptive permutation p-value relative to the
    # permutation distribution's centre.
    centre = np.mean(valid_sim)
    p_two_sided = (
        1 + np.sum(
            np.abs(valid_sim - centre) >= abs(observed - centre)
        )
    ) / (1 + len(valid_sim))

    return (
        observed,
        p_positive,
        p_two_sided,
        float(np.std(valid_sim, ddof=1)),
        simulated
    )


moran_rows = []
moran_scatter_rows = []

for season in ["pre", "post"]:
    sub = gdf[gdf["Season"] == season].copy().reset_index(drop=True)

    n = len(sub)
    if n < 10:
        warnings.warn(
            f"{season}: only {n} usable slopes; "
            "spatial-autocorrelation inference may be unstable."
        )

    values = sub["Sen_slope_m_per_year"].to_numpy()
    coords = sub[["x_m", "y_m"]].to_numpy()

    for k in [4, 6, 8]:
        if n <= k:
            continue

        W = make_knn_weights(coords, k)
        observed, p_positive, p_two, perm_sd, simulated = (
            permutation_moran(
                values, W, n_perm=999,
                seed=2026 + k + (0 if season == "pre" else 100)
            )
        )

        moran_rows.append({
            "Season": season,
            "n_series": n,
            "k_nearest_neighbours": k,
            "morans_I": observed,
            "permutation_p_positive_autocorrelation": p_positive,
            "permutation_p_two_sided": p_two,
            "permutation_sd": perm_sd,
            "permutations": 999,
            "interpretation": (
                "Positive spatial autocorrelation"
                if pd.notna(observed) and observed > 0
                else "Negative spatial autocorrelation"
                if pd.notna(observed) and observed < 0
                else "Unavailable"
            )
        })

        # Save values for a diagnostic Moran scatter plot.
        z = values - values.mean()
        spatial_lag = W @ z

        for i in range(n):
            moran_scatter_rows.append({
                "Season": season,
                "k": k,
                "Well_key": sub.loc[i, "Well_key"],
                "Block": sub.loc[i, "Block"],
                "Sen_slope_m_per_year": values[i],
                "centred_slope": z[i],
                "spatial_lag_centred_slope": spatial_lag[i]
            })

moran_results = pd.DataFrame(moran_rows)
moran_results.to_csv(
    OUT / "01_step18_global_morans_i.csv", index=False
)

pd.DataFrame(moran_scatter_rows).to_csv(
    OUT / "02_step18_moran_scatter_data.csv", index=False
)

# ------------------------------------------------
# 5. Bootstrap confidence intervals for block medians
# ------------------------------------------------
def bootstrap_median_ci(values, n_boot=5000, seed=2026):
    """
    Percentile bootstrap CI for a sample median.
    With very small n, intervals are unstable and must be flagged.
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)

    if n == 0:
        return np.nan, np.nan, np.nan

    median = float(np.median(x))

    if n == 1:
        return median, np.nan, np.nan

    rng = np.random.default_rng(seed)
    samples = rng.choice(
        x, size=(n_boot, n), replace=True
    )
    boot_medians = np.median(samples, axis=1)
    low, high = np.quantile(boot_medians, [0.025, 0.975])

    return median, float(low), float(high)


block_rows = []

for (block, season), sub in valid.groupby(["Block", "Season"]):
    slopes = sub["Sen_slope_m_per_year"].dropna().to_numpy()
    n = len(slopes)

    median, ci_low, ci_high = bootstrap_median_ci(
        slopes,
        n_boot=5000,
        seed=2026 + len(block) + (0 if season == "pre" else 100)
    )

    block_rows.append({
        "Block": block,
        "Season": season,
        "n_well_season_series_with_slope": n,
        "n_unique_wells_with_slope": sub["Well_key"].nunique(),
        "median_sen_slope_m_per_year": median,
        "bootstrap_95ci_lower_m_per_year": ci_low,
        "bootstrap_95ci_upper_m_per_year": ci_high,
        "mean_sen_slope_m_per_year": (
            float(np.mean(slopes)) if n else np.nan
        ),
        "q25_sen_slope_m_per_year": (
            float(np.quantile(slopes, 0.25)) if n else np.nan
        ),
        "q75_sen_slope_m_per_year": (
            float(np.quantile(slopes, 0.75)) if n else np.nan
        ),
        "small_sample_flag": (
            "VERY_SMALL_N_LT_5" if n < 5
            else "SMALL_N_5_TO_9" if n < 10
            else "N_AT_LEAST_10"
        ),
        "ci_note": (
            "Not calculated for n=1"
            if n == 1 else
            "Bootstrap interval; interpret cautiously for small n"
        )
    })

block_uncertainty = pd.DataFrame(block_rows)
block_uncertainty.to_csv(
    OUT / "03_step18_block_median_bootstrap_uncertainty.csv",
    index=False
)

# ------------------------------------------------
# 6. Robustness coverage by block and season
# ------------------------------------------------
# Step 16 records include all 180 candidate series. TFPW-not-assessed
# series are retained as a separate category.
coverage = (
    df.groupby(["Block", "Season", "Robustness"], dropna=False)
    .agg(
        series_count=("Well_key", "size"),
        unique_wells=("Well_key", "nunique")
    )
    .reset_index()
)

coverage.to_csv(
    OUT / "04_step18_robustness_coverage_by_block.csv",
    index=False
)

# Summary of missing slopes and spatial coverage.
coverage_summary = (
    df.groupby(["Season"])
    .agg(
        total_candidate_series=("Well_key", "size"),
        series_with_slope=("Sen_slope_m_per_year", "count"),
        unique_wells=("Well_key", "nunique"),
        missing_slope_count=("Sen_slope_m_per_year",
                             lambda s: int(s.isna().sum()))
    )
    .reset_index()
)

coverage_summary.to_csv(
    OUT / "05_step18_seasonal_coverage_summary.csv",
    index=False
)

# ------------------------------------------------
# 7. Moran scatter plots for k=6
# ------------------------------------------------
scatter_df = pd.DataFrame(moran_scatter_rows)

for season in ["pre", "post"]:
    sub = scatter_df[
        (scatter_df["Season"] == season)
        & (scatter_df["k"] == 6)
    ]

    if sub.empty:
        continue

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        sub["centred_slope"],
        sub["spatial_lag_centred_slope"]
    )
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Centred original Sen slope (m/year)")
    ax.set_ylabel("Spatial lag of centred Sen slope")
    ax.set_title(
        "Moran scatter plot, k=6 — "
        + ("Pre-monsoon" if season == "pre" else "Post-monsoon")
    )
    fig.tight_layout()
    fig.savefig(
        FIG / f"step18_{season}_moran_scatter_k6.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

# ------------------------------------------------
# 8. Save audit summary
# ------------------------------------------------
summary = {
    "step": "18A_Spatial_Clustering_Uncertainty",
    "input_step16_rows": int(len(spatial)),
    "valid_coordinate_and_slope_rows": int(len(valid)),
    "seasons": ["pre", "post"],
    "moran_k_values": [4, 6, 8],
    "permutations_per_test": 999,
    "spatial_weights": (
        "Binary k-nearest-neighbour weights, row-standardized"
    ),
    "primary_measure": "Original Step 15B Sen slope (m/year)",
    "block_uncertainty": (
        "Percentile bootstrap intervals for block medians; "
        "n=1 intervals unavailable; small samples flagged."
    ),
    "causation_inferred": False,
    "interpolation_performed": False,
    "original_input_files_modified": False,
    "output_directory": str(OUT)
}

with open(OUT / "06_step18_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("STEP 18A COMPLETED")
print("=" * 70)
print(json.dumps(summary, indent=2))

print("\nGLOBAL MORAN'S I RESULTS")
print(moran_results.to_string(index=False))

print("\nBLOCK MEDIAN UNCERTAINTY")
print(block_uncertainty.to_string(index=False))

print("\nSEASONAL COVERAGE")
print(coverage_summary.to_string(index=False))

print("\nOutputs:", OUT)
print("Figures:", FIG)


**[orig #142]** Step 18B — local Moran's I clusters/outliers (k = 4, 6, 8)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 18B — LOCAL MORAN'S I CLUSTERS AND OUTLIERS
# Primary analysis: k=6, with k=4 and k=8 sensitivity checks
# ================================================================

# Install PySAL packages in Google Colab if needed.
%pip -q install libpysal esda

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd

from libpysal.weights import KNN
from esda.moran import Moran_Local
from statsmodels.stats.multitest import multipletests

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

INPUT = (
    BASE / "Step_16_Spatial_Trend_Mapping"
    / "01_step16_all_mapped_well_season_trends.csv"
)

OUT = BASE / "Step_18_Spatial_Clustering_Uncertainty"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT)

required = [
    "Well_key_join", "Season_join", "Latitude", "Longitude",
    "sen_slope_m_per_year", "Block"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["Season_join"] = (
    df["Season_join"].astype(str).str.lower().str.strip()
)
df["sen_slope_m_per_year"] = pd.to_numeric(
    df["sen_slope_m_per_year"], errors="coerce"
)
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

df = df[
    df["Season_join"].isin(["pre", "post"])
    & df["sen_slope_m_per_year"].notna()
    & df["Latitude"].notna()
    & df["Longitude"].notna()
].copy()

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:32643")

results = []

for season in ["pre", "post"]:
    s = gdf[gdf["Season_join"] == season].copy().reset_index(drop=True)

    # Local Moran requires enough locations to define neighbours.
    if len(s) <= 8:
        print(f"Skipping {season}: insufficient locations.")
        continue

    coords = np.column_stack([s.geometry.x, s.geometry.y])
    values = s["sen_slope_m_per_year"].to_numpy(dtype=float)

    if np.nanstd(values) == 0:
        print(f"Skipping {season}: slopes have zero variance.")
        continue

    for k in [4, 6, 8]:
        w = KNN.from_array(coords, k=k)
        w.transform = "R"

        # Fixed seed for reproducibility.
        np.random.seed(2026 + k + (0 if season == "pre" else 100))

        local = Moran_Local(
            values,
            w,
            transformation="R",
            permutations=999
        )

        # Local p-values are permutation-based.
        p_raw = np.asarray(local.p_sim, dtype=float)
        q = np.asarray(local.q, dtype=int)
        z = (values - values.mean()) / values.std(ddof=0)

        # Spatial lag of standardized values.
        lag_z = np.asarray(w.sparse @ z).ravel()

        # FDR correction within each season and k specification.
        valid_p = np.isfinite(p_raw)
        p_fdr = np.full(len(s), np.nan)

        if valid_p.any():
            p_fdr[valid_p] = multipletests(
                p_raw[valid_p],
                method="fdr_bh"
            )[1]

        # PySAL quadrant codes:
        # 1=High-High, 2=Low-High, 3=Low-Low, 4=High-Low
        quadrant_names = {
            1: "HIGH_HIGH",
            2: "LOW_HIGH",
            3: "LOW_LOW",
            4: "HIGH_LOW"
        }

        for i in range(len(s)):
            significant = (
                np.isfinite(p_fdr[i]) and p_fdr[i] < 0.05
            )

            results.append({
                "Season": season,
                "k_neighbours": k,
                "Well_key": s.loc[i, "Well_key_join"],
                "Block": s.loc[i, "Block"],
                "Latitude": s.loc[i, "Latitude"],
                "Longitude": s.loc[i, "Longitude"],
                "original_sen_slope_m_per_year": values[i],
                "local_morans_I": float(local.Is[i]),
                "local_p_permutation": float(p_raw[i]),
                "local_p_fdr_bh": (
                    float(p_fdr[i]) if np.isfinite(p_fdr[i]) else np.nan
                ),
                "quadrant_code": int(q[i]),
                "quadrant": quadrant_names.get(q[i], "UNCLASSIFIED"),
                "standardized_slope": float(z[i]),
                "spatial_lag_standardized_slope": float(lag_z[i]),
                "fdr_significant": bool(significant),
                "cluster_class": (
                    quadrant_names.get(q[i], "UNCLASSIFIED")
                    if significant else "NOT_SIGNIFICANT_AFTER_FDR"
                )
            })

local_df = pd.DataFrame(results)

if local_df.empty:
    raise ValueError("No local Moran results were produced.")

local_df.to_csv(
    OUT / "07_step18b_local_moran_by_well_season.csv",
    index=False
)

# Summaries: local cluster classes by season and k.
summary = (
    local_df.groupby(
        ["Season", "k_neighbours", "cluster_class"],
        dropna=False
    )
    .size()
    .reset_index(name="series_count")
)

summary.to_csv(
    OUT / "08_step18b_local_cluster_summary.csv",
    index=False
)

# Save a concise diagnostic summary.
audit = {
    "step": "18B_Local_Morans_I",
    "input_file": str(INPUT),
    "valid_well_season_slope_records": int(len(gdf)),
    "neighbourhood_sizes": [4, 6, 8],
    "permutations": 999,
    "multiple_testing_correction": (
        "Benjamini-Hochberg within each season and k specification"
    ),
    "primary_neighbourhood": 6,
    "slope_measure": "Original Step 15B Sen slope (m/year)",
    "interpretation_note": (
        "Local clusters describe spatial similarity in slope values, "
        "not causation or necessarily clustering of significant trends. "
        "Local permutation tests and FDR correction do not remove all "
        "effects of spatially structured sampling."
    ),
    "output_directory": str(OUT)
}

with open(OUT / "09_step18b_summary.json", "w") as f:
    json.dump(audit, f, indent=2)

print("=" * 70)
print("STEP 18B COMPLETED")
print("=" * 70)
print(json.dumps(audit, indent=2))
print("\nLOCAL CLUSTER SUMMARY")
print(summary.to_string(index=False))
print("\nDetailed results saved to:", OUT)


**[orig #143]** Step 18C — cluster stability and spatial-weight audit  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 18C: LOCAL CLUSTER STABILITY + SPATIAL-WEIGHT AUDIT
# Study: Alwar groundwater trends
# Primary neighbourhood: k=6
# Sensitivity: k=4 and k=8
# Original data and previous outputs remain unchanged.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import warnings
import numpy as np
import pandas as pd

from libpysal.weights import KNN
from scipy.sparse.csgraph import connected_components

BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP16 = os.path.join(
    BASE, "Step_16_Spatial_Trend_Mapping",
    "01_step16_all_mapped_well_season_trends.csv"
)

OUTDIR = os.path.join(
    BASE, "Step_18_Spatial_Clustering_Uncertainty"
)

STEP18B = os.path.join(
    OUTDIR, "07_step18b_local_moran_by_well_season.csv"
)

os.makedirs(OUTDIR, exist_ok=True)

# ---------------------------
# 1. Load and inspect inputs
# ---------------------------
if not os.path.exists(STEP16):
    raise FileNotFoundError(f"Missing Step 16 file: {STEP16}")

if not os.path.exists(STEP18B):
    raise FileNotFoundError(f"Missing Step 18B file: {STEP18B}")

spatial = pd.read_csv(STEP16)
local = pd.read_csv(STEP18B)

print("STEP 16 COLUMNS:")
print(spatial.columns.tolist())
print("\nSTEP 18B COLUMNS:")
print(local.columns.tolist())

# ---------------------------
# 2. Find column names safely
# ---------------------------
def find_col(df, candidates, label):
    lookup = {c.lower().strip(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(
        f"Could not identify {label}. "
        f"Tried {candidates}. Available: {df.columns.tolist()}"
    )

well_s = find_col(
    spatial, ["Well_key_join", "well_key", "well_id"],
    "Step 16 well ID"
)
season_s = find_col(
    spatial, ["Season_join", "season"],
    "Step 16 season"
)
lat_s = find_col(
    spatial, ["Latitude", "latitude", "lat"],
    "latitude"
)
lon_s = find_col(
    spatial, ["Longitude", "longitude", "lon"],
    "longitude"
)
slope_s = find_col(
    spatial,
    ["sen_slope_m_per_year", "sen_slope", "sen_slope_m_year"],
    "Sen slope"
)

well_l = find_col(
    local, ["Well_key_join", "well_key", "well_id"],
    "Step 18B well ID"
)
season_l = find_col(
    local, ["Season", "Season_join", "season"],
    "Step 18B season"
)
k_l = find_col(
    local, ["k_neighbours", "k_neighbors", "k"],
    "neighbourhood size"
)
class_l = find_col(
    local, ["cluster_class", "local_cluster_class", "cluster"],
    "cluster class"
)

# Normalize keys for joining
def clean_season(series):
    return series.astype(str).str.strip().str.lower()

spatial["_well"] = spatial[well_s].astype(str).str.strip()
spatial["_season"] = clean_season(spatial[season_s])
local["_well"] = local[well_l].astype(str).str.strip()
local["_season"] = clean_season(local[season_l])

local[k_l] = pd.to_numeric(local[k_l], errors="coerce")
local = local[local[k_l].isin([4, 6, 8])].copy()

# ---------------------------
# 3. Normalize cluster labels
# ---------------------------
def normalize_cluster(value):
    x = str(value).strip().upper().replace("-", "_").replace(" ", "_")
    if "HIGH_HIGH" in x:
        return "HIGH_HIGH"
    if "LOW_LOW" in x:
        return "LOW_LOW"
    if "HIGH_LOW" in x:
        return "HIGH_LOW"
    if "LOW_HIGH" in x:
        return "LOW_HIGH"
    if "NOT_SIGNIFICANT" in x or "NON_SIGNIFICANT" in x:
        return "NOT_SIGNIFICANT"
    if "UNAVAILABLE" in x or "NOT_TESTED" in x:
        return "NOT_TESTED"
    return x

local["cluster_normalized"] = local[class_l].map(normalize_cluster)

# If the Step 18B output contains p-values, inspect whether the
# stored cluster class already reflects FDR correction.
print("\nCluster labels found:")
print(local["cluster_normalized"].value_counts(dropna=False))

# ---------------------------
# 4. Check duplicate records
# ---------------------------
key_cols = ["_well", "_season", k_l]
duplicates = local.duplicated(key_cols, keep=False)

duplicate_report = local.loc[duplicates].copy()
print("\nDuplicate well-season-k records:", int(duplicates.sum()))

if duplicates.any():
    duplicate_report.to_csv(
        os.path.join(OUTDIR, "10_step18c_duplicate_records.csv"),
        index=False
    )
    raise ValueError(
        "Duplicate well-season-k records found. "
        "Inspect 10_step18c_duplicate_records.csv before continuing."
    )

# ---------------------------
# 5. Compare classifications across k
# ---------------------------
wide = local.pivot(
    index=["_well", "_season"],
    columns=k_l,
    values="cluster_normalized"
).reset_index()

wide.columns.name = None
wide = wide.rename(columns={
    4: "cluster_k4",
    6: "cluster_k6",
    8: "cluster_k8"
})

for col in ["cluster_k4", "cluster_k6", "cluster_k8"]:
    if col not in wide.columns:
        wide[col] = np.nan

def stability_class(row):
    vals = [row["cluster_k4"], row["cluster_k6"], row["cluster_k8"]]
    cluster_vals = [
        v for v in vals
        if pd.notna(v) and v not in
        ["NOT_SIGNIFICANT", "NOT_TESTED", ""]
    ]

    if len(cluster_vals) == 3 and len(set(cluster_vals)) == 1:
        return "PERSISTENT_SAME_CLASS_ALL_K"

    if len(cluster_vals) > 0:
        if row["cluster_k6"] in [
            "HIGH_HIGH", "LOW_LOW", "HIGH_LOW", "LOW_HIGH"
        ]:
            return "SIGNIFICANT_AT_PRIMARY_K6_BUT_NOT_STABLE_ALL_K"
        return "CLUSTER_ONLY_AT_SENSITIVITY_K"

    if all(pd.notna(v) for v in vals):
        return "NO_SIGNIFICANT_CLUSTER_AT_ANY_K"

    return "INCOMPLETE_K_COVERAGE"

wide["stability_class"] = wide.apply(stability_class, axis=1)

wide.to_csv(
    os.path.join(OUTDIR, "10_step18c_cluster_stability_by_well.csv"),
    index=False
)

# Summaries by season and stability class
stability_summary = (
    wide.groupby(["_season", "stability_class"], dropna=False)
    .size()
    .reset_index(name="well_season_count")
    .rename(columns={"_season": "season"})
)

stability_summary.to_csv(
    os.path.join(OUTDIR, "11_step18c_stability_summary.csv"),
    index=False
)

# Class transition table: k6 compared with k4 and k8
transition_rows = []
for season, group in wide.groupby("_season"):
    for comparison in ["cluster_k4", "cluster_k8"]:
        counts = (
            group.groupby(["cluster_k6", comparison], dropna=False)
            .size()
            .reset_index(name="count")
        )
        counts.insert(0, "comparison", comparison)
        counts.insert(0, "season", season)
        transition_rows.append(counts)

if transition_rows:
    pd.concat(transition_rows, ignore_index=True).to_csv(
        os.path.join(OUTDIR, "12_step18c_cluster_transition_tables.csv"),
        index=False
    )

# ---------------------------
# 6. Reconstruct KNN graphs and check components
# ---------------------------
component_rows = []

for season in ["pre", "post"]:
    sub = spatial[
        (spatial["_season"] == season)
    ].copy()

    sub[lat_s] = pd.to_numeric(sub[lat_s], errors="coerce")
    sub[lon_s] = pd.to_numeric(sub[lon_s], errors="coerce")
    sub[slope_s] = pd.to_numeric(sub[slope_s], errors="coerce")

    sub = sub.dropna(subset=[well_s, lat_s, lon_s, slope_s])
    sub = sub.drop_duplicates(subset=[well_s])

    # KNN uses projected coordinates in metres rather than degrees.
    if len(sub) < 9:
        for k in [4, 6, 8]:
            component_rows.append({
                "season": season,
                "k_neighbours": k,
                "n_wells": len(sub),
                "n_connected_components": np.nan,
                "component_sizes": "INSUFFICIENT_WELLS"
            })
        continue

    try:
        import geopandas as gpd
        from shapely.geometry import Point

        gdf = gpd.GeoDataFrame(
            sub.copy(),
            geometry=[
                Point(xy) for xy in zip(sub[lon_s], sub[lat_s])
            ],
            crs="EPSG:4326"
        ).to_crs("EPSG:32643")

        coords = np.column_stack([
            gdf.geometry.x.to_numpy(),
            gdf.geometry.y.to_numpy()
        ])

        for k in [4, 6, 8]:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                w = KNN.from_array(coords, k=k)
                w.transform = "R"

            graph = w.sparse
            n_components, labels = connected_components(
                graph, directed=False, return_labels=True
            )
            sizes = np.bincount(labels)

            component_rows.append({
                "season": season,
                "k_neighbours": k,
                "n_wells": len(sub),
                "n_connected_components": int(n_components),
                "component_sizes": ";".join(
                    map(str, sorted(sizes.tolist(), reverse=True))
                ),
                "warnings": " | ".join(str(x.message) for x in caught)
            })

    except Exception as exc:
        print(f"Graph check failed for {season}: {exc}")
        for k in [4, 6, 8]:
            component_rows.append({
                "season": season,
                "k_neighbours": k,
                "n_wells": len(sub),
                "n_connected_components": np.nan,
                "component_sizes": f"ERROR: {exc}"
            })

components = pd.DataFrame(component_rows)
components.to_csv(
    os.path.join(OUTDIR, "13_step18c_connected_components.csv"),
    index=False
)

# ---------------------------
# 7. Summarize k coverage and class counts
# ---------------------------
class_summary = (
    local.groupby([season_l, k_l, "cluster_normalized"], dropna=False)
    .size()
    .reset_index(name="series_count")
    .rename(columns={
        season_l: "season",
        k_l: "k_neighbours",
        "cluster_normalized": "cluster_class"
    })
)

class_summary.to_csv(
    os.path.join(OUTDIR, "14_step18c_cluster_class_counts.csv"),
    index=False
)

# ---------------------------
# 8. Save JSON summary
# ---------------------------
summary = {
    "step": "18C_Cluster_Stability_and_Weights_Audit",
    "input_step16": STEP16,
    "input_step18b": STEP18B,
    "primary_neighbourhood": 6,
    "sensitivity_neighbourhoods": [4, 8],
    "local_records": int(len(local)),
    "unique_well_season_pairs": int(
        wide[["_well", "_season"]].drop_duplicates().shape[0]
    ),
    "duplicate_well_season_k_records": int(duplicates.sum()),
    "cluster_class_counts": class_summary.to_dict(orient="records"),
    "stability_counts": stability_summary.to_dict(orient="records"),
    "connected_component_audit": components.to_dict(orient="records"),
    "interpretation_note": (
        "Local Moran classes describe spatial association in Sen slopes, "
        "not causation or necessarily significant temporal trends. "
        "Classification stability is conditional on the supplied Step 18B "
        "cluster labels and FDR procedure. Connected-component checks "
        "reconstruct KNN graphs from available coordinates and may not "
        "exactly reproduce the original Step 18B weights if its input "
        "filtering or coordinate handling differed."
    )
}

with open(
    os.path.join(OUTDIR, "15_step18c_summary.json"), "w"
) as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 65)
print("STEP 18C COMPLETED")
print("=" * 65)
print("\nSTABILITY SUMMARY")
print(stability_summary.to_string(index=False))

print("\nCONNECTED COMPONENTS")
print(components.to_string(index=False))

print("\nCLUSTER COUNTS")
print(class_summary.to_string(index=False))

print("\nOutputs saved in:")
print(OUTDIR)


**[orig #144]** Step 18D — publication maps of local clusters  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 18D: PUBLICATION-READY LOCAL CLUSTER MAPS
# + k=6 vs k=8 sensitivity comparison
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------
# 1. Paths
# ---------------------------
BASE = (
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

INPUT = os.path.join(
    BASE,
    "Step_18_Spatial_Clustering_Uncertainty",
    "07_step18b_local_moran_by_well_season.csv"
)

OUTDIR = os.path.join(
    BASE,
    "Step_18_Spatial_Clustering_Uncertainty"
)

FIGDIR = os.path.join(OUTDIR, "figures_step18d")
os.makedirs(FIGDIR, exist_ok=True)

if not os.path.exists(INPUT):
    raise FileNotFoundError(f"Input file not found: {INPUT}")

df = pd.read_csv(INPUT)

print("Input rows:", len(df))
print("Columns:", df.columns.tolist())

# ---------------------------
# 2. Validate required columns
# ---------------------------
required = [
    "Season", "k_neighbours", "Well_key", "Latitude",
    "Longitude", "original_sen_slope_m_per_year",
    "local_p_fdr_bh", "cluster_class"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["Season"] = df["Season"].astype(str).str.strip().str.lower()
df["k_neighbours"] = pd.to_numeric(
    df["k_neighbours"], errors="coerce"
)
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["original_sen_slope_m_per_year"] = pd.to_numeric(
    df["original_sen_slope_m_per_year"], errors="coerce"
)
df["local_p_fdr_bh"] = pd.to_numeric(
    df["local_p_fdr_bh"], errors="coerce"
)

df["Well_key"] = df["Well_key"].astype(str).str.strip()
df = df[df["k_neighbours"].isin([4, 6, 8])].copy()

# ---------------------------
# 3. Normalize cluster labels
# ---------------------------
def normalize_class(value):
    x = str(value).strip().upper().replace("-", "_").replace(" ", "_")

    if "HIGH_HIGH" in x:
        return "HIGH_HIGH"
    if "LOW_LOW" in x:
        return "LOW_LOW"
    if "HIGH_LOW" in x:
        return "HIGH_LOW"
    if "LOW_HIGH" in x:
        return "LOW_HIGH"
    if "NOT_SIGNIFICANT" in x or "NON_SIGNIFICANT" in x:
        return "NOT_SIGNIFICANT"
    if "NOT_TESTED" in x or "UNAVAILABLE" in x:
        return "NOT_TESTED"
    return x

df["cluster_normalized"] = df["cluster_class"].map(normalize_class)

cluster_classes = [
    "HIGH_HIGH", "LOW_LOW", "HIGH_LOW",
    "LOW_HIGH", "NOT_SIGNIFICANT", "NOT_TESTED"
]

# Check that one record exists per well-season-k
dup = df.duplicated(
    subset=["Well_key", "Season", "k_neighbours"],
    keep=False
)

if dup.any():
    duplicate_path = os.path.join(
        OUTDIR, "16_step18d_duplicate_records.csv"
    )
    df.loc[dup].to_csv(duplicate_path, index=False)
    raise ValueError(
        f"Duplicate well-season-k records found. See {duplicate_path}"
    )

# ---------------------------
# 4. Well-level k=6 versus k=8 comparison
# ---------------------------
sub = df[df["k_neighbours"].isin([6, 8])].copy()

wide = sub.pivot(
    index=["Well_key", "Season"],
    columns="k_neighbours",
    values=[
        "cluster_normalized",
        "original_sen_slope_m_per_year",
        "local_p_fdr_bh"
    ]
)

wide.columns = [
    f"{metric}_k{int(k)}" for metric, k in wide.columns
]
wide = wide.reset_index()

# Attach well coordinates and block labels
metadata_cols = ["Well_key", "Season", "Latitude", "Longitude"]
if "Block" in df.columns:
    metadata_cols.append("Block")

metadata = (
    df[metadata_cols]
    .drop_duplicates(subset=["Well_key", "Season"])
)

wide = wide.merge(
    metadata,
    on=["Well_key", "Season"],
    how="left",
    validate="one_to_one"
)

def is_cluster(x):
    return x in ["HIGH_HIGH", "LOW_LOW", "HIGH_LOW", "LOW_HIGH"]

def compare_clusters(row):
    c6 = row.get("cluster_normalized_k6", np.nan)
    c8 = row.get("cluster_normalized_k8", np.nan)

    if pd.isna(c6) or pd.isna(c8):
        return "MISSING_K6_OR_K8"

    sig6, sig8 = is_cluster(c6), is_cluster(c8)

    if sig6 and sig8 and c6 == c8:
        return "SIGNIFICANT_SAME_CLASS_AT_K6_AND_K8"

    if sig6 and sig8 and c6 != c8:
        return "SIGNIFICANT_BOTH_BUT_CLASS_CHANGED"

    if sig6 and not sig8:
        return "SIGNIFICANT_AT_K6_ONLY"

    if sig8 and not sig6:
        return "SIGNIFICANT_AT_K8_ONLY"

    if c6 == "NOT_SIGNIFICANT" and c8 == "NOT_SIGNIFICANT":
        return "NOT_SIGNIFICANT_AT_BOTH"

    return "OTHER_OR_UNAVAILABLE_CLASS"

wide["k6_k8_comparison"] = wide.apply(compare_clusters, axis=1)

# Slope direction comparison; this is separate from cluster class.
def slope_direction(x):
    if pd.isna(x):
        return "UNAVAILABLE"
    if x > 0:
        return "DEEPENING"
    if x < 0:
        return "SHALLOWING"
    return "ZERO_SLOPE"

wide["slope_direction_k6"] = wide[
    "original_sen_slope_m_per_year_k6"
].map(slope_direction)

wide["slope_direction_k8"] = wide[
    "original_sen_slope_m_per_year_k8"
].map(slope_direction)

wide["slope_direction_consistency"] = np.where(
    (wide["slope_direction_k6"] == "UNAVAILABLE") |
    (wide["slope_direction_k8"] == "UNAVAILABLE"),
    "SLOPE_UNAVAILABLE",
    np.where(
        wide["slope_direction_k6"] == wide["slope_direction_k8"],
        "SAME_DIRECTION",
        "DIRECTION_CHANGED"
    )
)

comparison_file = os.path.join(
    OUTDIR, "16_step18d_k6_k8_well_comparison.csv"
)
wide.to_csv(comparison_file, index=False)

comparison_summary = (
    wide.groupby(["Season", "k6_k8_comparison"], dropna=False)
    .size()
    .reset_index(name="well_season_count")
)

comparison_summary.to_csv(
    os.path.join(OUTDIR, "17_step18d_k6_k8_comparison_summary.csv"),
    index=False
)

# ---------------------------
# 5. Make local cluster maps
# ---------------------------
# These are point maps, not interpolated surfaces.
# The labels show the FDR-adjusted Step 18B classifications.

class_colors = {
    "HIGH_HIGH": "#b2182b",
    "LOW_LOW": "#2166ac",
    "HIGH_LOW": "#ef8a62",
    "LOW_HIGH": "#67a9cf",
    "NOT_SIGNIFICANT": "#bdbdbd",
    "NOT_TESTED": "#737373"
}

class_labels = {
    "HIGH_HIGH": "HIGH-HIGH",
    "LOW_LOW": "LOW-LOW",
    "HIGH_LOW": "HIGH-LOW",
    "LOW_HIGH": "LOW-HIGH",
    "NOT_SIGNIFICANT": "Not significant after FDR",
    "NOT_TESTED": "Not tested / unavailable"
}

def make_cluster_map(season, k):
    part = df[
        (df["Season"] == season) &
        (df["k_neighbours"] == k)
    ].copy()

    part = part.dropna(subset=["Latitude", "Longitude"])

    if part.empty:
        print(f"No mappable records for {season}, k={k}")
        return

    fig, ax = plt.subplots(figsize=(8, 8))

    for cls in cluster_classes:
        g = part[part["cluster_normalized"] == cls]
        if g.empty:
            continue

        ax.scatter(
            g["Longitude"], g["Latitude"],
            s=45 if cls != "NOT_SIGNIFICANT" else 24,
            c=class_colors[cls],
            edgecolors="black" if cls != "NOT_SIGNIFICANT" else "none",
            linewidths=0.4,
            alpha=0.9,
            label=f"{class_labels[cls]} (n={len(g)})",
            zorder=3 if cls != "NOT_SIGNIFICANT" else 1
        )

    ax.set_xlabel("Longitude (degrees)")
    ax.set_ylabel("Latitude (degrees)")
    ax.set_title(
        f"Alwar groundwater Sen-slope local clusters\n"
        f"{season.capitalize()}-monsoon | KNN k={k}"
    )
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(loc="best", fontsize=8, frameon=True)
    fig.tight_layout()

    filename = f"step18d_{season}_local_clusters_k{k}.png"
    filepath = os.path.join(FIGDIR, filename)
    fig.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Saved map:", filepath)

make_cluster_map("pre", 6)
make_cluster_map("post", 6)
make_cluster_map("post", 8)

# ---------------------------
# 6. Export cluster class counts
# ---------------------------
class_counts = (
    df.groupby(["Season", "k_neighbours", "cluster_normalized"])
    .size()
    .reset_index(name="well_season_count")
)

class_counts.to_csv(
    os.path.join(OUTDIR, "18_step18d_cluster_class_counts.csv"),
    index=False
)

# ---------------------------
# 7. JSON summary
# ---------------------------
summary = {
    "step": "18D_Local_Cluster_Maps_and_k6_k8_Comparison",
    "input": INPUT,
    "primary_neighbourhood": 6,
    "sensitivity_neighbourhood": 8,
    "map_files_directory": FIGDIR,
    "comparison_file": comparison_file,
    "records_in_input": int(len(df)),
    "records_used_for_k6_k8_comparison": int(len(wide)),
    "comparison_counts": comparison_summary.to_dict(orient="records"),
    "class_counts": class_counts.to_dict(orient="records"),
    "interpretation_note": (
        "Maps display observed well locations only; no spatial interpolation "
        "is performed. HIGH-HIGH and LOW-LOW refer to relative local patterns "
        "in Sen slopes, not automatically to significant temporal trends. "
        "The k=8 analysis is a sensitivity test, not a replacement for k=6."
    )
}

with open(
    os.path.join(OUTDIR, "19_step18d_summary.json"), "w"
) as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 65)
print("STEP 18D COMPLETED")
print("=" * 65)

print("\nK=6 vs K=8 COMPARISON SUMMARY")
print(comparison_summary.to_string(index=False))

print("\nCLUSTER CLASS COUNTS")
print(class_counts.to_string(index=False))

print("\nComparison table:")
print(comparison_file)
print("\nFigures:")
print(FIGDIR)
print("\nJSON summary:")
print(os.path.join(OUTDIR, "19_step18d_summary.json"))
